https://www.gsmarena.com/makers.php3 -- ссылка с которой все начинается, это все мобильные бренды
дальше нужно смотреть верстку, 

в подробности вдаваться не буду, но в body есть табличка, которую нам и нужно выцепить, а дальше вытащить все ссылки на страницы девайсов с кнопок таблицы

In [1]:
from bs4 import BeautifulSoup
import re
import time
import json
import requests
import random
from pathlib import Path

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.google.com/",
})

html_pages_path = "crawler"
domain_name = 'https://www.gsmarena.com/'
brands_to_use = ["samsung", "acer", "apple", "honor"]

session.get(domain_name)

def get_bs(url, retries=3):
    for i in range(retries):
        try:
            time.sleep(10 + random.uniform(2.5, 6.0))
            r = session.get(url, timeout=(10, 40))
            r.raise_for_status()
            return BeautifulSoup(r.text, "html.parser")
        except Exception as e:
            print(f"[{i+1}/{retries}] Error: {e}")
            time.sleep(15 + random.uniform(2.5, 6.0))
    return None

def get_bs_file(path):
    folder = Path(path)
    with open(folder, "r", encoding="utf-8") as f:
        return BeautifulSoup(f, "html.parser")

In [53]:
def get_devices_with_links(url, filter=[]):
    bs = get_bs(url)
    table = bs.findChild('div', class_='st-text').table
    
    if not table:
        print(f"Error({get_devices_with_links.__name__}): No table")
        return
    rows = table.findChildren('td')
    entity_array = {}
    for row in rows:
        row_a_tag = row.a
        if (not row_a_tag):
            print(f"Error({get_devices_with_links.__name__}): no a tag in a row")
            continue
        device_link = domain_name + row_a_tag["href"]
        device_brand, num_devices = (item for item in row_a_tag.stripped_strings)
        num_devices = int(re.findall(re.compile('\d+'), num_devices)[0])
        if ((device_brand.lower() in filter) or (not filter)):
            entity_array[device_brand.lower()] = ([device_link, num_devices])
    return entity_array

In [77]:
start_url = domain_name + "makers.php3"
parsed_first_page = get_devices_with_links(start_url, filter=brands_to_use)
print(json.dumps(parsed_first_page, indent=2, ensure_ascii=False))

[1/3] Error: 429 Client Error: Too Many Requests for url: https://www.gsmarena.com/makers.php3


KeyboardInterrupt: 

In [2]:
def get_device_refs(url):
    bs = get_bs(url)
    if (not bs):
        return []
    device_refs = []
    makers = bs.findChild('div', class_='makers')
    if (not makers):
        print(f"Error({get_device_refs.__name__}): no makers")
        return []
    for device in makers.findChildren('a'):
       device_refs.append([device.find("strong").get_text(strip=True), domain_name + device['href']])
    return device_refs

def get_specific_devices_refs(url):
    bs = get_bs(url)
    if (not bs):
        return []
    print(url)
    makers = get_device_refs(url)
    
    nav_pages = bs.findChild('div', class_='nav-pages')
    if (not nav_pages):
        print(f"Error({get_specific_devices_refs.__name__}): no nav-pages")
        return makers
    next_page = nav_pages.findChild('a', title='Next page', recursive=False)
    if (not next_page):
        return makers
    next_page = domain_name + next_page["href"]
    
    makers_next = get_specific_devices_refs(next_page)
    makers.extend(makers_next)
    return makers


In [27]:
brand_devices_refs = dict()

for key, val in parsed_first_page.items():
    brand_devices_refs[key] = get_specific_devices_refs(val[0])

print(json.dumps(brand_devices_refs, indent = 2, ensure_ascii=False))

NameError: name 'parsed_first_page' is not defined

In [3]:
brand_devices_refs = {
  "acer": [
    [
      "Iconia V12",
      "https://www.gsmarena.com/acer_iconia_v12-14298.php"
    ],
    [
      "Iconia V11",
      "https://www.gsmarena.com/acer_iconia_v11-14297.php"
    ],
    [
      "Acerone Liquid S262F5",
      "https://www.gsmarena.com/acer_acerone_liquid_s262f5_5g-14250.php"
    ],
    [
      "Iconia A14",
      "https://www.gsmarena.com/acer_iconia_a14-14120.php"
    ],
    [
      "Iconia A16",
      "https://www.gsmarena.com/acer_iconia_a16-14119.php"
    ],
    [
      "Iconia X14",
      "https://www.gsmarena.com/acer_iconia_x14-14117.php"
    ],
    [
      "Iconia X12",
      "https://www.gsmarena.com/acer_iconia_x12-14118.php"
    ],
    [
      "Iconia Tab P11 (2025)",
      "https://www.gsmarena.com/acer_iconia_tab_p11_(2025)-14299.php"
    ],
    [
      "Iconia Tab P10 (2025)",
      "https://www.gsmarena.com/acer_iconia_tab_p10_(2025)-14300.php"
    ],
    [
      "Super ZX Pro",
      "https://www.gsmarena.com/acer_super_zx_pro_5g-13795.php"
    ],
    [
      "Super ZX",
      "https://www.gsmarena.com/acer_super_zx_5g-13796.php"
    ],
    [
      "Acerone Liquid S272E4",
      "https://www.gsmarena.com/acer_acerone_liquid_s272e4-13757.php"
    ],
    [
      "Acerone Liquid S162E4",
      "https://www.gsmarena.com/acer_acerone_liquid_s162e4-13756.php"
    ],
    [
      "Chromebook Tab 10",
      "https://www.gsmarena.com/acer_chromebook_tab_10-9139.php"
    ],
    [
      "Iconia Talk S",
      "https://www.gsmarena.com/acer_iconia_talk_s-8306.php"
    ],
    [
      "Liquid Z6 Plus",
      "https://www.gsmarena.com/acer_liquid_z6_plus-8305.php"
    ],
    [
      "Liquid Z6",
      "https://www.gsmarena.com/acer_liquid_z6-8304.php"
    ],
    [
      "Iconia Tab 10 A3-A40",
      "https://www.gsmarena.com/acer_iconia_tab_10_a3_a40-8080.php"
    ],
    [
      "Liquid X2",
      "https://www.gsmarena.com/acer_liquid_x2-8034.php"
    ],
    [
      "Liquid Jade 2",
      "https://www.gsmarena.com/acer_liquid_jade_2-7956.php"
    ],
    [
      "Liquid Zest Plus",
      "https://www.gsmarena.com/acer_liquid_zest_plus-8059.php"
    ],
    [
      "Liquid Zest",
      "https://www.gsmarena.com/acer_liquid_zest-7955.php"
    ],
    [
      "Predator 8",
      "https://www.gsmarena.com/acer_predator_8-7750.php"
    ],
    [
      "Liquid Jade Primo",
      "https://www.gsmarena.com/acer_liquid_jade_primo-7650.php"
    ],
    [
      "Liquid Z330",
      "https://www.gsmarena.com/acer_liquid_z330-7530.php"
    ],
    [
      "Liquid Z320",
      "https://www.gsmarena.com/acer_liquid_z320-7531.php"
    ],
    [
      "Liquid Z630S",
      "https://www.gsmarena.com/acer_liquid_z630s-7529.php"
    ],
    [
      "Liquid Z630",
      "https://www.gsmarena.com/acer_liquid_z630-7528.php"
    ],
    [
      "Liquid Z530S",
      "https://www.gsmarena.com/acer_liquid_z530s-7527.php"
    ],
    [
      "Liquid Z530",
      "https://www.gsmarena.com/acer_liquid_z530-7526.php"
    ],
    [
      "Liquid M330",
      "https://www.gsmarena.com/acer_liquid_m330-7524.php"
    ],
    [
      "Liquid M320",
      "https://www.gsmarena.com/acer_liquid_m320-7525.php"
    ],
    [
      "Iconia Tab 10 A3-A30",
      "https://www.gsmarena.com/acer_iconia_tab_10_a3_a30-7218.php"
    ],
    [
      "Iconia One 8 B1-820",
      "https://www.gsmarena.com/acer_iconia_one_8_b1_820-7217.php"
    ],
    [
      "Iconia Tab A3-A20",
      "https://www.gsmarena.com/acer_iconia_tab_a3_a20-7136.php"
    ],
    [
      "Iconia Tab A3-A20FHD",
      "https://www.gsmarena.com/acer_iconia_tab_a3_a20fhd-7135.php"
    ],
    [
      "Liquid Jade Z",
      "https://www.gsmarena.com/acer_liquid_jade_z-7072.php"
    ],
    [
      "Liquid Z520",
      "https://www.gsmarena.com/acer_liquid_z520-7073.php"
    ],
    [
      "Liquid Z220",
      "https://www.gsmarena.com/acer_liquid_z220-7074.php"
    ],
    [
      "Liquid M220",
      "https://www.gsmarena.com/acer_liquid_m220-7071.php"
    ],
    [
      "Liquid Z410",
      "https://www.gsmarena.com/acer_liquid_z410-6912.php"
    ],
    [
      "Liquid Jade S",
      "https://www.gsmarena.com/acer_liquid_jade_s-6864.php"
    ],
    [
      "Liquid Z500",
      "https://www.gsmarena.com/acer_liquid_z500-6635.php"
    ],
    [
      "Liquid X1",
      "https://www.gsmarena.com/acer_liquid_x1-6419.php"
    ],
    [
      "Liquid Jade",
      "https://www.gsmarena.com/acer_liquid_jade-6423.php"
    ],
    [
      "Liquid E700",
      "https://www.gsmarena.com/acer_liquid_e700-6420.php"
    ],
    [
      "Liquid E600",
      "https://www.gsmarena.com/acer_liquid_e600-6421.php"
    ],
    [
      "Liquid Z200",
      "https://www.gsmarena.com/acer_liquid_z200-6422.php"
    ],
    [
      "Iconia Tab 8 A1-840FHD",
      "https://www.gsmarena.com/acer_iconia_tab_8_a1_840fhd-6424.php"
    ],
    [
      "Iconia Tab 7 A1-713",
      "https://www.gsmarena.com/acer_iconia_tab_7_a1_713-6343.php"
    ],
    [
      "Iconia Tab 7 A1-713HD",
      "https://www.gsmarena.com/acer_iconia_tab_7_a1_713hd-6342.php"
    ],
    [
      "Iconia One 7 B1-730",
      "https://www.gsmarena.com/acer_iconia_one_7_b1_730-6341.php"
    ],
    [
      "Liquid E3 Duo Plus",
      "https://www.gsmarena.com/acer_liquid_e3_duo_plus-6680.php"
    ],
    [
      "Liquid E3",
      "https://www.gsmarena.com/acer_liquid_e3-6116.php"
    ],
    [
      "Liquid Z4",
      "https://www.gsmarena.com/acer_liquid_z4-6115.php"
    ],
    [
      "Iconia B1-721",
      "https://www.gsmarena.com/acer_iconia_b1_721-5933.php"
    ],
    [
      "Iconia B1-720",
      "https://www.gsmarena.com/acer_iconia_b1_720-5932.php"
    ],
    [
      "Iconia A1-830",
      "https://www.gsmarena.com/acer_iconia_a1_830-5930.php"
    ],
    [
      "Liquid Z5",
      "https://www.gsmarena.com/acer_liquid_z5-5929.php"
    ],
    [
      "Liquid S2",
      "https://www.gsmarena.com/acer_liquid_s2-5670.php"
    ],
    [
      "Liquid Z3",
      "https://www.gsmarena.com/acer_liquid_z3-5624.php"
    ],
    [
      "Liquid S1",
      "https://www.gsmarena.com/acer_liquid_s1-5496.php"
    ],
    [
      "Iconia Tab A3",
      "https://www.gsmarena.com/acer_iconia_tab_a3-5700.php"
    ],
    [
      "Iconia Tab A1-811",
      "https://www.gsmarena.com/acer_iconia_tab_a1_811-5892.php"
    ],
    [
      "Iconia Tab A1-810",
      "https://www.gsmarena.com/acer_iconia_tab_a1_810-5399.php"
    ],
    [
      "Liquid E2",
      "https://www.gsmarena.com/acer_liquid_e2-5418.php"
    ],
    [
      "Liquid Z2",
      "https://www.gsmarena.com/acer_liquid_z2-5302.php"
    ],
    [
      "Liquid C1",
      "https://www.gsmarena.com/acer_liquid_c1-5276.php"
    ],
    [
      "Liquid E1",
      "https://www.gsmarena.com/acer_liquid_e1-5262.php"
    ],
    [
      "Iconia Tab B1-710",
      "https://www.gsmarena.com/acer_iconia_tab_b1_710-5897.php"
    ],
    [
      "Iconia Tab B1-A71",
      "https://www.gsmarena.com/acer_iconia_tab_b1_a71-5239.php"
    ],
    [
      "Iconia Tab A110",
      "https://www.gsmarena.com/acer_iconia_tab_a110-5067.php"
    ],
    [
      "Liquid Z110",
      "https://www.gsmarena.com/acer_liquid_z110-5057.php"
    ],
    [
      "Liquid Gallant E350",
      "https://www.gsmarena.com/acer_liquid_gallant_e350-4924.php"
    ],
    [
      "Liquid Gallant Duo",
      "https://www.gsmarena.com/acer_liquid_gallant_duo-4871.php"
    ],
    [
      "Liquid Glow E330",
      "https://www.gsmarena.com/acer_liquid_glow_e330-4589.php"
    ],
    [
      "CloudMobile S500",
      "https://www.gsmarena.com/acer_cloudmobile_s500-4542.php"
    ],
    [
      "Iconia Tab A210",
      "https://www.gsmarena.com/acer_iconia_tab_a210-5066.php"
    ],
    [
      "Iconia Tab A200",
      "https://www.gsmarena.com/acer_iconia_tab_a200-4450.php"
    ],
    [
      "Iconia Tab A701",
      "https://www.gsmarena.com/acer_iconia_tab_a701-4934.php"
    ],
    [
      "Iconia Tab A700",
      "https://www.gsmarena.com/acer_iconia_tab_a700-4425.php"
    ],
    [
      "Iconia Tab A511",
      "https://www.gsmarena.com/acer_iconia_tab_a511-4643.php"
    ],
    [
      "Iconia Tab A510",
      "https://www.gsmarena.com/acer_iconia_tab_a510-4642.php"
    ],
    [
      "Allegro",
      "https://www.gsmarena.com/acer_allegro-3966.php"
    ],
    [
      "Liquid Express E320",
      "https://www.gsmarena.com/acer_liquid_express_e320-4261.php"
    ],
    [
      "Iconia Tab A501",
      "https://www.gsmarena.com/acer_iconia_tab_a501-3898.php"
    ],
    [
      "Iconia Tab A500",
      "https://www.gsmarena.com/acer_iconia_tab_a500-3907.php"
    ],
    [
      "Iconia Smart",
      "https://www.gsmarena.com/acer_iconia_smart-3662.php"
    ],
    [
      "Iconia Tab A101",
      "https://www.gsmarena.com/acer_iconia_tab_a101-3906.php"
    ],
    [
      "Iconia Tab A100",
      "https://www.gsmarena.com/acer_iconia_tab_a100-3841.php"
    ],
    [
      "Liquid mini E310",
      "https://www.gsmarena.com/acer_liquid_mini_e310-3711.php"
    ],
    [
      "beTouch E210",
      "https://www.gsmarena.com/acer_betouch_e210-3712.php"
    ],
    [
      "beTouch E140",
      "https://www.gsmarena.com/acer_betouch_e140-3666.php"
    ],
    [
      "beTouch T500",
      "https://www.gsmarena.com/acer_betouch_t500-3587.php"
    ],
    [
      "Liquid mt",
      "https://www.gsmarena.com/acer_liquid_mt-3519.php"
    ],
    [
      "beTouch E130",
      "https://www.gsmarena.com/acer_betouch_e130-3373.php"
    ],
    [
      "beTouch E120",
      "https://www.gsmarena.com/acer_betouch_e120-3406.php"
    ],
    [
      "Stream",
      "https://www.gsmarena.com/acer_stream-3358.php"
    ],
    [
      "Liquid E",
      "https://www.gsmarena.com/acer_liquid_e-3156.php"
    ],
    [
      "neoTouch P400",
      "https://www.gsmarena.com/acer_neotouch_p400-3155.php"
    ],
    [
      "beTouch E400",
      "https://www.gsmarena.com/acer_betouch_e400-3154.php"
    ],
    [
      "neoTouch P300",
      "https://www.gsmarena.com/acer_neotouch_p300-3138.php"
    ],
    [
      "beTouch E110",
      "https://www.gsmarena.com/acer_betouch_e110-3137.php"
    ],
    [
      "Liquid",
      "https://www.gsmarena.com/acer_liquid-2968.php"
    ],
    [
      "neoTouch",
      "https://www.gsmarena.com/acer_neotouch-2958.php"
    ],
    [
      "beTouch E200",
      "https://www.gsmarena.com/acer_betouch_e200-2961.php"
    ],
    [
      "beTouch E100",
      "https://www.gsmarena.com/acer_betouch_e100-2959.php"
    ],
    [
      "beTouch E101",
      "https://www.gsmarena.com/acer_betouch_e101-2960.php"
    ],
    [
      "DX650",
      "https://www.gsmarena.com/acer_dx650-2888.php"
    ],
    [
      "M900",
      "https://www.gsmarena.com/acer_m900-2719.php"
    ],
    [
      "F900",
      "https://www.gsmarena.com/acer_f900-2717.php"
    ],
    [
      "X960",
      "https://www.gsmarena.com/acer_x960-2716.php"
    ],
    [
      "DX900",
      "https://www.gsmarena.com/acer_dx900-2718.php"
    ]
  ],
  "apple": [
    [
      "iPad Pro 13 (2025)",
      "https://www.gsmarena.com/apple_ipad_pro_13_5g_(2025)-14233.php"
    ],
    [
      "iPad Pro 11 (2025)",
      "https://www.gsmarena.com/apple_ipad_pro_11_5g_(2025)-14234.php"
    ],
    [
      "iPhone 17 Pro Max",
      "https://www.gsmarena.com/apple_iphone_17_pro_max-13964.php"
    ],
    [
      "iPhone 17 Pro",
      "https://www.gsmarena.com/apple_iphone_17_pro-14049.php"
    ],
    [
      "iPhone Air",
      "https://www.gsmarena.com/apple_iphone_17_air-13502.php"
    ],
    [
      "iPhone 17",
      "https://www.gsmarena.com/apple_iphone_17-14050.php"
    ],
    [
      "Watch Ultra 3",
      "https://www.gsmarena.com/apple_watch_ultra_3-14130.php"
    ],
    [
      "Watch Series 11",
      "https://www.gsmarena.com/apple_watch_series_11-14131.php"
    ],
    [
      "Watch Series 11 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_11_aluminum-14132.php"
    ],
    [
      "Watch SE 3",
      "https://www.gsmarena.com/apple_watch_se_3-14133.php"
    ],
    [
      "iPad Air 13 (2025)",
      "https://www.gsmarena.com/apple_ipad_air_13_(2025)-13704.php"
    ],
    [
      "iPad Air 11 (2025)",
      "https://www.gsmarena.com/apple_ipad_air_11_(2025)-13703.php"
    ],
    [
      "iPad (2025)",
      "https://www.gsmarena.com/apple_ipad_(2025)-13702.php"
    ],
    [
      "iPhone 16e",
      "https://www.gsmarena.com/apple_iphone_16e-13395.php"
    ],
    [
      "iPad mini (2024)",
      "https://www.gsmarena.com/apple_ipad_mini_(2024)-13437.php"
    ],
    [
      "iPhone 16 Pro Max",
      "https://www.gsmarena.com/apple_iphone_16_pro_max-13123.php"
    ],
    [
      "iPhone 16 Pro",
      "https://www.gsmarena.com/apple_iphone_16_pro-13315.php"
    ],
    [
      "iPhone 16 Plus",
      "https://www.gsmarena.com/apple_iphone_16_plus-13316.php"
    ],
    [
      "iPhone 16",
      "https://www.gsmarena.com/apple_iphone_16-13317.php"
    ],
    [
      "Watch Series 10",
      "https://www.gsmarena.com/apple_watch_series_10-13318.php"
    ],
    [
      "Watch Series 10 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_10_aluminum-13319.php"
    ],
    [
      "iPad Pro 13 (2024)",
      "https://www.gsmarena.com/apple_ipad_pro_13_(2024)-12987.php"
    ],
    [
      "iPad Pro 11 (2024)",
      "https://www.gsmarena.com/apple_ipad_pro_11_(2024)-12986.php"
    ],
    [
      "iPad Air 13 (2024)",
      "https://www.gsmarena.com/apple_ipad_air_13_(2024)-12985.php"
    ],
    [
      "iPad Air 11 (2024)",
      "https://www.gsmarena.com/apple_ipad_air_11_(2024)-12984.php"
    ],
    [
      "iPhone 15 Pro Max",
      "https://www.gsmarena.com/apple_iphone_15_pro_max-12548.php"
    ],
    [
      "iPhone 15 Pro",
      "https://www.gsmarena.com/apple_iphone_15_pro-12557.php"
    ],
    [
      "iPhone 15 Plus",
      "https://www.gsmarena.com/apple_iphone_15_plus-12558.php"
    ],
    [
      "iPhone 15",
      "https://www.gsmarena.com/apple_iphone_15-12559.php"
    ],
    [
      "Watch Ultra 2",
      "https://www.gsmarena.com/apple_watch_ultra_2-12560.php"
    ],
    [
      "Watch Series 9",
      "https://www.gsmarena.com/apple_watch_series_9-12561.php"
    ],
    [
      "Watch Series 9 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_9_aluminum-12562.php"
    ],
    [
      "iPad Pro 12.9 (2022)",
      "https://www.gsmarena.com/apple_ipad_pro_12_9_(2022)-11939.php"
    ],
    [
      "iPad Pro 11 (2022)",
      "https://www.gsmarena.com/apple_ipad_pro_11_(2022)-11940.php"
    ],
    [
      "iPad (2022)",
      "https://www.gsmarena.com/apple_ipad_(2022)-11941.php"
    ],
    [
      "iPhone 14 Pro Max",
      "https://www.gsmarena.com/apple_iphone_14_pro_max-11773.php"
    ],
    [
      "iPhone 14 Pro",
      "https://www.gsmarena.com/apple_iphone_14_pro-11860.php"
    ],
    [
      "iPhone 14 Plus",
      "https://www.gsmarena.com/apple_iphone_14_plus-11862.php"
    ],
    [
      "iPhone 14",
      "https://www.gsmarena.com/apple_iphone_14-11861.php"
    ],
    [
      "Watch Ultra",
      "https://www.gsmarena.com/apple_watch_ultra-11827.php"
    ],
    [
      "Watch Series 8",
      "https://www.gsmarena.com/apple_watch_series_8-11866.php"
    ],
    [
      "Watch Series 8 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_8_aluminum-11864.php"
    ],
    [
      "Watch SE 2",
      "https://www.gsmarena.com/apple_watch_se_(2022)-11865.php"
    ],
    [
      "iPhone SE (2022)",
      "https://www.gsmarena.com/apple_iphone_se_(2022)-11410.php"
    ],
    [
      "iPad Air (2022)",
      "https://www.gsmarena.com/apple_ipad_air_(2022)-11411.php"
    ],
    [
      "iPhone 13 Pro Max",
      "https://www.gsmarena.com/apple_iphone_13_pro_max-11089.php"
    ],
    [
      "iPhone 13 Pro",
      "https://www.gsmarena.com/apple_iphone_13_pro-11102.php"
    ],
    [
      "iPhone 13",
      "https://www.gsmarena.com/apple_iphone_13-11103.php"
    ],
    [
      "iPhone 13 mini",
      "https://www.gsmarena.com/apple_iphone_13_mini-11104.php"
    ],
    [
      "iPad mini (2021)",
      "https://www.gsmarena.com/apple_ipad_mini_(2021)-11105.php"
    ],
    [
      "iPad 10.2 (2021)",
      "https://www.gsmarena.com/apple_ipad_10_2_(2021)-11106.php"
    ],
    [
      "Watch Edition Series 7",
      "https://www.gsmarena.com/apple_watch_edition_series_7-11147.php"
    ],
    [
      "Watch Series 7",
      "https://www.gsmarena.com/apple_watch_series_7-11146.php"
    ],
    [
      "Watch Series 7 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_7_aluminum-11107.php"
    ],
    [
      "iPad Pro 12.9 (2021)",
      "https://www.gsmarena.com/apple_ipad_pro_12_9_(2021)-10864.php"
    ],
    [
      "iPad Pro 11 (2021)",
      "https://www.gsmarena.com/apple_ipad_pro_11_(2021)-10865.php"
    ],
    [
      "iPhone 12 Pro Max",
      "https://www.gsmarena.com/apple_iphone_12_pro_max-10237.php"
    ],
    [
      "iPhone 12 Pro",
      "https://www.gsmarena.com/apple_iphone_12_pro-10508.php"
    ],
    [
      "iPhone 12",
      "https://www.gsmarena.com/apple_iphone_12-10509.php"
    ],
    [
      "iPhone 12 mini",
      "https://www.gsmarena.com/apple_iphone_12_mini-10510.php"
    ],
    [
      "iPad Air (2020)",
      "https://www.gsmarena.com/apple_ipad_air_(2020)-10444.php"
    ],
    [
      "iPad 10.2 (2020)",
      "https://www.gsmarena.com/apple_ipad_10_2_(2020)-10445.php"
    ],
    [
      "Watch SE",
      "https://www.gsmarena.com/apple_watch_se-10446.php"
    ],
    [
      "Watch Series 6 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_6_aluminum-10447.php"
    ],
    [
      "Watch Series 6",
      "https://www.gsmarena.com/apple_watch_series_6-10448.php"
    ],
    [
      "Watch Edition Series 6",
      "https://www.gsmarena.com/apple_watch_edition_series_6-10449.php"
    ],
    [
      "iPhone SE (2020)",
      "https://www.gsmarena.com/apple_iphone_se_(2020)-10170.php"
    ],
    [
      "iPad Pro 12.9 (2020)",
      "https://www.gsmarena.com/apple_ipad_pro_12_9_(2020)-10136.php"
    ],
    [
      "iPad Pro 11 (2020)",
      "https://www.gsmarena.com/apple_ipad_pro_11_(2020)-10137.php"
    ],
    [
      "iPhone 11 Pro Max",
      "https://www.gsmarena.com/apple_iphone_11_pro_max-9846.php"
    ],
    [
      "iPhone 11 Pro",
      "https://www.gsmarena.com/apple_iphone_11_pro-9847.php"
    ],
    [
      "iPhone 11",
      "https://www.gsmarena.com/apple_iphone_11-9848.php"
    ],
    [
      "iPad 10.2 (2019)",
      "https://www.gsmarena.com/apple_ipad_10_2_(2019)-9857.php"
    ],
    [
      "Watch Edition Series 5",
      "https://www.gsmarena.com/apple_watch_edition_series_5-9860.php"
    ],
    [
      "Watch Series 5",
      "https://www.gsmarena.com/apple_watch_series_5-9859.php"
    ],
    [
      "Watch Series 5 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_5_aluminum-9858.php"
    ],
    [
      "iPad Air (2019)",
      "https://www.gsmarena.com/apple_ipad_air_(2019)-9638.php"
    ],
    [
      "iPad mini (2019)",
      "https://www.gsmarena.com/apple_ipad_mini_(2019)-9637.php"
    ],
    [
      "iPad Pro 12.9 (2018)",
      "https://www.gsmarena.com/apple_ipad_pro_12_9_(2018)-9387.php"
    ],
    [
      "iPad Pro 11 (2018)",
      "https://www.gsmarena.com/apple_ipad_pro_11_(2018)-9386.php"
    ],
    [
      "iPhone XS Max",
      "https://www.gsmarena.com/apple_iphone_xs_max-9319.php"
    ],
    [
      "iPhone XS",
      "https://www.gsmarena.com/apple_iphone_xs-9318.php"
    ],
    [
      "iPhone XR",
      "https://www.gsmarena.com/apple_iphone_xr-9320.php"
    ],
    [
      "Watch Series 4",
      "https://www.gsmarena.com/apple_watch_series_4-9321.php"
    ],
    [
      "Watch Series 4 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_4_aluminum-9322.php"
    ],
    [
      "iPad 9.7 (2018)",
      "https://www.gsmarena.com/apple_ipad_9_7_(2018)-9142.php"
    ],
    [
      "iPhone X",
      "https://www.gsmarena.com/apple_iphone_x-8858.php"
    ],
    [
      "iPhone 8 Plus",
      "https://www.gsmarena.com/apple_iphone_8_plus-8131.php"
    ],
    [
      "iPhone 8",
      "https://www.gsmarena.com/apple_iphone_8-8573.php"
    ],
    [
      "Watch Edition Series 3",
      "https://www.gsmarena.com/apple_watch_edition_series_3-8861.php"
    ],
    [
      "Watch Series 3",
      "https://www.gsmarena.com/apple_watch_series_3-8860.php"
    ],
    [
      "Watch Series 3 Aluminum",
      "https://www.gsmarena.com/apple_watch_series_3_aluminum-8859.php"
    ],
    [
      "iPad Pro 12.9 (2017)",
      "https://www.gsmarena.com/apple_ipad_pro_12_9_(2017)-8717.php"
    ],
    [
      "iPad Pro 10.5 (2017)",
      "https://www.gsmarena.com/apple_ipad_pro_10_5_(2017)-8716.php"
    ],
    [
      "iPad 9.7 (2017)",
      "https://www.gsmarena.com/apple_ipad_9_7_(2017)-8620.php"
    ],
    [
      "Watch Edition Series 2 42mm",
      "https://www.gsmarena.com/apple_watch_edition_series_2_42mm-8331.php"
    ],
    [
      "Watch Edition Series 2 38mm",
      "https://www.gsmarena.com/apple_watch_edition_series_2_38mm-8332.php"
    ],
    [
      "Watch Series 2 42mm",
      "https://www.gsmarena.com/apple_watch_series_2_42mm-8329.php"
    ],
    [
      "Watch Series 2 38mm",
      "https://www.gsmarena.com/apple_watch_series_2_38mm-8330.php"
    ],
    [
      "Watch Series 2 Aluminum 42mm",
      "https://www.gsmarena.com/apple_watch_series_2_aluminum_42mm-8328.php"
    ],
    [
      "Watch Series 1 Aluminum 42mm",
      "https://www.gsmarena.com/apple_watch_series_1_aluminum_42mm-8334.php"
    ],
    [
      "Watch Series 2 Aluminum 38mm",
      "https://www.gsmarena.com/apple_watch_series_2_aluminum_38mm-8327.php"
    ],
    [
      "Watch Series 1 Aluminum 38mm",
      "https://www.gsmarena.com/apple_watch_series_1_aluminum_38mm-8333.php"
    ],
    [
      "iPhone 7 Plus",
      "https://www.gsmarena.com/apple_iphone_7_plus-8065.php"
    ],
    [
      "iPhone 7",
      "https://www.gsmarena.com/apple_iphone_7-8064.php"
    ],
    [
      "iPad Pro 9.7 (2016)",
      "https://www.gsmarena.com/apple_ipad_pro_9_7_(2016)-7984.php"
    ],
    [
      "iPhone SE",
      "https://www.gsmarena.com/apple_iphone_se-7969.php"
    ],
    [
      "iPhone 6s Plus",
      "https://www.gsmarena.com/apple_iphone_6s_plus-7243.php"
    ],
    [
      "iPhone 6s",
      "https://www.gsmarena.com/apple_iphone_6s-7242.php"
    ],
    [
      "iPad Pro 12.9 (2015)",
      "https://www.gsmarena.com/apple_ipad_pro_12_9_(2015)-7562.php"
    ],
    [
      "iPad mini 4 (2015)",
      "https://www.gsmarena.com/apple_ipad_mini_4_(2015)-7561.php"
    ],
    [
      "Watch Edition 42mm (1st gen)",
      "https://www.gsmarena.com/apple_watch_edition_42mm_(1st_gen)-7698.php"
    ],
    [
      "Watch Edition 38mm (1st gen)",
      "https://www.gsmarena.com/apple_watch_edition_38mm_(1st_gen)-7697.php"
    ],
    [
      "Watch 42mm (1st gen)",
      "https://www.gsmarena.com/apple_watch_42mm_(1st_gen)-7696.php"
    ],
    [
      "Watch 38mm (1st gen)",
      "https://www.gsmarena.com/apple_watch_38mm_(1st_gen)-7695.php"
    ],
    [
      "Watch Sport 42mm (1st gen)",
      "https://www.gsmarena.com/apple_watch_sport_42mm_(1st_gen)-7694.php"
    ],
    [
      "Watch Sport 38mm (1st gen)",
      "https://www.gsmarena.com/apple_watch_sport_38mm_(1st_gen)-7693.php"
    ],
    [
      "iPad Air 2",
      "https://www.gsmarena.com/apple_ipad_air_2-6742.php"
    ],
    [
      "iPad mini 3",
      "https://www.gsmarena.com/apple_ipad_mini_3-6741.php"
    ],
    [
      "iPhone 6 Plus",
      "https://www.gsmarena.com/apple_iphone_6_plus-6665.php"
    ],
    [
      "iPhone 6",
      "https://www.gsmarena.com/apple_iphone_6-6378.php"
    ],
    [
      "iPad Air",
      "https://www.gsmarena.com/apple_ipad_air-5797.php"
    ],
    [
      "iPad mini 2",
      "https://www.gsmarena.com/apple_ipad_mini_2-5735.php"
    ],
    [
      "iPhone 5s",
      "https://www.gsmarena.com/apple_iphone_5s-5685.php"
    ],
    [
      "iPhone 5c",
      "https://www.gsmarena.com/apple_iphone_5c-5690.php"
    ],
    [
      "iPad mini Wi-Fi",
      "https://www.gsmarena.com/apple_ipad_mini_wi_fi-5070.php"
    ],
    [
      "iPad mini Wi-Fi + Cellular",
      "https://www.gsmarena.com/apple_ipad_mini_wi_fi_+_cellular-5061.php"
    ],
    [
      "iPad 4 Wi-Fi",
      "https://www.gsmarena.com/apple_ipad_4_wi_fi-5072.php"
    ],
    [
      "iPad 4 Wi-Fi + Cellular",
      "https://www.gsmarena.com/apple_ipad_4_wi_fi_+_cellular-5071.php"
    ],
    [
      "iPhone 5",
      "https://www.gsmarena.com/apple_iphone_5-4910.php"
    ],
    [
      "iPad 3 Wi-Fi + Cellular",
      "https://www.gsmarena.com/apple_ipad_3_wi_fi_+_cellular-4620.php"
    ],
    [
      "iPad 3 Wi-Fi",
      "https://www.gsmarena.com/apple_ipad_3_wi_fi-4621.php"
    ],
    [
      "iPhone 4s",
      "https://www.gsmarena.com/apple_iphone_4s-4212.php"
    ],
    [
      "iPad 2 Wi-Fi + 3G",
      "https://www.gsmarena.com/apple_ipad_2_wi_fi_+_3g-3848.php"
    ],
    [
      "iPad 2 Wi-Fi",
      "https://www.gsmarena.com/apple_ipad_2_wi_fi-3847.php"
    ],
    [
      "iPad 2 CDMA",
      "https://www.gsmarena.com/apple_ipad_2_cdma-3849.php"
    ],
    [
      "iPhone 4",
      "https://www.gsmarena.com/apple_iphone_4-3275.php"
    ],
    [
      "iPhone 4 CDMA",
      "https://www.gsmarena.com/apple_iphone_4_cdma-3716.php"
    ],
    [
      "iPad Wi-Fi + 3G",
      "https://www.gsmarena.com/apple_ipad_wi_fi_+_3g-3827.php"
    ],
    [
      "iPad Wi-Fi",
      "https://www.gsmarena.com/apple_ipad_wi_fi-3828.php"
    ],
    [
      "iPhone 3GS",
      "https://www.gsmarena.com/apple_iphone_3gs-2826.php"
    ],
    [
      "iPhone 3G",
      "https://www.gsmarena.com/apple_iphone_3g-2424.php"
    ],
    [
      "iPhone",
      "https://www.gsmarena.com/apple_iphone-1827.php"
    ]
  ],
  "honor": [
    [
      "Magic8 RSR Porsche Design",
      "https://www.gsmarena.com/honor_magic8_rsr_porsche_design_5g-14429.php"
    ],
    [
      "Magic8 Pro Air",
      "https://www.gsmarena.com/honor_magic8_pro_air_5g-14405.php"
    ],
    [
      "Watch GS 5",
      "https://www.gsmarena.com/honor_watch_gs_5-14431.php"
    ],
    [
      "Power2",
      "https://www.gsmarena.com/honor_power_2_5g-14364.php"
    ],
    [
      "Win",
      "https://www.gsmarena.com/honor_win_5g-14377.php"
    ],
    [
      "Win RT",
      "https://www.gsmarena.com/honor_win_rt_5g-14376.php"
    ],
    [
      "Play 60A",
      "https://www.gsmarena.com/honor_play_60a_5g-14357.php"
    ],
    [
      "X8d",
      "https://www.gsmarena.com/honor_x8d-14351.php"
    ],
    [
      "Magic8 Lite",
      "https://www.gsmarena.com/honor_magic8_lite_5g-14341.php"
    ],
    [
      "Watch X5",
      "https://www.gsmarena.com/honor_watch_x5-14317.php"
    ],
    [
      "500 Pro (China)",
      "https://www.gsmarena.com/honor_500_pro_5g_(china)-14294.php"
    ],
    [
      "500",
      "https://www.gsmarena.com/honor_500_5g-14296.php"
    ],
    [
      "Magic8 Pro",
      "https://www.gsmarena.com/honor_magic8_pro_5g-14231.php"
    ],
    [
      "Magic8",
      "https://www.gsmarena.com/honor_magic8_5g-14232.php"
    ],
    [
      "Watch 5 Pro",
      "https://www.gsmarena.com/honor_watch_5_pro-14242.php"
    ],
    [
      "MagicPad 3 Pro",
      "https://www.gsmarena.com/honor_magicpad_3_pro-14243.php"
    ],
    [
      "Play10A",
      "https://www.gsmarena.com/honor_play10a-14253.php"
    ],
    [
      "X5c",
      "https://www.gsmarena.com/honor_x5c-14251.php"
    ],
    [
      "X5c Plus",
      "https://www.gsmarena.com/honor_x5c_plus-14156.php"
    ],
    [
      "X6b",
      "https://www.gsmarena.com/honor_x6b_5g-14195.php"
    ],
    [
      "X9d",
      "https://www.gsmarena.com/honor_x9d_5g-14176.php"
    ],
    [
      "X7d",
      "https://www.gsmarena.com/honor_x7d_5g-14175.php"
    ],
    [
      "400 Smart 4G",
      "https://www.gsmarena.com/honor_400_smart-14142.php"
    ],
    [
      "Play10",
      "https://www.gsmarena.com/honor_play10-14126.php"
    ],
    [
      "X7d 4G",
      "https://www.gsmarena.com/honor_x7d-14093.php"
    ],
    [
      "Magic V Flip 2",
      "https://www.gsmarena.com/honor_magic_v_flip_2_5g-14080.php"
    ],
    [
      "X7c",
      "https://www.gsmarena.com/honor_x7c_5g-14065.php"
    ],
    [
      "Play10C",
      "https://www.gsmarena.com/honor_play10c_5g-14060.php"
    ],
    [
      "400 Smart",
      "https://www.gsmarena.com/honor_400_smart_5g-14054.php"
    ],
    [
      "Pad X7",
      "https://www.gsmarena.com/honor_pad_x7-14026.php"
    ],
    [
      "X70",
      "https://www.gsmarena.com/honor_x70_5g-13994.php"
    ],
    [
      "Pad GT2 Pro",
      "https://www.gsmarena.com/honor_pad_gt2_pro-14015.php"
    ],
    [
      "MagicPad 3",
      "https://www.gsmarena.com/honor_magicpad_3-14016.php"
    ],
    [
      "Magic V5",
      "https://www.gsmarena.com/honor_magic_v5-13982.php"
    ],
    [
      "X6c",
      "https://www.gsmarena.com/honor_x6c-13938.php"
    ],
    [
      "400 Pro (China)",
      "https://www.gsmarena.com/honor_400_pro_5g_(china)-13929.php"
    ],
    [
      "400 (China)",
      "https://www.gsmarena.com/honor_400_5g_(china)-13930.php"
    ],
    [
      "400 Pro",
      "https://www.gsmarena.com/honor_400_pro_5g-13798.php"
    ],
    [
      "400",
      "https://www.gsmarena.com/honor_400_5g-13799.php"
    ],
    [
      "Pad 10",
      "https://www.gsmarena.com/honor_pad_10-13897.php"
    ],
    [
      "X70i",
      "https://www.gsmarena.com/honor_x70i_5g-13816.php"
    ],
    [
      "GT Pro",
      "https://www.gsmarena.com/honor_gt_pro_5g-13810.php"
    ],
    [
      "Pad GT",
      "https://www.gsmarena.com/honor_tablet_gt-13814.php"
    ],
    [
      "X60 GT",
      "https://www.gsmarena.com/honor_x60_gt_5g-13807.php"
    ],
    [
      "Power",
      "https://www.gsmarena.com/honor_power_5g-13794.php"
    ],
    [
      "400 Lite",
      "https://www.gsmarena.com/honor_400_lite-13742.php"
    ],
    [
      "Play 60",
      "https://www.gsmarena.com/honor_play_60_5g-13803.php"
    ],
    [
      "Pad X9a",
      "https://www.gsmarena.com/honor_pad_x9a-13751.php"
    ],
    [
      "Play9A",
      "https://www.gsmarena.com/honor_play9a-13743.php"
    ],
    [
      "Watch 5 Ultra",
      "https://www.gsmarena.com/honor_watch_5_ultra-13685.php"
    ],
    [
      "X8c",
      "https://www.gsmarena.com/honor_x8c-13621.php"
    ],
    [
      "Magic7 Lite",
      "https://www.gsmarena.com/honor_magic7_lite-13581.php"
    ],
    [
      "Magic7 RSR Porsche Design",
      "https://www.gsmarena.com/honor_magic7_rsr_porsche_design-13588.php"
    ],
    [
      "GT",
      "https://www.gsmarena.com/honor_gt-13569.php"
    ],
    [
      "Pad V9",
      "https://www.gsmarena.com/honor_pad_v9-13568.php"
    ],
    [
      "300 Ultra",
      "https://www.gsmarena.com/honor_300_ultra-13537.php"
    ],
    [
      "300 Pro",
      "https://www.gsmarena.com/honor_300_pro-13538.php"
    ],
    [
      "300",
      "https://www.gsmarena.com/honor_300-13539.php"
    ],
    [
      "X9c Smart",
      "https://www.gsmarena.com/honor_x9c_smart-13536.php"
    ],
    [
      "Play9C (China)",
      "https://www.gsmarena.com/honor_play9c-13493.php"
    ],
    [
      "Play9T",
      "https://www.gsmarena.com/honor_play9t-13494.php"
    ],
    [
      "X9c",
      "https://www.gsmarena.com/honor_x9c-13490.php"
    ],
    [
      "Magic7 Pro",
      "https://www.gsmarena.com/honor_magic7_pro-13480.php"
    ],
    [
      "Magic7",
      "https://www.gsmarena.com/honor_magic7-13481.php"
    ],
    [
      "X5b Plus",
      "https://www.gsmarena.com/honor_x5b_plus-13464.php"
    ],
    [
      "X5b",
      "https://www.gsmarena.com/honor_x5b-13463.php"
    ],
    [
      "X7c 4G",
      "https://www.gsmarena.com/honor_x7c-13416.php"
    ],
    [
      "X60 Pro",
      "https://www.gsmarena.com/honor_x60_pro-13443.php"
    ],
    [
      "X60",
      "https://www.gsmarena.com/honor_x60-13415.php"
    ],
    [
      "Pad GT Pro",
      "https://www.gsmarena.com/honor_pad_gt_pro-13417.php"
    ],
    [
      "200 Smart",
      "https://www.gsmarena.com/honor_200_smart-13281.php"
    ],
    [
      "Watch 5",
      "https://www.gsmarena.com/honor_watch_5-13307.php"
    ],
    [
      "Pad X8a",
      "https://www.gsmarena.com/honor_pad_x8a-13259.php"
    ],
    [
      "X60i",
      "https://www.gsmarena.com/honor_x60i-13208.php"
    ],
    [
      "MagicPad 2 12.3",
      "https://www.gsmarena.com/honor_magicpad_2_12_3-13205.php"
    ],
    [
      "Pad 9 Pro",
      "https://www.gsmarena.com/honor_pad_9_pro-13204.php"
    ],
    [
      "Magic Vs3",
      "https://www.gsmarena.com/honor_magic_vs3-13203.php"
    ],
    [
      "Magic V3",
      "https://www.gsmarena.com/honor_magic_v3-13202.php"
    ],
    [
      "Play 60 Plus",
      "https://www.gsmarena.com/honor_play_60_plus-13161.php"
    ],
    [
      "X7b 5G (50 MP)",
      "https://www.gsmarena.com/honor_x7b_5g_(50_mp)-13154.php"
    ],
    [
      "Magic V Flip",
      "https://www.gsmarena.com/honor_magic_v_flip-13142.php"
    ],
    [
      "X6b 4G",
      "https://www.gsmarena.com/honor_x6b-13143.php"
    ],
    [
      "200 Pro",
      "https://www.gsmarena.com/honor_200_pro-13049.php"
    ],
    [
      "200",
      "https://www.gsmarena.com/honor_200-13050.php"
    ],
    [
      "200 Lite",
      "https://www.gsmarena.com/honor_200_lite-12948.php"
    ],
    [
      "90 Smart",
      "https://www.gsmarena.com/honor_90_smart-12931.php"
    ],
    [
      "X7b 5G",
      "https://www.gsmarena.com/honor_x7b_5g-12913.php"
    ],
    [
      "Magic6 RSR Porsche Design",
      "https://www.gsmarena.com/honor_magic6_rsr_porsche_design-12873.php"
    ],
    [
      "Magic6 Ultimate",
      "https://www.gsmarena.com/honor_magic6_ultimate-12864.php"
    ],
    [
      "Watch GS 4",
      "https://www.gsmarena.com/honor_watch_gs_4-12879.php"
    ],
    [
      "Choice Watch",
      "https://www.gsmarena.com/honor_choice_watch-12832.php"
    ],
    [
      "Magic V2 RSR Porsche Design",
      "https://www.gsmarena.com/honor_magic_v2_rsr_porsche_design-12789.php"
    ],
    [
      "Magic6 Pro",
      "https://www.gsmarena.com/honor_magic6_pro-12786.php"
    ],
    [
      "Magic6",
      "https://www.gsmarena.com/honor_magic6-12788.php"
    ],
    [
      "X50 GT",
      "https://www.gsmarena.com/honor_x50_gt-12777.php"
    ],
    [
      "X50 Pro",
      "https://www.gsmarena.com/honor_x50_pro-12763.php"
    ],
    [
      "90 GT",
      "https://www.gsmarena.com/honor_90_gt-12755.php"
    ],
    [
      "Pad 9",
      "https://www.gsmarena.com/honor_pad_9-12756.php"
    ],
    [
      "X8b",
      "https://www.gsmarena.com/honor_x8b-12748.php"
    ],
    [
      "Magic6 Lite",
      "https://www.gsmarena.com/honor_magic6_lite-12730.php"
    ],
    [
      "X7b",
      "https://www.gsmarena.com/honor_x7b-12721.php"
    ],
    [
      "100 Pro",
      "https://www.gsmarena.com/honor_100_pro-12699.php"
    ],
    [
      "100",
      "https://www.gsmarena.com/honor_100-12700.php"
    ],
    [
      "X50i+",
      "https://www.gsmarena.com/honor_x50i+-12693.php"
    ],
    [
      "X5 Plus",
      "https://www.gsmarena.com/honor_x5_plus-12645.php"
    ],
    [
      "X9b",
      "https://www.gsmarena.com/honor_x9b-12629.php"
    ],
    [
      "Play 8T",
      "https://www.gsmarena.com/honor_play_8t-12628.php"
    ],
    [
      "Magic Vs2",
      "https://www.gsmarena.com/honor_magic_vs2-12622.php"
    ],
    [
      "Watch 4 Pro",
      "https://www.gsmarena.com/honor_watch_4_pro-12623.php"
    ],
    [
      "Play 50 Plus",
      "https://www.gsmarena.com/honor_play_50_plus-12618.php"
    ],
    [
      "V Purse",
      "https://www.gsmarena.com/honor_v_purse-12570.php"
    ],
    [
      "X40 GT Racing",
      "https://www.gsmarena.com/honor_x40_gt_racing-12569.php"
    ],
    [
      "X6a",
      "https://www.gsmarena.com/honor_x6a-12442.php"
    ],
    [
      "Magic V2",
      "https://www.gsmarena.com/honor_magic_v2-12383.php"
    ],
    [
      "Watch 4",
      "https://www.gsmarena.com/honor_watch_4-12416.php"
    ],
    [
      "MagicPad 13",
      "https://www.gsmarena.com/honor_magicpad_13-12417.php"
    ],
    [
      "Pad X9",
      "https://www.gsmarena.com/honor_pad_x9-12404.php"
    ],
    [
      "Pad X8 Pro",
      "https://www.gsmarena.com/honor_pad_x8_pro-12403.php"
    ],
    [
      "X50",
      "https://www.gsmarena.com/honor_x50-12384.php"
    ],
    [
      "Play 40",
      "https://www.gsmarena.com/honor_play_40-12401.php"
    ],
    [
      "X50i",
      "https://www.gsmarena.com/honor_x50i-12400.php"
    ],
    [
      "90 Lite",
      "https://www.gsmarena.com/honor_90_lite-12377.php"
    ],
    [
      "90 Pro",
      "https://www.gsmarena.com/honor_90_pro-12296.php"
    ],
    [
      "90",
      "https://www.gsmarena.com/honor_90-12297.php"
    ],
    [
      "Pad V8",
      "https://www.gsmarena.com/honor_pad_v8-12248.php"
    ],
    [
      "Play7T Pro",
      "https://www.gsmarena.com/honor_play7t_pro-12200.php"
    ],
    [
      "Play7T",
      "https://www.gsmarena.com/honor_play7t-12201.php"
    ],
    [
      "70 Lite",
      "https://www.gsmarena.com/honor_70_lite-12180.php"
    ],
    [
      "Magic5 Ultimate",
      "https://www.gsmarena.com/honor_magic5_ultimate-12155.php"
    ],
    [
      "Magic5 Pro",
      "https://www.gsmarena.com/honor_magic5_pro-12148.php"
    ],
    [
      "Magic5",
      "https://www.gsmarena.com/honor_magic5-12149.php"
    ],
    [
      "Magic5 Lite",
      "https://www.gsmarena.com/honor_magic5_lite-12107.php"
    ],
    [
      "X8a",
      "https://www.gsmarena.com/honor_x8a-12113.php"
    ],
    [
      "X5",
      "https://www.gsmarena.com/honor_x5-12093.php"
    ],
    [
      "X9a",
      "https://www.gsmarena.com/honor_x9a-12075.php"
    ],
    [
      "X7a",
      "https://www.gsmarena.com/honor_x7a-12076.php"
    ],
    [
      "80 Pro Flat",
      "https://www.gsmarena.com/honor_80_pro_flat-12065.php"
    ],
    [
      "80 GT",
      "https://www.gsmarena.com/honor_80_gt-12043.php"
    ],
    [
      "Pad V8 Pro",
      "https://www.gsmarena.com/honor_pad_v8_pro-12044.php"
    ],
    [
      "80 Pro",
      "https://www.gsmarena.com/honor_80_pro-11992.php"
    ],
    [
      "80",
      "https://www.gsmarena.com/honor_80-11993.php"
    ],
    [
      "80 SE",
      "https://www.gsmarena.com/honor_80_se-11995.php"
    ],
    [
      "Magic Vs Ultimate",
      "https://www.gsmarena.com/honor_magic_vs_ultimate-11994.php"
    ],
    [
      "Magic Vs",
      "https://www.gsmarena.com/honor_magic_vs-11991.php"
    ],
    [
      "Play 40 Plus",
      "https://www.gsmarena.com/honor_play_40_plus-11943.php"
    ],
    [
      "X40 GT",
      "https://www.gsmarena.com/honor_x40_gt-11933.php"
    ],
    [
      "Play6C",
      "https://www.gsmarena.com/honor_play6c-11932.php"
    ],
    [
      "X6",
      "https://www.gsmarena.com/honor_x6-11894.php"
    ],
    [
      "Pad X8",
      "https://www.gsmarena.com/honor_pad_x8-11889.php"
    ],
    [
      "Pad X8 Lite",
      "https://www.gsmarena.com/honor_pad_x8_lite-11890.php"
    ],
    [
      "X40",
      "https://www.gsmarena.com/honor_x40-11875.php"
    ],
    [
      "X8 5G",
      "https://www.gsmarena.com/honor_x8_5g-11718.php"
    ],
    [
      "Pad 8",
      "https://www.gsmarena.com/honor_pad_8-11712.php"
    ],
    [
      "X40i",
      "https://www.gsmarena.com/honor_x40i-11671.php"
    ],
    [
      "70 Pro+",
      "https://www.gsmarena.com/honor_70_pro+-11577.php"
    ],
    [
      "70 Pro",
      "https://www.gsmarena.com/honor_70_pro-11576.php"
    ],
    [
      "70",
      "https://www.gsmarena.com/honor_70-11575.php"
    ],
    [
      "Play 30",
      "https://www.gsmarena.com/honor_play_30-11520.php"
    ],
    [
      "Play6T Pro",
      "https://www.gsmarena.com/honor_play6t_pro-11463.php"
    ],
    [
      "Play6T",
      "https://www.gsmarena.com/honor_play6t-11464.php"
    ],
    [
      "Magic4 Lite",
      "https://www.gsmarena.com/honor_magic4_lite-11423.php"
    ],
    [
      "X7",
      "https://www.gsmarena.com/honor_x7-11452.php"
    ],
    [
      "X9",
      "https://www.gsmarena.com/honor_x9-11456.php"
    ],
    [
      "X9 5G",
      "https://www.gsmarena.com/honor_x9_5g-11447.php"
    ],
    [
      "Magic4 Ultimate",
      "https://www.gsmarena.com/honor_magic4_ultimate-11432.php"
    ],
    [
      "X8",
      "https://www.gsmarena.com/honor_x8-11417.php"
    ],
    [
      "Magic4 Pro",
      "https://www.gsmarena.com/honor_magic4_pro-11390.php"
    ],
    [
      "Magic4",
      "https://www.gsmarena.com/honor_magic4-11389.php"
    ],
    [
      "60 SE",
      "https://www.gsmarena.com/honor_60_se-11354.php"
    ],
    [
      "Watch GS 3",
      "https://www.gsmarena.com/honor_watch_gs_3-11312.php"
    ],
    [
      "Magic V",
      "https://www.gsmarena.com/honor_magic_v-11311.php"
    ],
    [
      "X30",
      "https://www.gsmarena.com/honor_x30-11270.php"
    ],
    [
      "Play 30 Plus",
      "https://www.gsmarena.com/honor_play_30_plus-11271.php"
    ],
    [
      "60 Pro",
      "https://www.gsmarena.com/honor_60_pro-11248.php"
    ],
    [
      "60",
      "https://www.gsmarena.com/honor_60-11255.php"
    ],
    [
      "X30 Max",
      "https://www.gsmarena.com/honor_x30_max-11178.php"
    ],
    [
      "X30i",
      "https://www.gsmarena.com/honor_x30i-11179.php"
    ],
    [
      "50 Lite",
      "https://www.gsmarena.com/honor_50_lite-11175.php"
    ],
    [
      "Play5 Youth",
      "https://www.gsmarena.com/honor_play5_youth-11173.php"
    ],
    [
      "Tablet V7",
      "https://www.gsmarena.com/honor_tablet_v7-11125.php"
    ],
    [
      "Tablet V7 Pro",
      "https://www.gsmarena.com/honor_tablet_v7_pro-11055.php"
    ],
    [
      "X20",
      "https://www.gsmarena.com/honor_x20-11052.php"
    ],
    [
      "Magic3 Pro+",
      "https://www.gsmarena.com/honor_magic3_pro+-11051.php"
    ],
    [
      "Magic3 Pro",
      "https://www.gsmarena.com/honor_magic3_pro-11050.php"
    ],
    [
      "Magic3",
      "https://www.gsmarena.com/honor_magic3-11049.php"
    ],
    [
      "Play 5T Pro",
      "https://www.gsmarena.com/honor_play_5t_pro-11038.php"
    ],
    [
      "X20 SE",
      "https://www.gsmarena.com/honor_x20_se-10995.php"
    ],
    [
      "50 Pro",
      "https://www.gsmarena.com/honor_50_pro-10958.php"
    ],
    [
      "50",
      "https://www.gsmarena.com/honor_50-10962.php"
    ],
    [
      "50 SE",
      "https://www.gsmarena.com/honor_50_se-10963.php"
    ],
    [
      "Play5 5G",
      "https://www.gsmarena.com/honor_play5_5g-10899.php"
    ],
    [
      "Tablet X7",
      "https://www.gsmarena.com/honor_tablet_x7-10912.php"
    ],
    [
      "Play 20",
      "https://www.gsmarena.com/honor_play_20-10876.php"
    ],
    [
      "Play 5T Youth",
      "https://www.gsmarena.com/honor_play_5t_youth-10863.php"
    ],
    [
      "Tab 7",
      "https://www.gsmarena.com/honor_tab_7-10813.php"
    ],
    [
      "V40 Lite",
      "https://www.gsmarena.com/honor_v40_lite-10804.php"
    ],
    [
      "View40",
      "https://www.gsmarena.com/honor_view40-10701.php"
    ],
    [
      "V40 5G",
      "https://www.gsmarena.com/honor_v40_5g-10687.php"
    ],
    [
      "10X Lite",
      "https://www.gsmarena.com/honor_10x_lite-10565.php"
    ],
    [
      "30i",
      "https://www.gsmarena.com/honor_30i-10438.php"
    ],
    [
      "Watch GS Pro",
      "https://www.gsmarena.com/honor_watch_gs_pro-10422.php"
    ],
    [
      "Watch ES",
      "https://www.gsmarena.com/honor_watch_es-10423.php"
    ],
    [
      "Pad 6",
      "https://www.gsmarena.com/honor_pad_6-10424.php"
    ],
    [
      "Pad X6",
      "https://www.gsmarena.com/honor_pad_x6-10425.php"
    ],
    [
      "X10 Max 5G",
      "https://www.gsmarena.com/honor_x10_max_5g-10306.php"
    ],
    [
      "30 Youth",
      "https://www.gsmarena.com/honor_30_youth-10304.php"
    ],
    [
      "8S 2020",
      "https://www.gsmarena.com/honor_8s_2020-10218.php"
    ],
    [
      "Play4",
      "https://www.gsmarena.com/honor_play4-10283.php"
    ],
    [
      "Play4 Pro",
      "https://www.gsmarena.com/honor_play4_pro-10258.php"
    ],
    [
      "X10 5G",
      "https://www.gsmarena.com/honor_x10_5g-10228.php"
    ],
    [
      "V6",
      "https://www.gsmarena.com/honor_v6-10253.php"
    ],
    [
      "9A",
      "https://www.gsmarena.com/honor_9a-10215.php"
    ],
    [
      "9C",
      "https://www.gsmarena.com/honor_9c-10214.php"
    ],
    [
      "9S",
      "https://www.gsmarena.com/honor_9s-10213.php"
    ],
    [
      "9X Lite",
      "https://www.gsmarena.com/honor_9x_lite-10085.php"
    ],
    [
      "30 Pro+",
      "https://www.gsmarena.com/honor_30_pro+-10187.php"
    ],
    [
      "30 Pro",
      "https://www.gsmarena.com/honor_30_pro-10181.php"
    ],
    [
      "30",
      "https://www.gsmarena.com/honor_30-10188.php"
    ],
    [
      "20e",
      "https://www.gsmarena.com/honor_20e-10191.php"
    ],
    [
      "Play 4T Pro",
      "https://www.gsmarena.com/honor_play_4t_pro-10168.php"
    ],
    [
      "Play 4T",
      "https://www.gsmarena.com/honor_play_4t-10167.php"
    ],
    [
      "8A 2020",
      "https://www.gsmarena.com/honor_8a_2020-10182.php"
    ],
    [
      "8A Prime",
      "https://www.gsmarena.com/honor_8a_prime-10163.php"
    ],
    [
      "30S",
      "https://www.gsmarena.com/honor_30s-10162.php"
    ],
    [
      "Play 9A",
      "https://www.gsmarena.com/honor_play_9a-10155.php"
    ],
    [
      "View30 Pro",
      "https://www.gsmarena.com/honor_view30_pro-10099.php"
    ],
    [
      "View30",
      "https://www.gsmarena.com/honor_view30-10101.php"
    ],
    [
      "V30 Pro",
      "https://www.gsmarena.com/honor_v30_pro-9976.php"
    ],
    [
      "V30",
      "https://www.gsmarena.com/honor_v30-9977.php"
    ],
    [
      "MagicWatch 2",
      "https://www.gsmarena.com/honor_magicwatch_2-9974.php"
    ],
    [
      "9X",
      "https://www.gsmarena.com/honor_9x-9931.php"
    ],
    [
      "20 lite (China)",
      "https://www.gsmarena.com/honor_20_lite_(china)-9914.php"
    ],
    [
      "Play 3e",
      "https://www.gsmarena.com/honor_play_3e-9878.php"
    ],
    [
      "Play 3",
      "https://www.gsmarena.com/honor_play_3-9829.php"
    ],
    [
      "20S",
      "https://www.gsmarena.com/honor_20s-9832.php"
    ],
    [
      "9X Pro",
      "https://www.gsmarena.com/honor_9x_pro-9772.php"
    ],
    [
      "9X (China)",
      "https://www.gsmarena.com/honor_9x_(china)-9771.php"
    ],
    [
      "8S",
      "https://www.gsmarena.com/honor_8s-9664.php"
    ],
    [
      "Pad 5 10.1",
      "https://www.gsmarena.com/honor_pad_5_10_1-9740.php"
    ],
    [
      "Pad 5 8",
      "https://www.gsmarena.com/honor_pad_5_8-9741.php"
    ],
    [
      "20 Pro",
      "https://www.gsmarena.com/honor_20_pro-9710.php"
    ],
    [
      "20",
      "https://www.gsmarena.com/honor_20-9615.php"
    ],
    [
      "20 lite",
      "https://www.gsmarena.com/honor_20_lite-9641.php"
    ],
    [
      "20i",
      "https://www.gsmarena.com/honor_20i-9663.php"
    ],
    [
      "8A Pro",
      "https://www.gsmarena.com/honor_8a_pro-9662.php"
    ],
    [
      "Tab 5",
      "https://www.gsmarena.com/honor_tab_5-9619.php"
    ],
    [
      "View 20",
      "https://www.gsmarena.com/honor_view_20-9468.php"
    ],
    [
      "Magic 2 3D",
      "https://www.gsmarena.com/honor_magic_2_3d-9628.php"
    ],
    [
      "Magic 2",
      "https://www.gsmarena.com/honor_magic_2-9381.php"
    ],
    [
      "Play 8A",
      "https://www.gsmarena.com/honor_play_8a-9509.php"
    ],
    [
      "10 Lite",
      "https://www.gsmarena.com/honor_10_lite-9403.php"
    ],
    [
      "8C",
      "https://www.gsmarena.com/honor_8c-9366.php"
    ],
    [
      "8X",
      "https://www.gsmarena.com/honor_8x-9313.php"
    ],
    [
      "8X Max",
      "https://www.gsmarena.com/honor_8x_max-9306.php"
    ],
    [
      "Note 10",
      "https://www.gsmarena.com/honor_note_10-8799.php"
    ],
    [
      "9N (9i)",
      "https://www.gsmarena.com/honor_9n_(9i)-9229.php"
    ],
    [
      "Play",
      "https://www.gsmarena.com/honor_play-9230.php"
    ],
    [
      "7S",
      "https://www.gsmarena.com/honor_7s-9211.php"
    ],
    [
      "10",
      "https://www.gsmarena.com/honor_10-9157.php"
    ],
    [
      "7A",
      "https://www.gsmarena.com/honor_7a-9075.php"
    ],
    [
      "7C",
      "https://www.gsmarena.com/honor_7c-9111.php"
    ],
    [
      "9 Lite",
      "https://www.gsmarena.com/honor_9_lite-8962.php"
    ],
    [
      "View 10",
      "https://www.gsmarena.com/honor_view_10-8938.php"
    ],
    [
      "7X",
      "https://www.gsmarena.com/honor_7x-8880.php"
    ],
    [
      "6C Pro",
      "https://www.gsmarena.com/honor_6c_pro-8856.php"
    ],
    [
      "9",
      "https://www.gsmarena.com/honor_9-8704.php"
    ],
    [
      "6A (Pro)",
      "https://www.gsmarena.com/honor_6a_(pro)-8700.php"
    ],
    [
      "8 Pro",
      "https://www.gsmarena.com/honor_8_pro-8568.php"
    ],
    [
      "Magic",
      "https://www.gsmarena.com/honor_magic-8482.php"
    ],
    [
      "V8",
      "https://www.gsmarena.com/honor_v8-8061.php"
    ],
    [
      "Pad 2",
      "https://www.gsmarena.com/honor_pad_2-8390.php"
    ],
    [
      "6X",
      "https://www.gsmarena.com/honor_6x-8388.php"
    ],
    [
      "Holly 3",
      "https://www.gsmarena.com/honor_holly_3-8383.php"
    ],
    [
      "Note 8",
      "https://www.gsmarena.com/honor_note_8-8231.php"
    ],
    [
      "8",
      "https://www.gsmarena.com/honor_8-8195.php"
    ],
    [
      "5A",
      "https://www.gsmarena.com/honor_5a-8149.php"
    ],
    [
      "5c",
      "https://www.gsmarena.com/honor_5c-8074.php"
    ],
    [
      "Holly 2 Plus",
      "https://www.gsmarena.com/honor_holly_2_plus-7882.php"
    ],
    [
      "5X",
      "https://www.gsmarena.com/honor_5x-7590.php"
    ],
    [
      "7i",
      "https://www.gsmarena.com/honor_7i-7510.php"
    ],
    [
      "7",
      "https://www.gsmarena.com/honor_7-7269.php"
    ],
    [
      "Bee",
      "https://www.gsmarena.com/honor_bee-7235.php"
    ],
    [
      "4C",
      "https://www.gsmarena.com/honor_4c-7203.php"
    ],
    [
      "6 Plus",
      "https://www.gsmarena.com/honor_6_plus-6777.php"
    ],
    [
      "4X",
      "https://www.gsmarena.com/honor_4x-6754.php"
    ],
    [
      "Holly",
      "https://www.gsmarena.com/honor_holly-6738.php"
    ],
    [
      "4 Play",
      "https://www.gsmarena.com/honor_4_play-6692.php"
    ],
    [
      "3C Play",
      "https://www.gsmarena.com/honor_3c_play-6603.php"
    ],
    [
      "6",
      "https://www.gsmarena.com/honor_6-6461.php"
    ],
    [
      "3X Pro",
      "https://www.gsmarena.com/honor_3x_pro-6398.php"
    ],
    [
      "3C 4G",
      "https://www.gsmarena.com/honor_3c_4g-6395.php"
    ],
    [
      "3X G750",
      "https://www.gsmarena.com/honor_3x_g750-5902.php"
    ],
    [
      "3C",
      "https://www.gsmarena.com/honor_3c-5903.php"
    ],
    [
      "3",
      "https://www.gsmarena.com/honor_3-5664.php"
    ],
    [
      "2",
      "https://www.gsmarena.com/honor_2-5092.php"
    ],
    [
      "U8860",
      "https://www.gsmarena.com/honor_u8860-4197.php"
    ],
    [
      "View",
      "https://www.gsmarena.com/honor_view-9970.php"
    ]
  ],
  "samsung": [
    [
      "Galaxy F70e",
      "https://www.gsmarena.com/samsung_galaxy_f70e_5g-14459.php"
    ],
    [
      "Galaxy A07",
      "https://www.gsmarena.com/samsung_galaxy_a07_5g-14409.php"
    ],
    [
      "Galaxy Z TriFold",
      "https://www.gsmarena.com/samsung_galaxy_z_trifold_5g-14292.php"
    ],
    [
      "Galaxy M17",
      "https://www.gsmarena.com/samsung_galaxy_m17_5g-14221.php"
    ],
    [
      "Galaxy F07",
      "https://www.gsmarena.com/samsung_galaxy_f07-14205.php"
    ],
    [
      "Galaxy M07",
      "https://www.gsmarena.com/samsung_galaxy_m07-14100.php"
    ],
    [
      "Galaxy A17 4G",
      "https://www.gsmarena.com/samsung_galaxy_a17-14157.php"
    ],
    [
      "Galaxy Tab A11+",
      "https://www.gsmarena.com/samsung_galaxy_tab_a11+-14192.php"
    ],
    [
      "Galaxy Tab A11",
      "https://www.gsmarena.com/samsung_galaxy_tab_a11-14141.php"
    ],
    [
      "Galaxy F17",
      "https://www.gsmarena.com/samsung_galaxy_f17_5g-14107.php"
    ],
    [
      "Galaxy S25 FE",
      "https://www.gsmarena.com/samsung_galaxy_s25_fe_5g-14042.php"
    ],
    [
      "Galaxy Tab S11 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_tab_s11_ultra_5g-14057.php"
    ],
    [
      "Galaxy Tab S11",
      "https://www.gsmarena.com/samsung_galaxy_tab_s11_5g-14058.php"
    ],
    [
      "Galaxy Tab S10 Lite",
      "https://www.gsmarena.com/samsung_galaxy_tab_s10_lite_5g-14059.php"
    ],
    [
      "Galaxy A07 4G",
      "https://www.gsmarena.com/samsung_galaxy_a07-14066.php"
    ],
    [
      "Galaxy A17",
      "https://www.gsmarena.com/samsung_galaxy_a17_5g-14041.php"
    ],
    [
      "Galaxy F36",
      "https://www.gsmarena.com/samsung_galaxy_f36_5g-14009.php"
    ],
    [
      "Galaxy Z Fold7",
      "https://www.gsmarena.com/samsung_galaxy_z_fold7-13826.php"
    ],
    [
      "Galaxy Z Flip7",
      "https://www.gsmarena.com/samsung_galaxy_z_flip7-13712.php"
    ],
    [
      "Galaxy Z Flip7 FE",
      "https://www.gsmarena.com/samsung_galaxy_z_flip7_fe_5g-13844.php"
    ],
    [
      "Galaxy Watch8 Classic",
      "https://www.gsmarena.com/samsung_galaxy_watch8_classic-13998.php"
    ],
    [
      "Galaxy Watch8",
      "https://www.gsmarena.com/samsung_galaxy_watch8-13997.php"
    ],
    [
      "Galaxy M36",
      "https://www.gsmarena.com/samsung_galaxy_m36_5g-13967.php"
    ],
    [
      "Galaxy S25 Edge",
      "https://www.gsmarena.com/samsung_galaxy_s25_edge-13506.php"
    ],
    [
      "Galaxy F56",
      "https://www.gsmarena.com/samsung_galaxy_f56_5g-13855.php"
    ],
    [
      "Galaxy M56",
      "https://www.gsmarena.com/samsung_galaxy_m56_5g-13801.php"
    ],
    [
      "Galaxy XCover7 Pro",
      "https://www.gsmarena.com/samsung_galaxy_xcover7_pro-13780.php"
    ],
    [
      "Galaxy Tab Active5 Pro",
      "https://www.gsmarena.com/samsung_galaxy_tab_active5_pro-13790.php"
    ],
    [
      "Galaxy Tab S10 FE+",
      "https://www.gsmarena.com/samsung_galaxy_tab_s10_fe+-13760.php"
    ],
    [
      "Galaxy Tab S10 FE",
      "https://www.gsmarena.com/samsung_galaxy_tab_s10_fe-13761.php"
    ],
    [
      "Galaxy F16",
      "https://www.gsmarena.com/samsung_galaxy_f16-13721.php"
    ],
    [
      "Galaxy A56",
      "https://www.gsmarena.com/samsung_galaxy_a56-13603.php"
    ],
    [
      "Galaxy A36",
      "https://www.gsmarena.com/samsung_galaxy_a36-13497.php"
    ],
    [
      "Galaxy A26",
      "https://www.gsmarena.com/samsung_galaxy_a26-13679.php"
    ],
    [
      "Galaxy M16",
      "https://www.gsmarena.com/samsung_galaxy_m16-13680.php"
    ],
    [
      "Galaxy M06",
      "https://www.gsmarena.com/samsung_galaxy_m06-13681.php"
    ],
    [
      "Galaxy A06 5G",
      "https://www.gsmarena.com/samsung_galaxy_a06_5g-13662.php"
    ],
    [
      "Galaxy F06 5G",
      "https://www.gsmarena.com/samsung_galaxy_f06_5g-13663.php"
    ],
    [
      "Galaxy S25 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_s25_ultra-13322.php"
    ],
    [
      "Galaxy S25+",
      "https://www.gsmarena.com/samsung_galaxy_s25+-13609.php"
    ],
    [
      "Galaxy S25",
      "https://www.gsmarena.com/samsung_galaxy_s25-13610.php"
    ],
    [
      "Galaxy Z Fold Special",
      "https://www.gsmarena.com/samsung_galaxy_z_fold_special-13452.php"
    ],
    [
      "Galaxy A16",
      "https://www.gsmarena.com/samsung_galaxy_a16-13383.php"
    ],
    [
      "Galaxy A16 5G",
      "https://www.gsmarena.com/samsung_galaxy_a16_5g-13346.php"
    ],
    [
      "Galaxy S24 FE",
      "https://www.gsmarena.com/samsung_galaxy_s24_fe-13262.php"
    ],
    [
      "Galaxy Tab S10 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_tab_s10_ultra-13362.php"
    ],
    [
      "Galaxy Tab S10+",
      "https://www.gsmarena.com/samsung_galaxy_tab_s10+-13363.php"
    ],
    [
      "Galaxy M55s",
      "https://www.gsmarena.com/samsung_galaxy_m55s-13354.php"
    ],
    [
      "Galaxy F05",
      "https://www.gsmarena.com/samsung_galaxy_f05-13345.php"
    ],
    [
      "Galaxy M05",
      "https://www.gsmarena.com/samsung_galaxy_m05-13327.php"
    ],
    [
      "Galaxy A06",
      "https://www.gsmarena.com/samsung_galaxy_a06-13265.php"
    ],
    [
      "Galaxy F14 4G",
      "https://www.gsmarena.com/samsung_galaxy_f14_4g-13247.php"
    ],
    [
      "Galaxy Z Fold6",
      "https://www.gsmarena.com/samsung_galaxy_z_fold6-13147.php"
    ],
    [
      "Galaxy Z Flip6",
      "https://www.gsmarena.com/samsung_galaxy_z_flip6-13192.php"
    ],
    [
      "Galaxy Watch Ultra",
      "https://www.gsmarena.com/samsung_galaxy_watch_ultra-13127.php"
    ],
    [
      "Galaxy Watch7",
      "https://www.gsmarena.com/samsung_galaxy_watch7-13128.php"
    ],
    [
      "Galaxy Watch FE",
      "https://www.gsmarena.com/samsung_galaxy_watch_fe-13112.php"
    ],
    [
      "Galaxy M35",
      "https://www.gsmarena.com/samsung_galaxy_m35-13006.php"
    ],
    [
      "Galaxy F55",
      "https://www.gsmarena.com/samsung_galaxy_f55-12885.php"
    ],
    [
      "Galaxy C55",
      "https://www.gsmarena.com/samsung_galaxy_c55-12945.php"
    ],
    [
      "Galaxy M55",
      "https://www.gsmarena.com/samsung_galaxy_m55-12896.php"
    ],
    [
      "Galaxy Tab S6 Lite (2024)",
      "https://www.gsmarena.com/samsung_galaxy_tab_s6_lite_(2024)-12897.php"
    ],
    [
      "Galaxy A55",
      "https://www.gsmarena.com/samsung_galaxy_a55-12824.php"
    ],
    [
      "Galaxy A35",
      "https://www.gsmarena.com/samsung_galaxy_a35-12705.php"
    ],
    [
      "Galaxy M15",
      "https://www.gsmarena.com/samsung_galaxy_m15-12833.php"
    ],
    [
      "Galaxy M14 4G",
      "https://www.gsmarena.com/samsung_galaxy_m14_4g-12862.php"
    ],
    [
      "Galaxy F15",
      "https://www.gsmarena.com/samsung_galaxy_f15-12831.php"
    ],
    [
      "Galaxy S24 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_s24_ultra-12771.php"
    ],
    [
      "Galaxy S24+",
      "https://www.gsmarena.com/samsung_galaxy_s24+-12772.php"
    ],
    [
      "Galaxy S24",
      "https://www.gsmarena.com/samsung_galaxy_s24-12773.php"
    ],
    [
      "Galaxy XCover7",
      "https://www.gsmarena.com/samsung_galaxy_xcover7-12784.php"
    ],
    [
      "Galaxy Tab Active5",
      "https://www.gsmarena.com/samsung_galaxy_tab_active5-12785.php"
    ],
    [
      "Galaxy A25",
      "https://www.gsmarena.com/samsung_galaxy_a25-12555.php"
    ],
    [
      "Galaxy A15 5G",
      "https://www.gsmarena.com/samsung_galaxy_a15_5g-12638.php"
    ],
    [
      "Galaxy A15",
      "https://www.gsmarena.com/samsung_galaxy_a15-12637.php"
    ],
    [
      "Galaxy Tab A9+",
      "https://www.gsmarena.com/samsung_galaxy_tab_a9+-12617.php"
    ],
    [
      "Galaxy Tab A9",
      "https://www.gsmarena.com/samsung_galaxy_tab_a9-12616.php"
    ],
    [
      "Galaxy S23 FE",
      "https://www.gsmarena.com/samsung_galaxy_s23_fe-12520.php"
    ],
    [
      "Galaxy Tab S9 FE+",
      "https://www.gsmarena.com/samsung_galaxy_tab_s9_fe+-12609.php"
    ],
    [
      "Galaxy Tab S9 FE",
      "https://www.gsmarena.com/samsung_galaxy_tab_s9_fe-12517.php"
    ],
    [
      "Galaxy A05s",
      "https://www.gsmarena.com/samsung_galaxy_a05s-12584.php"
    ],
    [
      "Galaxy A05",
      "https://www.gsmarena.com/samsung_galaxy_a05-12583.php"
    ],
    [
      "Galaxy F34",
      "https://www.gsmarena.com/samsung_galaxy_f34-12453.php"
    ],
    [
      "Galaxy Z Fold5",
      "https://www.gsmarena.com/samsung_galaxy_z_fold5-12418.php"
    ],
    [
      "Galaxy Z Flip5",
      "https://www.gsmarena.com/samsung_galaxy_z_flip5-12252.php"
    ],
    [
      "Galaxy Tab S9 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_tab_s9_ultra-12217.php"
    ],
    [
      "Galaxy Tab S9+",
      "https://www.gsmarena.com/samsung_galaxy_tab_s9+-12440.php"
    ],
    [
      "Galaxy Tab S9",
      "https://www.gsmarena.com/samsung_galaxy_tab_s9-12439.php"
    ],
    [
      "Galaxy Watch6 Classic",
      "https://www.gsmarena.com/samsung_galaxy_watch6_classic-12438.php"
    ],
    [
      "Galaxy Watch6",
      "https://www.gsmarena.com/samsung_galaxy_watch6-12437.php"
    ],
    [
      "Galaxy M34 5G",
      "https://www.gsmarena.com/samsung_galaxy_m34_5g-11290.php"
    ],
    [
      "Galaxy F54",
      "https://www.gsmarena.com/samsung_galaxy_f54-12239.php"
    ],
    [
      "Galaxy A24 4G",
      "https://www.gsmarena.com/samsung_galaxy_a24_4g-12176.php"
    ],
    [
      "Galaxy F14",
      "https://www.gsmarena.com/samsung_galaxy_f14-12175.php"
    ],
    [
      "Galaxy M54",
      "https://www.gsmarena.com/samsung_galaxy_m54-12189.php"
    ],
    [
      "Galaxy A54",
      "https://www.gsmarena.com/samsung_galaxy_a54-12070.php"
    ],
    [
      "Galaxy A34",
      "https://www.gsmarena.com/samsung_galaxy_a34-12074.php"
    ],
    [
      "Galaxy M14",
      "https://www.gsmarena.com/samsung_galaxy_m14-12160.php"
    ],
    [
      "Galaxy S23 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_s23_ultra-12024.php"
    ],
    [
      "Galaxy S23+",
      "https://www.gsmarena.com/samsung_galaxy_s23+-12083.php"
    ],
    [
      "Galaxy S23",
      "https://www.gsmarena.com/samsung_galaxy_s23-12082.php"
    ],
    [
      "Galaxy A14",
      "https://www.gsmarena.com/samsung_galaxy_a14-12151.php"
    ],
    [
      "Galaxy A14 5G",
      "https://www.gsmarena.com/samsung_galaxy_a14_5g-12004.php"
    ],
    [
      "Galaxy F04",
      "https://www.gsmarena.com/samsung_galaxy_f04-12054.php"
    ],
    [
      "Galaxy M04",
      "https://www.gsmarena.com/samsung_galaxy_m04-12022.php"
    ],
    [
      "Galaxy Tab A7 10.4 (2022)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a7_10_4_(2022)-11988.php"
    ],
    [
      "Galaxy A04e",
      "https://www.gsmarena.com/samsung_galaxy_a04e-11945.php"
    ],
    [
      "Galaxy Tab Active4 Pro",
      "https://www.gsmarena.com/samsung_galaxy_tab_active4_pro-11840.php"
    ],
    [
      "Galaxy A04s",
      "https://www.gsmarena.com/samsung_galaxy_a04s-11803.php"
    ],
    [
      "Galaxy A04",
      "https://www.gsmarena.com/samsung_galaxy_a04-11817.php"
    ],
    [
      "Galaxy Z Fold4",
      "https://www.gsmarena.com/samsung_galaxy_z_fold4-11737.php"
    ],
    [
      "Galaxy Z Flip4",
      "https://www.gsmarena.com/samsung_galaxy_z_flip4-11538.php"
    ],
    [
      "Galaxy Watch5 Pro",
      "https://www.gsmarena.com/samsung_galaxy_watch5_pro-11749.php"
    ],
    [
      "Galaxy Watch5",
      "https://www.gsmarena.com/samsung_galaxy_watch5-11748.php"
    ],
    [
      "Galaxy A23 5G",
      "https://www.gsmarena.com/samsung_galaxy_a23_5g-11736.php"
    ],
    [
      "Galaxy M13 5G",
      "https://www.gsmarena.com/samsung_galaxy_m13_5g-11633.php"
    ],
    [
      "Galaxy M13 (India)",
      "https://www.gsmarena.com/samsung_galaxy_m13_(india)-11654.php"
    ],
    [
      "Galaxy A13 (SM-A137)",
      "https://www.gsmarena.com/samsung_galaxy_a13_(sm_a137)-11665.php"
    ],
    [
      "Galaxy XCover6 Pro",
      "https://www.gsmarena.com/samsung_galaxy_xcover6_pro-11600.php"
    ],
    [
      "Galaxy F13",
      "https://www.gsmarena.com/samsung_galaxy_f13-11624.php"
    ],
    [
      "Galaxy M13",
      "https://www.gsmarena.com/samsung_galaxy_m13-11564.php"
    ],
    [
      "Galaxy Tab S6 Lite (2022)",
      "https://www.gsmarena.com/samsung_galaxy_tab_s6_lite_(2022)-11524.php"
    ],
    [
      "Galaxy M53",
      "https://www.gsmarena.com/samsung_galaxy_m53-11439.php"
    ],
    [
      "Galaxy S20 FE 2022",
      "https://www.gsmarena.com/samsung_galaxy_s20_fe_2022-11459.php"
    ],
    [
      "Galaxy A73 5G",
      "https://www.gsmarena.com/samsung_galaxy_a73_5g-11257.php"
    ],
    [
      "Galaxy A53 5G",
      "https://www.gsmarena.com/samsung_galaxy_a53_5g-11268.php"
    ],
    [
      "Galaxy A33 5G",
      "https://www.gsmarena.com/samsung_galaxy_a33_5g-11429.php"
    ],
    [
      "Galaxy F23",
      "https://www.gsmarena.com/samsung_galaxy_f23-11406.php"
    ],
    [
      "Galaxy M33",
      "https://www.gsmarena.com/samsung_galaxy_m33-11404.php"
    ],
    [
      "Galaxy M23",
      "https://www.gsmarena.com/samsung_galaxy_m23-11403.php"
    ],
    [
      "Galaxy A23",
      "https://www.gsmarena.com/samsung_galaxy_a23-11373.php"
    ],
    [
      "Galaxy A13",
      "https://www.gsmarena.com/samsung_galaxy_a13-11402.php"
    ],
    [
      "Galaxy S22 Ultra 5G",
      "https://www.gsmarena.com/samsung_galaxy_s22_ultra_5g-11251.php"
    ],
    [
      "Galaxy S22+ 5G",
      "https://www.gsmarena.com/samsung_galaxy_s22+_5g-11252.php"
    ],
    [
      "Galaxy S22 5G",
      "https://www.gsmarena.com/samsung_galaxy_s22_5g-11253.php"
    ],
    [
      "Galaxy Tab S8 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_tab_s8_ultra-11274.php"
    ],
    [
      "Galaxy Tab S8+",
      "https://www.gsmarena.com/samsung_galaxy_tab_s8+-11342.php"
    ],
    [
      "Galaxy Tab S8",
      "https://www.gsmarena.com/samsung_galaxy_tab_s8-11343.php"
    ],
    [
      "Galaxy S21 FE 5G",
      "https://www.gsmarena.com/samsung_galaxy_s21_fe_5g-10954.php"
    ],
    [
      "Galaxy Tab A8 10.5 (2021)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a8_10_5_(2021)-11265.php"
    ],
    [
      "Galaxy A13 5G",
      "https://www.gsmarena.com/samsung_galaxy_a13_5g-11149.php"
    ],
    [
      "Galaxy A03",
      "https://www.gsmarena.com/samsung_galaxy_a03-11244.php"
    ],
    [
      "Galaxy A03 Core",
      "https://www.gsmarena.com/samsung_galaxy_a03_core-11215.php"
    ],
    [
      "Galaxy F42 5G",
      "https://www.gsmarena.com/samsung_galaxy_f42_5g-11124.php"
    ],
    [
      "Galaxy M52 5G",
      "https://www.gsmarena.com/samsung_galaxy_m52_5g-11110.php"
    ],
    [
      "Galaxy M22",
      "https://www.gsmarena.com/samsung_galaxy_m22-11011.php"
    ],
    [
      "Galaxy M32 5G",
      "https://www.gsmarena.com/samsung_galaxy_m32_5g-11062.php"
    ],
    [
      "Galaxy A03s",
      "https://www.gsmarena.com/samsung_galaxy_a03s-10937.php"
    ],
    [
      "Galaxy A52s 5G",
      "https://www.gsmarena.com/samsung_galaxy_a52s_5g-11039.php"
    ],
    [
      "Galaxy Z Fold3 5G",
      "https://www.gsmarena.com/samsung_galaxy_z_fold3_5g-10906.php"
    ],
    [
      "Galaxy Z Flip3 5G",
      "https://www.gsmarena.com/samsung_galaxy_z_flip3_5g-11044.php"
    ],
    [
      "Galaxy Watch4 Classic",
      "https://www.gsmarena.com/samsung_galaxy_watch4_classic-11046.php"
    ],
    [
      "Galaxy Watch4",
      "https://www.gsmarena.com/samsung_galaxy_watch4-11045.php"
    ],
    [
      "Galaxy A12 (India)",
      "https://www.gsmarena.com/samsung_galaxy_a12_(india)-11087.php"
    ],
    [
      "Galaxy A12 Nacho",
      "https://www.gsmarena.com/samsung_galaxy_a12_nacho-11041.php"
    ],
    [
      "Galaxy M21 2021",
      "https://www.gsmarena.com/samsung_galaxy_m21_2021-11015.php"
    ],
    [
      "Galaxy F22",
      "https://www.gsmarena.com/samsung_galaxy_f22-10996.php"
    ],
    [
      "Galaxy M32",
      "https://www.gsmarena.com/samsung_galaxy_m32-10887.php"
    ],
    [
      "Galaxy A22 5G",
      "https://www.gsmarena.com/samsung_galaxy_a22_5g-10873.php"
    ],
    [
      "Galaxy A22",
      "https://www.gsmarena.com/samsung_galaxy_a22-10948.php"
    ],
    [
      "Galaxy Tab A7 Lite",
      "https://www.gsmarena.com/samsung_galaxy_tab_a7_lite-10933.php"
    ],
    [
      "Galaxy Tab S7 FE",
      "https://www.gsmarena.com/samsung_galaxy_tab_s7_fe-10922.php"
    ],
    [
      "Galaxy F52 5G",
      "https://www.gsmarena.com/samsung_galaxy_f52_5g-10900.php"
    ],
    [
      "Galaxy M42 5G",
      "https://www.gsmarena.com/samsung_galaxy_m42_5g-10848.php"
    ],
    [
      "Galaxy M12",
      "https://www.gsmarena.com/samsung_galaxy_m12-10860.php"
    ],
    [
      "Galaxy Quantum 2",
      "https://www.gsmarena.com/samsung_galaxy_quantum_2-10850.php"
    ],
    [
      "Galaxy F12",
      "https://www.gsmarena.com/samsung_galaxy_f12-10825.php"
    ],
    [
      "Galaxy F02s",
      "https://www.gsmarena.com/samsung_galaxy_f02s-10824.php"
    ],
    [
      "Galaxy A72",
      "https://www.gsmarena.com/samsung_galaxy_a72-10469.php"
    ],
    [
      "Galaxy A52 5G",
      "https://www.gsmarena.com/samsung_galaxy_a52_5g-10631.php"
    ],
    [
      "Galaxy A52",
      "https://www.gsmarena.com/samsung_galaxy_a52-10641.php"
    ],
    [
      "Galaxy XCover 5",
      "https://www.gsmarena.com/samsung_galaxy_xcover_5-10718.php"
    ],
    [
      "Galaxy A32",
      "https://www.gsmarena.com/samsung_galaxy_a32-10753.php"
    ],
    [
      "Galaxy M62",
      "https://www.gsmarena.com/samsung_galaxy_m62-10749.php"
    ],
    [
      "Galaxy F62",
      "https://www.gsmarena.com/samsung_galaxy_f62-10740.php"
    ],
    [
      "Galaxy M12 (India)",
      "https://www.gsmarena.com/samsung_galaxy_m12_(india)-10578.php"
    ],
    [
      "Galaxy S21 Ultra 5G",
      "https://www.gsmarena.com/samsung_galaxy_s21_ultra_5g-10596.php"
    ],
    [
      "Galaxy S21+ 5G",
      "https://www.gsmarena.com/samsung_galaxy_s21+_5g-10625.php"
    ],
    [
      "Galaxy S21 5G",
      "https://www.gsmarena.com/samsung_galaxy_s21_5g-10626.php"
    ],
    [
      "Galaxy A32 5G",
      "https://www.gsmarena.com/samsung_galaxy_a32_5g-10648.php"
    ],
    [
      "Galaxy M02s",
      "https://www.gsmarena.com/samsung_galaxy_m02s-10663.php"
    ],
    [
      "Galaxy A12",
      "https://www.gsmarena.com/samsung_galaxy_a12-10604.php"
    ],
    [
      "Galaxy M02",
      "https://www.gsmarena.com/samsung_galaxy_m02-10709.php"
    ],
    [
      "Galaxy A02",
      "https://www.gsmarena.com/samsung_galaxy_a02-10708.php"
    ],
    [
      "Galaxy A02s",
      "https://www.gsmarena.com/samsung_galaxy_a02s-10603.php"
    ],
    [
      "Galaxy M21s",
      "https://www.gsmarena.com/samsung_galaxy_m21s-10580.php"
    ],
    [
      "Galaxy M31 Prime",
      "https://www.gsmarena.com/samsung_galaxy_m31_prime-10496.php"
    ],
    [
      "Galaxy F41",
      "https://www.gsmarena.com/samsung_galaxy_f41-10498.php"
    ],
    [
      "Galaxy Tab Active3",
      "https://www.gsmarena.com/samsung_galaxy_tab_active3-10476.php"
    ],
    [
      "Galaxy S20 FE 5G",
      "https://www.gsmarena.com/samsung_galaxy_s20_fe_5g-10377.php"
    ],
    [
      "Galaxy S20 FE",
      "https://www.gsmarena.com/samsung_galaxy_s20_fe-10428.php"
    ],
    [
      "Galaxy A42 5G",
      "https://www.gsmarena.com/samsung_galaxy_a42_5g-10412.php"
    ],
    [
      "Galaxy Tab A7 10.4 (2020)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a7_10_4_(2020)-10411.php"
    ],
    [
      "Galaxy M51",
      "https://www.gsmarena.com/samsung_galaxy_m51-10148.php"
    ],
    [
      "Galaxy A51 5G UW",
      "https://www.gsmarena.com/samsung_galaxy_a51_5g_uw-10371.php"
    ],
    [
      "Galaxy Z Fold2 5G",
      "https://www.gsmarena.com/samsung_galaxy_z_fold2_5g-10342.php"
    ],
    [
      "Galaxy Note20 Ultra 5G",
      "https://www.gsmarena.com/samsung_galaxy_note20_ultra_5g-10261.php"
    ],
    [
      "Galaxy Note20 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_note20_ultra-10355.php"
    ],
    [
      "Galaxy Note20 5G",
      "https://www.gsmarena.com/samsung_galaxy_note20_5g-10325.php"
    ],
    [
      "Galaxy Note20",
      "https://www.gsmarena.com/samsung_galaxy_note20-10338.php"
    ],
    [
      "Galaxy Watch3",
      "https://www.gsmarena.com/samsung_galaxy_watch3-10315.php"
    ],
    [
      "Galaxy Tab S7+",
      "https://www.gsmarena.com/samsung_galaxy_tab_s7+-10336.php"
    ],
    [
      "Galaxy Tab S7",
      "https://www.gsmarena.com/samsung_galaxy_tab_s7-10337.php"
    ],
    [
      "Galaxy Z Flip 5G",
      "https://www.gsmarena.com/samsung_galaxy_z_flip_5g-10244.php"
    ],
    [
      "Galaxy M31s",
      "https://www.gsmarena.com/samsung_galaxy_m31s-10333.php"
    ],
    [
      "Galaxy M01s",
      "https://www.gsmarena.com/samsung_galaxy_m01s-10335.php"
    ],
    [
      "Galaxy M01 Core",
      "https://www.gsmarena.com/samsung_galaxy_m01_core-10316.php"
    ],
    [
      "Galaxy A01 Core",
      "https://www.gsmarena.com/samsung_galaxy_a01_core-10314.php"
    ],
    [
      "Galaxy A71 5G UW",
      "https://www.gsmarena.com/samsung_galaxy_a71_5g_uw-10276.php"
    ],
    [
      "Galaxy M01",
      "https://www.gsmarena.com/samsung_galaxy_m01-10216.php"
    ],
    [
      "Galaxy A21s",
      "https://www.gsmarena.com/samsung_galaxy_a21s-10239.php"
    ],
    [
      "Galaxy J2 Core (2020)",
      "https://www.gsmarena.com/samsung_galaxy_j2_core_(2020)-10208.php"
    ],
    [
      "Galaxy A Quantum",
      "https://www.gsmarena.com/samsung_galaxy_a_quantum-10243.php"
    ],
    [
      "Galaxy A71 5G",
      "https://www.gsmarena.com/samsung_galaxy_a71_5g-10146.php"
    ],
    [
      "Galaxy A51 5G",
      "https://www.gsmarena.com/samsung_galaxy_a51_5g-10157.php"
    ],
    [
      "Galaxy A21",
      "https://www.gsmarena.com/samsung_galaxy_a21-10172.php"
    ],
    [
      "Galaxy Tab A 8.4 (2020)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_8_4_(2020)-10483.php"
    ],
    [
      "Galaxy Tab S6 Lite (2020)",
      "https://www.gsmarena.com/samsung_galaxy_tab_s6_lite_(2020)-10158.php"
    ],
    [
      "Galaxy M11",
      "https://www.gsmarena.com/samsung_galaxy_m11-10124.php"
    ],
    [
      "Galaxy A31",
      "https://www.gsmarena.com/samsung_galaxy_a31-10149.php"
    ],
    [
      "Galaxy A41",
      "https://www.gsmarena.com/samsung_galaxy_a41-10138.php"
    ],
    [
      "Galaxy M21",
      "https://www.gsmarena.com/samsung_galaxy_m21-10111.php"
    ],
    [
      "Galaxy A11",
      "https://www.gsmarena.com/samsung_galaxy_a11-10132.php"
    ],
    [
      "Galaxy M31",
      "https://www.gsmarena.com/samsung_galaxy_m31-10079.php"
    ],
    [
      "Galaxy S20 Ultra 5G",
      "https://www.gsmarena.com/samsung_galaxy_s20_ultra_5g-10040.php"
    ],
    [
      "Galaxy S20 Ultra",
      "https://www.gsmarena.com/samsung_galaxy_s20_ultra-10084.php"
    ],
    [
      "Galaxy S20+ 5G",
      "https://www.gsmarena.com/samsung_galaxy_s20+_5g-10041.php"
    ],
    [
      "Galaxy S20+",
      "https://www.gsmarena.com/samsung_galaxy_s20+-10080.php"
    ],
    [
      "Galaxy S20 5G UW",
      "https://www.gsmarena.com/samsung_galaxy_s20_5g_uw-10254.php"
    ],
    [
      "Galaxy S20 5G",
      "https://www.gsmarena.com/samsung_galaxy_s20_5g-10044.php"
    ],
    [
      "Galaxy S20",
      "https://www.gsmarena.com/samsung_galaxy_s20-10081.php"
    ],
    [
      "Galaxy Z Flip",
      "https://www.gsmarena.com/samsung_galaxy_z_flip-10054.php"
    ],
    [
      "Galaxy Tab S6 5G",
      "https://www.gsmarena.com/samsung_galaxy_tab_s6_5g-10004.php"
    ],
    [
      "Galaxy XCover Pro",
      "https://www.gsmarena.com/samsung_galaxy_xcover_pro-10001.php"
    ],
    [
      "Galaxy Note10 Lite",
      "https://www.gsmarena.com/samsung_galaxy_note10_lite-10003.php"
    ],
    [
      "Galaxy S10 Lite",
      "https://www.gsmarena.com/samsung_galaxy_s10_lite-9917.php"
    ],
    [
      "Galaxy A01",
      "https://www.gsmarena.com/samsung_galaxy_a01-9999.php"
    ],
    [
      "Galaxy A71",
      "https://www.gsmarena.com/samsung_galaxy_a71-9995.php"
    ],
    [
      "Galaxy A51",
      "https://www.gsmarena.com/samsung_galaxy_a51-9963.php"
    ],
    [
      "Galaxy XCover FieldPro",
      "https://www.gsmarena.com/samsung_galaxy_xcover_fieldpro-9937.php"
    ],
    [
      "Galaxy A70s",
      "https://www.gsmarena.com/samsung_galaxy_a70s-9899.php"
    ],
    [
      "Galaxy A20s",
      "https://www.gsmarena.com/samsung_galaxy_a20s-9852.php"
    ],
    [
      "Galaxy M30s",
      "https://www.gsmarena.com/samsung_galaxy_m30s-9818.php"
    ],
    [
      "Galaxy M10s",
      "https://www.gsmarena.com/samsung_galaxy_m10s-9861.php"
    ],
    [
      "Galaxy Fold 5G",
      "https://www.gsmarena.com/samsung_galaxy_fold_5g-9851.php"
    ],
    [
      "Galaxy Fold",
      "https://www.gsmarena.com/samsung_galaxy_fold-9523.php"
    ],
    [
      "Galaxy Tab Active Pro",
      "https://www.gsmarena.com/samsung_galaxy_tab_active_pro-9850.php"
    ],
    [
      "Galaxy A90 5G",
      "https://www.gsmarena.com/samsung_galaxy_a90_5g-9795.php"
    ],
    [
      "Galaxy A30s",
      "https://www.gsmarena.com/samsung_galaxy_a30s-9796.php"
    ],
    [
      "Galaxy A50s",
      "https://www.gsmarena.com/samsung_galaxy_a50s-9806.php"
    ],
    [
      "Galaxy Note10+ 5G",
      "https://www.gsmarena.com/samsung_galaxy_note10+_5g-9787.php"
    ],
    [
      "Galaxy Note10+",
      "https://www.gsmarena.com/samsung_galaxy_note10+-9732.php"
    ],
    [
      "Galaxy Note10 5G",
      "https://www.gsmarena.com/samsung_galaxy_note10_5g-9789.php"
    ],
    [
      "Galaxy Note10",
      "https://www.gsmarena.com/samsung_galaxy_note10-9788.php"
    ],
    [
      "Galaxy Watch Active2",
      "https://www.gsmarena.com/samsung_galaxy_watch_active2-9784.php"
    ],
    [
      "Galaxy Watch Active2 Aluminum",
      "https://www.gsmarena.com/samsung_galaxy_watch_active2_aluminum-9786.php"
    ],
    [
      "Galaxy A10s",
      "https://www.gsmarena.com/samsung_galaxy_a10s-9793.php"
    ],
    [
      "Galaxy A10e",
      "https://www.gsmarena.com/samsung_galaxy_a10e-9790.php"
    ],
    [
      "Galaxy Tab S6",
      "https://www.gsmarena.com/samsung_galaxy_tab_s6-9781.php"
    ],
    [
      "Galaxy Tab A 8.0 (2019)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_8_0_(2019)-9760.php"
    ],
    [
      "Galaxy XCover 4s",
      "https://www.gsmarena.com/samsung_galaxy_xcover_4s-9730.php"
    ],
    [
      "Galaxy A2 Core",
      "https://www.gsmarena.com/samsung_galaxy_a2_core-9636.php"
    ],
    [
      "Galaxy Watch Active",
      "https://www.gsmarena.com/samsung_galaxy_watch_active-9589.php"
    ],
    [
      "Galaxy View2",
      "https://www.gsmarena.com/samsung_galaxy_view2-9672.php"
    ],
    [
      "Galaxy S10 5G",
      "https://www.gsmarena.com/samsung_galaxy_s10_5g-9588.php"
    ],
    [
      "Galaxy S10+",
      "https://www.gsmarena.com/samsung_galaxy_s10+-9535.php"
    ],
    [
      "Galaxy S10",
      "https://www.gsmarena.com/samsung_galaxy_s10-9536.php"
    ],
    [
      "Galaxy S10e",
      "https://www.gsmarena.com/samsung_galaxy_s10e-9537.php"
    ],
    [
      "Galaxy M40",
      "https://www.gsmarena.com/samsung_galaxy_m40-9721.php"
    ],
    [
      "Galaxy M30",
      "https://www.gsmarena.com/samsung_galaxy_m30-9505.php"
    ],
    [
      "Galaxy M20",
      "https://www.gsmarena.com/samsung_galaxy_m20-9506.php"
    ],
    [
      "Galaxy M10",
      "https://www.gsmarena.com/samsung_galaxy_m10-9508.php"
    ],
    [
      "Galaxy A80",
      "https://www.gsmarena.com/samsung_galaxy_a80-9659.php"
    ],
    [
      "Galaxy A70",
      "https://www.gsmarena.com/samsung_galaxy_a70-9646.php"
    ],
    [
      "Galaxy A60",
      "https://www.gsmarena.com/samsung_galaxy_a60-9616.php"
    ],
    [
      "Galaxy A50",
      "https://www.gsmarena.com/samsung_galaxy_a50-9554.php"
    ],
    [
      "Galaxy A40",
      "https://www.gsmarena.com/samsung_galaxy_a40-9642.php"
    ],
    [
      "Galaxy A30",
      "https://www.gsmarena.com/samsung_galaxy_a30-9579.php"
    ],
    [
      "Galaxy A20e",
      "https://www.gsmarena.com/samsung_galaxy_a20e-9661.php"
    ],
    [
      "Galaxy A20",
      "https://www.gsmarena.com/samsung_galaxy_a20-9640.php"
    ],
    [
      "Galaxy A10",
      "https://www.gsmarena.com/samsung_galaxy_a10-9580.php"
    ],
    [
      "Galaxy Tab S5e",
      "https://www.gsmarena.com/samsung_galaxy_tab_s5e-9581.php"
    ],
    [
      "Galaxy Tab A 10.1 (2019)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_10_1_(2019)-9582.php"
    ],
    [
      "Galaxy Tab A 8.0 & S Pen (2019)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_8_0_&_s_pen_(2019)-9651.php"
    ],
    [
      "Galaxy Tab Advanced2",
      "https://www.gsmarena.com/samsung_galaxy_tab_advanced2-9264.php"
    ],
    [
      "Galaxy Tab A 8.0 (2018)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_8_0_(2018)-9665.php"
    ],
    [
      "Galaxy Tab S4 10.5",
      "https://www.gsmarena.com/samsung_galaxy_tab_s4_10_5-9262.php"
    ],
    [
      "Galaxy Tab A 10.5",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_10_5-9263.php"
    ],
    [
      "Galaxy A8s",
      "https://www.gsmarena.com/samsung_galaxy_a8s-9432.php"
    ],
    [
      "Galaxy A6s (2018)",
      "https://www.gsmarena.com/samsung_galaxy_a6s-9352.php"
    ],
    [
      "Galaxy A9 (2018)",
      "https://www.gsmarena.com/samsung_galaxy_a9_(2018)-9344.php"
    ],
    [
      "Galaxy A7 (2018)",
      "https://www.gsmarena.com/samsung_galaxy_a7_(2018)-9340.php"
    ],
    [
      "Galaxy Note9",
      "https://www.gsmarena.com/samsung_galaxy_note9-9163.php"
    ],
    [
      "Galaxy Watch",
      "https://www.gsmarena.com/samsung_galaxy_watch-9289.php"
    ],
    [
      "Galaxy J6+",
      "https://www.gsmarena.com/samsung_galaxy_j6+-9334.php"
    ],
    [
      "Galaxy J4 Core",
      "https://www.gsmarena.com/samsung_galaxy_j4_core-9397.php"
    ],
    [
      "Galaxy J4+",
      "https://www.gsmarena.com/samsung_galaxy_j4+-9335.php"
    ],
    [
      "Galaxy J2 Core",
      "https://www.gsmarena.com/samsung_galaxy_j2_core-9255.php"
    ],
    [
      "Galaxy On6",
      "https://www.gsmarena.com/samsung_galaxy_on6-9260.php"
    ],
    [
      "Galaxy J7 (2018)",
      "https://www.gsmarena.com/samsung_galaxy_j7_(2018)-9234.php"
    ],
    [
      "Galaxy J3 (2018)",
      "https://www.gsmarena.com/samsung_galaxy_j3_(2018)-8928.php"
    ],
    [
      "Galaxy A8 Star (A9 Star)",
      "https://www.gsmarena.com/samsung_galaxy_a8_star_(a9_star)-9220.php"
    ],
    [
      "Galaxy S Light Luxury",
      "https://www.gsmarena.com/samsung_galaxy_s_light_luxury-9202.php"
    ],
    [
      "Galaxy J8",
      "https://www.gsmarena.com/samsung_galaxy_j8-9205.php"
    ],
    [
      "Galaxy J6",
      "https://www.gsmarena.com/samsung_galaxy_j6-9203.php"
    ],
    [
      "Galaxy J4",
      "https://www.gsmarena.com/samsung_galaxy_j4-9204.php"
    ],
    [
      "Galaxy A6+ (2018)",
      "https://www.gsmarena.com/samsung_galaxy_a6+_(2018)-9156.php"
    ],
    [
      "Galaxy A6 (2018)",
      "https://www.gsmarena.com/samsung_galaxy_a6_(2018)-9155.php"
    ],
    [
      "Galaxy J7 Duo",
      "https://www.gsmarena.com/samsung_galaxy_j7_duo-9153.php"
    ],
    [
      "Galaxy J7 Prime 2",
      "https://www.gsmarena.com/samsung_galaxy_j7_prime_2-9135.php"
    ],
    [
      "Galaxy S9+",
      "https://www.gsmarena.com/samsung_galaxy_s9+-8967.php"
    ],
    [
      "Galaxy S9",
      "https://www.gsmarena.com/samsung_galaxy_s9-8966.php"
    ],
    [
      "Galaxy J2 Pro (2018)",
      "https://www.gsmarena.com/samsung_galaxy_j2_pro_(2018)-8904.php"
    ],
    [
      "Galaxy A8+ (2018)",
      "https://www.gsmarena.com/samsung_galaxy_a8+_(2018)-8790.php"
    ],
    [
      "Galaxy A8 (2018)",
      "https://www.gsmarena.com/samsung_galaxy_a8_(2018)-8886.php"
    ],
    [
      "Galaxy J2 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_j2_(2017)-8900.php"
    ],
    [
      "Galaxy Tab Active2",
      "https://www.gsmarena.com/samsung_galaxy_tab_active2-8897.php"
    ],
    [
      "Galaxy Tab A 8.0 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_8_0_(2017)-8725.php"
    ],
    [
      "Galaxy C7 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_c7_(2017)-8789.php"
    ],
    [
      "Gear Sport",
      "https://www.gsmarena.com/samsung_gear_sport-8839.php"
    ],
    [
      "Galaxy Note8",
      "https://www.gsmarena.com/samsung_galaxy_note8-8505.php"
    ],
    [
      "Galaxy S8 Active",
      "https://www.gsmarena.com/samsung_galaxy_s8_active-8676.php"
    ],
    [
      "Galaxy J7 V",
      "https://www.gsmarena.com/samsung_galaxy_j7_v-8778.php"
    ],
    [
      "Galaxy Note FE",
      "https://www.gsmarena.com/samsung_galaxy_note_fe-8683.php"
    ],
    [
      "Galaxy J7 Max",
      "https://www.gsmarena.com/samsung_galaxy_j7_max-8684.php"
    ],
    [
      "Galaxy J7 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_j7_(2017)-8675.php"
    ],
    [
      "Galaxy J7 Pro",
      "https://www.gsmarena.com/samsung_galaxy_j7_pro-8561.php"
    ],
    [
      "Galaxy J5 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_j5_(2017)-8705.php"
    ],
    [
      "Galaxy J3 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_j3_(2017)-8438.php"
    ],
    [
      "Galaxy Folder2",
      "https://www.gsmarena.com/samsung_galaxy_folder2-9331.php"
    ],
    [
      "Z4",
      "https://www.gsmarena.com/samsung_z4-8682.php"
    ],
    [
      "Galaxy S8",
      "https://www.gsmarena.com/samsung_galaxy_s8-8161.php"
    ],
    [
      "Galaxy S8+",
      "https://www.gsmarena.com/samsung_galaxy_s8+-8523.php"
    ],
    [
      "Gear S3 classic LTE",
      "https://www.gsmarena.com/samsung_gear_s3_classic_lte-8627.php"
    ],
    [
      "Galaxy C5 Pro",
      "https://www.gsmarena.com/samsung_galaxy_c5_pro-8437.php"
    ],
    [
      "Galaxy XCover 4",
      "https://www.gsmarena.com/samsung_galaxy_xcover_4-8577.php"
    ],
    [
      "Galaxy Tab S3 9.7",
      "https://www.gsmarena.com/samsung_galaxy_tab_s3_9_7-8554.php"
    ],
    [
      "Galaxy J1 mini prime",
      "https://www.gsmarena.com/samsung_galaxy_j1_mini_prime-8397.php"
    ],
    [
      "Galaxy J3 Emerge",
      "https://www.gsmarena.com/samsung_galaxy_j3_emerge-8486.php"
    ],
    [
      "Galaxy C7 Pro",
      "https://www.gsmarena.com/samsung_galaxy_c7_pro-8435.php"
    ],
    [
      "Galaxy A7 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_a7_(2017)-8335.php"
    ],
    [
      "Galaxy A5 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_a5_(2017)-8494.php"
    ],
    [
      "Galaxy A3 (2017)",
      "https://www.gsmarena.com/samsung_galaxy_a3_(2017)-8336.php"
    ],
    [
      "Galaxy Grand Prime Plus",
      "https://www.gsmarena.com/samsung_galaxy_grand_prime_plus-8385.php"
    ],
    [
      "Galaxy J2 Prime",
      "https://www.gsmarena.com/samsung_galaxy_j2_prime-8424.php"
    ],
    [
      "Galaxy C9 Pro",
      "https://www.gsmarena.com/samsung_galaxy_c9_pro-8347.php"
    ],
    [
      "Galaxy A8 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_a8_(2016)-7801.php"
    ],
    [
      "Galaxy On8",
      "https://www.gsmarena.com/samsung_galaxy_on8-8367.php"
    ],
    [
      "Galaxy On7 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_on7_(2016)-8265.php"
    ],
    [
      "Gear S3 classic",
      "https://www.gsmarena.com/samsung_gear_s3_classic-8309.php"
    ],
    [
      "Gear S3 frontier",
      "https://www.gsmarena.com/samsung_gear_s3_frontier-8308.php"
    ],
    [
      "Gear S3 frontier LTE",
      "https://www.gsmarena.com/samsung_gear_s3_frontier_lte-8307.php"
    ],
    [
      "Galaxy J5 Prime",
      "https://www.gsmarena.com/samsung_galaxy_j5_prime-8342.php"
    ],
    [
      "Galaxy J7 Prime",
      "https://www.gsmarena.com/samsung_galaxy_j7_prime-8314.php"
    ],
    [
      "Z2",
      "https://www.gsmarena.com/samsung_z2-8288.php"
    ],
    [
      "Galaxy Note7 (USA)",
      "https://www.gsmarena.com/samsung_galaxy_note7_(usa)-8242.php"
    ],
    [
      "Galaxy Note7",
      "https://www.gsmarena.com/samsung_galaxy_note7-8082.php"
    ],
    [
      "Galaxy On7 Pro",
      "https://www.gsmarena.com/samsung_galaxy_on7_pro-8140.php"
    ],
    [
      "Galaxy On5 Pro",
      "https://www.gsmarena.com/samsung_galaxy_on5_pro-8192.php"
    ],
    [
      "Galaxy Tab J",
      "https://www.gsmarena.com/samsung_galaxy_tab_j-8227.php"
    ],
    [
      "Galaxy J Max",
      "https://www.gsmarena.com/samsung_galaxy_j_max-8186.php"
    ],
    [
      "Galaxy J2 Pro (2016)",
      "https://www.gsmarena.com/samsung_galaxy_j2_pro_(2016)-8228.php"
    ],
    [
      "Galaxy J2 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_j2_(2016)-8083.php"
    ],
    [
      "Z3 Corporate",
      "https://www.gsmarena.com/samsung_z3_corporate-8170.php"
    ],
    [
      "Galaxy Xcover 3 G389F",
      "https://www.gsmarena.com/samsung_galaxy_xcover_3_g389f-8455.php"
    ],
    [
      "Galaxy S7 active",
      "https://www.gsmarena.com/samsung_galaxy_s7_active-8004.php"
    ],
    [
      "Galaxy J3 Pro",
      "https://www.gsmarena.com/samsung_galaxy_j3_pro-8126.php"
    ],
    [
      "Galaxy C7",
      "https://www.gsmarena.com/samsung_galaxy_c7-8046.php"
    ],
    [
      "Galaxy C5",
      "https://www.gsmarena.com/samsung_galaxy_c5-8047.php"
    ],
    [
      "Galaxy A9 Pro (2016)",
      "https://www.gsmarena.com/samsung_galaxy_a9_pro_(2016)-7903.php"
    ],
    [
      "Galaxy J7 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_j7_(2016)-7870.php"
    ],
    [
      "Galaxy J5 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_j5_(2016)-7869.php"
    ],
    [
      "Galaxy Tab A 10.1 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_10_1_(2016)-8090.php"
    ],
    [
      "Galaxy Tab A 7.0 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_7_0_(2016)-7880.php"
    ],
    [
      "Galaxy S7",
      "https://www.gsmarena.com/samsung_galaxy_s7-7821.php"
    ],
    [
      "Galaxy S7 edge",
      "https://www.gsmarena.com/samsung_galaxy_s7_edge-7945.php"
    ],
    [
      "Galaxy S7 edge (USA)",
      "https://www.gsmarena.com/samsung_galaxy_s7_edge_(usa)-7961.php"
    ],
    [
      "Galaxy S7 (USA)",
      "https://www.gsmarena.com/samsung_galaxy_s7_(usa)-7960.php"
    ],
    [
      "Galaxy J1 Nxt",
      "https://www.gsmarena.com/samsung_galaxy_j1_nxt-7768.php"
    ],
    [
      "Gear S2 classic 3G",
      "https://www.gsmarena.com/samsung_gear_s2_classic_3g-7934.php"
    ],
    [
      "Galaxy Tab E 8.0",
      "https://www.gsmarena.com/samsung_galaxy_tab_e_8_0-7879.php"
    ],
    [
      "Galaxy J1 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_j1_(2016)-7864.php"
    ],
    [
      "Galaxy A9 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_a9_(2016)-7641.php"
    ],
    [
      "Galaxy A7 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_a7_(2016)-7759.php"
    ],
    [
      "Galaxy A5 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_a5_(2016)-7789.php"
    ],
    [
      "Galaxy A3 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_a3_(2016)-7791.php"
    ],
    [
      "Galaxy Express Prime",
      "https://www.gsmarena.com/samsung_galaxy_express_prime-8022.php"
    ],
    [
      "Galaxy J3 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_j3_(2016)-7760.php"
    ],
    [
      "Galaxy View",
      "https://www.gsmarena.com/samsung_galaxy_view-7704.php"
    ],
    [
      "Galaxy On7",
      "https://www.gsmarena.com/samsung_galaxy_on7-7679.php"
    ],
    [
      "Galaxy On5",
      "https://www.gsmarena.com/samsung_galaxy_on5-7648.php"
    ],
    [
      "Z3",
      "https://www.gsmarena.com/samsung_z3-7273.php"
    ],
    [
      "Galaxy J1 Ace",
      "https://www.gsmarena.com/samsung_galaxy_j1_ace-7673.php"
    ],
    [
      "Gear S2 classic",
      "https://www.gsmarena.com/samsung_gear_s2_classic-7677.php"
    ],
    [
      "Gear S2",
      "https://www.gsmarena.com/samsung_gear_s2-7678.php"
    ],
    [
      "Gear S2 3G",
      "https://www.gsmarena.com/samsung_gear_s2_3g-7585.php"
    ],
    [
      "Galaxy Note5 (USA)",
      "https://www.gsmarena.com/samsung_galaxy_note5_(usa)-7504.php"
    ],
    [
      "Galaxy Note5",
      "https://www.gsmarena.com/samsung_galaxy_note5-7431.php"
    ],
    [
      "Galaxy Note5 Duos",
      "https://www.gsmarena.com/samsung_galaxy_note5_duos-7508.php"
    ],
    [
      "Galaxy S6 edge+ (USA)",
      "https://www.gsmarena.com/samsung_galaxy_s6_edge+_(usa)-7505.php"
    ],
    [
      "Galaxy S6 edge+",
      "https://www.gsmarena.com/samsung_galaxy_s6_edge+-7467.php"
    ],
    [
      "Galaxy S6 edge+ Duos",
      "https://www.gsmarena.com/samsung_galaxy_s6_edge+_duos-7509.php"
    ],
    [
      "Galaxy S5 Neo",
      "https://www.gsmarena.com/samsung_galaxy_s5_neo-6506.php"
    ],
    [
      "Galaxy S4 mini I9195I",
      "https://www.gsmarena.com/samsung_galaxy_s4_mini_i9195i-7468.php"
    ],
    [
      "Galaxy Folder",
      "https://www.gsmarena.com/samsung_galaxy_folder-7453.php"
    ],
    [
      "Galaxy Tab S2 9.7",
      "https://www.gsmarena.com/samsung_galaxy_tab_s2_9_7-7438.php"
    ],
    [
      "Galaxy Tab S2 8.0",
      "https://www.gsmarena.com/samsung_galaxy_tab_s2_8_0-7439.php"
    ],
    [
      "Galaxy A8 Duos",
      "https://www.gsmarena.com/samsung_galaxy_a8_duos-7506.php"
    ],
    [
      "Galaxy A8",
      "https://www.gsmarena.com/samsung_galaxy_a8-7249.php"
    ],
    [
      "Galaxy V Plus",
      "https://www.gsmarena.com/samsung_galaxy_v_plus-7395.php"
    ],
    [
      "Galaxy J7 Nxt",
      "https://www.gsmarena.com/samsung_galaxy_j7_nxt-8785.php"
    ],
    [
      "Galaxy J7",
      "https://www.gsmarena.com/samsung_galaxy_j7-7185.php"
    ],
    [
      "Galaxy J5",
      "https://www.gsmarena.com/samsung_galaxy_j5-7184.php"
    ],
    [
      "Galaxy Tab 4 10.1 (2015)",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_10_1_(2015)-6980.php"
    ],
    [
      "Galaxy Tab E 9.6",
      "https://www.gsmarena.com/samsung_galaxy_tab_e_9_6-7295.php"
    ],
    [
      "Guru Plus",
      "https://www.gsmarena.com/samsung_guru_plus_-7794.php"
    ],
    [
      "Metro 360",
      "https://www.gsmarena.com/samsung_metro_360-7751.php"
    ],
    [
      "Xcover 550",
      "https://www.gsmarena.com/samsung_xcover_550-7119.php"
    ],
    [
      "Galaxy S6 active",
      "https://www.gsmarena.com/samsung_galaxy_s6_active-7114.php"
    ],
    [
      "Galaxy Tab 3 V",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_v-7134.php"
    ],
    [
      "Galaxy Tab A 9.7 & S Pen",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_9_7_&_s_pen-7443.php"
    ],
    [
      "Galaxy Tab A 9.7",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_9_7-7122.php"
    ],
    [
      "Galaxy Tab A 8.0 & S Pen (2015)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_8_0_&_s_pen_(2015)-8883.php"
    ],
    [
      "Galaxy Tab A 8.0 (2015)",
      "https://www.gsmarena.com/samsung_galaxy_tab_a_8_0_(2015)-7121.php"
    ],
    [
      "Galaxy XCover 3",
      "https://www.gsmarena.com/samsung_galaxy_xcover_3-6990.php"
    ],
    [
      "Galaxy S6 edge (USA)",
      "https://www.gsmarena.com/samsung_galaxy_s6_edge_(usa)-7166.php"
    ],
    [
      "Galaxy S6 (USA)",
      "https://www.gsmarena.com/samsung_galaxy_s6_(usa)-7164.php"
    ],
    [
      "Galaxy S6 edge",
      "https://www.gsmarena.com/samsung_galaxy_s6_edge-7079.php"
    ],
    [
      "Galaxy S6 Duos",
      "https://www.gsmarena.com/samsung_galaxy_s6_duos-7377.php"
    ],
    [
      "Galaxy S6",
      "https://www.gsmarena.com/samsung_galaxy_s6-6849.php"
    ],
    [
      "Galaxy J1 4G",
      "https://www.gsmarena.com/samsung_galaxy_j1_4g-7034.php"
    ],
    [
      "Galaxy J2",
      "https://www.gsmarena.com/samsung_galaxy_j2-7357.php"
    ],
    [
      "Galaxy J1",
      "https://www.gsmarena.com/samsung_galaxy_j1-6907.php"
    ],
    [
      "Galaxy Tab 3 Lite 7.0 VE",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_lite_7_0_ve-7110.php"
    ],
    [
      "Z1",
      "https://www.gsmarena.com/samsung_z1-6894.php"
    ],
    [
      "Galaxy A7 Duos",
      "https://www.gsmarena.com/samsung_galaxy_a7_duos-7132.php"
    ],
    [
      "Galaxy A7",
      "https://www.gsmarena.com/samsung_galaxy_a7-6763.php"
    ],
    [
      "Galaxy Grand Max",
      "https://www.gsmarena.com/samsung_galaxy_grand_max-6905.php"
    ],
    [
      "Galaxy E7",
      "https://www.gsmarena.com/samsung_galaxy_e7-6879.php"
    ],
    [
      "Galaxy E5",
      "https://www.gsmarena.com/samsung_galaxy_e5-6906.php"
    ],
    [
      "Galaxy Core Prime",
      "https://www.gsmarena.com/samsung_galaxy_core_prime-6716.php"
    ],
    [
      "Galaxy A5 Duos",
      "https://www.gsmarena.com/samsung_galaxy_a5_duos-6802.php"
    ],
    [
      "Galaxy A5",
      "https://www.gsmarena.com/samsung_galaxy_a5-6761.php"
    ],
    [
      "Galaxy A3 Duos",
      "https://www.gsmarena.com/samsung_galaxy_a3_duos-6803.php"
    ],
    [
      "Galaxy A3",
      "https://www.gsmarena.com/samsung_galaxy_a3-6762.php"
    ],
    [
      "Galaxy S5 Plus",
      "https://www.gsmarena.com/samsung_galaxy_s5_plus-6748.php"
    ],
    [
      "Galaxy Pocket 2",
      "https://www.gsmarena.com/samsung_galaxy_pocket_2-4881.php"
    ],
    [
      "Galaxy V",
      "https://www.gsmarena.com/samsung_galaxy_v-6725.php"
    ],
    [
      "Galaxy Grand Prime Duos TV",
      "https://www.gsmarena.com/samsung_galaxy_grand_prime_duos_tv-6790.php"
    ],
    [
      "Galaxy Grand Prime",
      "https://www.gsmarena.com/samsung_galaxy_grand_prime-6708.php"
    ],
    [
      "Galaxy Ace Style LTE G357",
      "https://www.gsmarena.com/samsung_galaxy_ace_style_lte_g357-6706.php"
    ],
    [
      "Galaxy Note Edge",
      "https://www.gsmarena.com/samsung_galaxy_note_edge-6631.php"
    ],
    [
      "Galaxy Note 4 Duos",
      "https://www.gsmarena.com/samsung_galaxy_note_4_duos-6749.php"
    ],
    [
      "Galaxy Note 4 (USA)",
      "https://www.gsmarena.com/samsung_galaxy_note_4_(usa)-6758.php"
    ],
    [
      "Galaxy Note 4",
      "https://www.gsmarena.com/samsung_galaxy_note_4-6434.php"
    ],
    [
      "Galaxy Tab Active LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_active_lte-6658.php"
    ],
    [
      "Galaxy Tab Active",
      "https://www.gsmarena.com/samsung_galaxy_tab_active-6659.php"
    ],
    [
      "Galaxy Mega 2",
      "https://www.gsmarena.com/samsung_galaxy_mega_2-6482.php"
    ],
    [
      "Gear S",
      "https://www.gsmarena.com/samsung_gear_s-6620.php"
    ],
    [
      "Gear 2 Neo",
      "https://www.gsmarena.com/samsung_gear_2_neo-7715.php"
    ],
    [
      "Gear Live",
      "https://www.gsmarena.com/samsung_gear_live-7714.php"
    ],
    [
      "Gear 2",
      "https://www.gsmarena.com/samsung_gear_2-7717.php"
    ],
    [
      "Galaxy Gear",
      "https://www.gsmarena.com/samsung_galaxy_gear-7716.php"
    ],
    [
      "Galaxy S5 LTE-A G901F",
      "https://www.gsmarena.com/samsung_galaxy_s5_lte_a_g901f-6576.php"
    ],
    [
      "Galaxy Alpha (S801)",
      "https://www.gsmarena.com/samsung_galaxy_alpha_(s801)-6580.php"
    ],
    [
      "Galaxy Alpha",
      "https://www.gsmarena.com/samsung_galaxy_alpha-6573.php"
    ],
    [
      "Galaxy S5 mini Duos",
      "https://www.gsmarena.com/samsung_galaxy_s5_mini_duos-6563.php"
    ],
    [
      "Galaxy Avant",
      "https://www.gsmarena.com/samsung_galaxy_avant-6542.php"
    ],
    [
      "Galaxy S Duos 3",
      "https://www.gsmarena.com/samsung_galaxy_s_duos_3-6662.php"
    ],
    [
      "Guru Music 2",
      "https://www.gsmarena.com/samsung_guru_music_2-6651.php"
    ],
    [
      "Metro 312",
      "https://www.gsmarena.com/samsung_metro_312-6650.php"
    ],
    [
      "Galaxy Tab 4 8.0 (2015)",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_8_0_(2015)-6969.php"
    ],
    [
      "Galaxy Ace NXT",
      "https://www.gsmarena.com/samsung_galaxy_ace_nxt-6534.php"
    ],
    [
      "Galaxy Star 2 Plus",
      "https://www.gsmarena.com/samsung_galaxy_star_2_plus-6391.php"
    ],
    [
      "Galaxy S5 mini",
      "https://www.gsmarena.com/samsung_galaxy_s5_mini-6252.php"
    ],
    [
      "Galaxy Ace 4 LTE G313",
      "https://www.gsmarena.com/samsung_galaxy_ace_4_lte_g313-6478.php"
    ],
    [
      "Galaxy Ace 4",
      "https://www.gsmarena.com/samsung_galaxy_ace_4-6479.php"
    ],
    [
      "Galaxy Young 2",
      "https://www.gsmarena.com/samsung_galaxy_young_2-6480.php"
    ],
    [
      "Galaxy Star 2",
      "https://www.gsmarena.com/samsung_galaxy_star_2-6481.php"
    ],
    [
      "Galaxy Core II",
      "https://www.gsmarena.com/samsung_galaxy_core_ii-6331.php"
    ],
    [
      "Galaxy S5 Sport",
      "https://www.gsmarena.com/samsung_galaxy_s5_sport-6460.php"
    ],
    [
      "Galaxy S5 LTE-A G906S",
      "https://www.gsmarena.com/samsung_galaxy_s5_lte_a_g906s-6450.php"
    ],
    [
      "Galaxy Tab S 8.4 LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_s_8_4_lte-6435.php"
    ],
    [
      "Galaxy Tab S 8.4",
      "https://www.gsmarena.com/samsung_galaxy_tab_s_8_4-6439.php"
    ],
    [
      "Galaxy Tab S 10.5 LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_s_10_5_lte-6235.php"
    ],
    [
      "Galaxy Tab S 10.5",
      "https://www.gsmarena.com/samsung_galaxy_tab_s_10_5-6438.php"
    ],
    [
      "Galaxy Core Lite LTE",
      "https://www.gsmarena.com/samsung_galaxy_core_lite_lte-6432.php"
    ],
    [
      "I9301I Galaxy S3 Neo",
      "https://www.gsmarena.com/samsung_i9301i_galaxy_s3_neo-6433.php"
    ],
    [
      "Galaxy W",
      "https://www.gsmarena.com/samsung_galaxy_w-6404.php"
    ],
    [
      "Z",
      "https://www.gsmarena.com/samsung_z-6403.php"
    ],
    [
      "Galaxy S5 Active",
      "https://www.gsmarena.com/samsung_galaxy_s5_active-6356.php"
    ],
    [
      "Galaxy K zoom",
      "https://www.gsmarena.com/samsung_galaxy_k_zoom-6210.php"
    ],
    [
      "Galaxy Beam2",
      "https://www.gsmarena.com/samsung_galaxy_beam2-6328.php"
    ],
    [
      "I9300I Galaxy S3 Neo",
      "https://www.gsmarena.com/samsung_i9300i_galaxy_s3_neo-6289.php"
    ],
    [
      "Galaxy Ace Style",
      "https://www.gsmarena.com/samsung_galaxy_ace_style-6284.php"
    ],
    [
      "ATIV SE",
      "https://www.gsmarena.com/samsung_ativ_se-6245.php"
    ],
    [
      "Galaxy Tab 4 7.0",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_7_0-6251.php"
    ],
    [
      "Galaxy Tab 4 7.0 3G",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_7_0_3g-6250.php"
    ],
    [
      "Galaxy Tab 4 7.0 LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_7_0_lte-6241.php"
    ],
    [
      "Galaxy Tab 4 8.0",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_8_0-6249.php"
    ],
    [
      "Galaxy Tab 4 8.0 3G",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_8_0_3g-6248.php"
    ],
    [
      "Galaxy Tab 4 8.0 LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_8_0_lte-6240.php"
    ],
    [
      "Galaxy Tab 4 10.1",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_10_1-6247.php"
    ],
    [
      "Galaxy Tab 4 10.1 3G",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_10_1_3g-6246.php"
    ],
    [
      "Galaxy Tab 4 10.1 LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_4_10_1_lte-6239.php"
    ],
    [
      "G3812B Galaxy S3 Slim",
      "https://www.gsmarena.com/samsung_g3812b_galaxy_s3_slim-6209.php"
    ],
    [
      "I8200 Galaxy S III mini VE",
      "https://www.gsmarena.com/samsung_i8200_galaxy_s_iii_mini_ve-6190.php"
    ],
    [
      "Galaxy S5 Duos",
      "https://www.gsmarena.com/samsung_galaxy_s5_duos-6272.php"
    ],
    [
      "Galaxy S5 (octa-core)",
      "https://www.gsmarena.com/samsung_galaxy_s5_(octa_core)-6237.php"
    ],
    [
      "Galaxy S5 (USA)",
      "https://www.gsmarena.com/samsung_galaxy_s5_(usa)-6338.php"
    ],
    [
      "Galaxy S5",
      "https://www.gsmarena.com/samsung_galaxy_s5-6033.php"
    ],
    [
      "Galaxy Core LTE G386W",
      "https://www.gsmarena.com/samsung_galaxy_core_lte_g386w-6846.php"
    ],
    [
      "Galaxy Core LTE",
      "https://www.gsmarena.com/samsung_galaxy_core_lte-6099.php"
    ],
    [
      "S5611",
      "https://www.gsmarena.com/samsung_s5611-6505.php"
    ],
    [
      "E1272",
      "https://www.gsmarena.com/samsung_e1272-6170.php"
    ],
    [
      "Galaxy Star Trios S5283",
      "https://www.gsmarena.com/samsung_galaxy_star_trios_s5283-6054.php"
    ],
    [
      "Galaxy Note 3 Neo Duos",
      "https://www.gsmarena.com/samsung_galaxy_note_3_neo_duos-6015.php"
    ],
    [
      "Galaxy Note 3 Neo",
      "https://www.gsmarena.com/samsung_galaxy_note_3_neo-5961.php"
    ],
    [
      "Galaxy Tab 3 Lite 7.0 3G",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_lite_7_0_3g-5975.php"
    ],
    [
      "Galaxy Tab 3 Lite 7.0",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_lite_7_0-5969.php"
    ],
    [
      "Galaxy Grand Neo",
      "https://www.gsmarena.com/samsung_galaxy_grand_neo-5958.php"
    ],
    [
      "Galaxy Note Pro 12.2 LTE",
      "https://www.gsmarena.com/samsung_galaxy_note_pro_12_2_lte-5945.php"
    ],
    [
      "Galaxy Note Pro 12.2 3G",
      "https://www.gsmarena.com/samsung_galaxy_note_pro_12_2_3g-5944.php"
    ],
    [
      "Galaxy Note Pro 12.2",
      "https://www.gsmarena.com/samsung_galaxy_note_pro_12_2-6048.php"
    ],
    [
      "Galaxy Tab Pro 12.2 LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_pro_12_2_lte-5943.php"
    ],
    [
      "Galaxy Tab Pro 12.2 3G",
      "https://www.gsmarena.com/samsung_galaxy_tab_pro_12_2_3g-5942.php"
    ],
    [
      "Galaxy Tab Pro 12.2",
      "https://www.gsmarena.com/samsung_galaxy_tab_pro_12_2-6212.php"
    ],
    [
      "Galaxy Tab Pro 10.1 LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_pro_10_1_lte-5941.php"
    ],
    [
      "Galaxy Tab Pro 10.1",
      "https://www.gsmarena.com/samsung_galaxy_tab_pro_10_1-5940.php"
    ],
    [
      "Galaxy Tab Pro 8.4 3G/LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_pro_8_4_3g_lte-6049.php"
    ],
    [
      "Galaxy Tab Pro 8.4",
      "https://www.gsmarena.com/samsung_galaxy_tab_pro_8_4-5946.php"
    ],
    [
      "Galaxy Camera 2 GC200",
      "https://www.gsmarena.com/samsung_galaxy_camera_2_gc200-5928.php"
    ],
    [
      "Galaxy Core Advance",
      "https://www.gsmarena.com/samsung_galaxy_core_advance-5904.php"
    ],
    [
      "Galaxy S4 Active LTE-A",
      "https://www.gsmarena.com/samsung_galaxy_s4_active_lte_a-5889.php"
    ],
    [
      "Galaxy J",
      "https://www.gsmarena.com/samsung_galaxy_j-5887.php"
    ],
    [
      "Galaxy Win Pro G3812",
      "https://www.gsmarena.com/samsung_galaxy_win_pro_g3812-5886.php"
    ],
    [
      "Galaxy S Duos 2 S7582",
      "https://www.gsmarena.com/samsung_galaxy_s_duos_2_s7582-5876.php"
    ],
    [
      "Galaxy Grand 2",
      "https://www.gsmarena.com/samsung_galaxy_grand_2-5862.php"
    ],
    [
      "I9230 Galaxy Golden",
      "https://www.gsmarena.com/samsung_i9230_galaxy_golden-5814.php"
    ],
    [
      "C3590",
      "https://www.gsmarena.com/samsung_c3590-5846.php"
    ],
    [
      "Galaxy Express 2",
      "https://www.gsmarena.com/samsung_galaxy_express_2-5803.php"
    ],
    [
      "I9506 Galaxy S4",
      "https://www.gsmarena.com/samsung_i9506_galaxy_s4-5542.php"
    ],
    [
      "Galaxy Light",
      "https://www.gsmarena.com/samsung_galaxy_light-5786.php"
    ],
    [
      "Galaxy Round G910S",
      "https://www.gsmarena.com/samsung_galaxy_round_g910s-5766.php"
    ],
    [
      "Galaxy Fresh S7390",
      "https://www.gsmarena.com/samsung_galaxy_fresh_s7390-5841.php"
    ],
    [
      "Galaxy Core Plus",
      "https://www.gsmarena.com/samsung_galaxy_core_plus-5842.php"
    ],
    [
      "Galaxy Fame Lite Duos S6792L",
      "https://www.gsmarena.com/samsung_galaxy_fame_lite_duos_s6792l-6216.php"
    ],
    [
      "Galaxy Fame Lite S6790",
      "https://www.gsmarena.com/samsung_galaxy_fame_lite_s6790-6213.php"
    ],
    [
      "Galaxy Star Pro S7260",
      "https://www.gsmarena.com/samsung_galaxy_star_pro_s7260-5749.php"
    ],
    [
      "Galaxy Note 10.1 (2014)",
      "https://www.gsmarena.com/samsung_galaxy_note_10_1_(2014)-5677.php"
    ],
    [
      "Galaxy Note 3",
      "https://www.gsmarena.com/samsung_galaxy_note_3-5665.php"
    ],
    [
      "Ch@t 333",
      "https://www.gsmarena.com/samsung_ch@t_333-5634.php"
    ],
    [
      "Galaxy Prevail 2",
      "https://www.gsmarena.com/samsung_galaxy_prevail_2-5597.php"
    ],
    [
      "Gravity Q T289",
      "https://www.gsmarena.com/samsung_gravity_q_t289-5599.php"
    ],
    [
      "ATIV S Neo",
      "https://www.gsmarena.com/samsung_ativ_s_neo-5548.php"
    ],
    [
      "Galaxy S4 zoom",
      "https://www.gsmarena.com/samsung_galaxy_s4_zoom-5447.php"
    ],
    [
      "Galaxy S II TV",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_tv-6785.php"
    ],
    [
      "Galaxy Ace 3",
      "https://www.gsmarena.com/samsung_galaxy_ace_3-5479.php"
    ],
    [
      "I9190 Galaxy S4 mini",
      "https://www.gsmarena.com/samsung_i9190_galaxy_s4_mini-5375.php"
    ],
    [
      "I9295 Galaxy S4 Active",
      "https://www.gsmarena.com/samsung_i9295_galaxy_s4_active-5446.php"
    ],
    [
      "Galaxy Tab 3 8.0",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_8_0-5456.php"
    ],
    [
      "Galaxy Tab 3 10.1 P5220",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_10_1_p5220-5491.php"
    ],
    [
      "Galaxy Tab 3 10.1 P5200",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_10_1_p5200-5490.php"
    ],
    [
      "Galaxy Tab 3 10.1 P5210",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_10_1_p5210-5478.php"
    ],
    [
      "Galaxy Exhibit T599",
      "https://www.gsmarena.com/samsung_galaxy_exhibit_t599-5468.php"
    ],
    [
      "Galaxy Core I8260",
      "https://www.gsmarena.com/samsung_galaxy_core_i8260-5419.php"
    ],
    [
      "Galaxy Tab 3 7.0 WiFi",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_7_0_wifi-5423.php"
    ],
    [
      "Galaxy Tab 3 7.0",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_7_0-5422.php"
    ],
    [
      "Galaxy Mega 6.3 I9200",
      "https://www.gsmarena.com/samsung_galaxy_mega_6_3_i9200-5398.php"
    ],
    [
      "Galaxy Mega 5.8 I9150",
      "https://www.gsmarena.com/samsung_galaxy_mega_5_8_i9150-5396.php"
    ],
    [
      "Galaxy Trend II Duos S7572",
      "https://www.gsmarena.com/samsung_galaxy_trend_ii_duos_s7572-5394.php"
    ],
    [
      "Galaxy Win I8550",
      "https://www.gsmarena.com/samsung_galaxy_win_i8550-5392.php"
    ],
    [
      "Galaxy Pocket Neo S5310",
      "https://www.gsmarena.com/samsung_galaxy_pocket_neo_s5310-5391.php"
    ],
    [
      "Galaxy Star S5280",
      "https://www.gsmarena.com/samsung_galaxy_star_s5280-5314.php"
    ],
    [
      "I9505 Galaxy S4",
      "https://www.gsmarena.com/samsung_i9505_galaxy_s4-5371.php"
    ],
    [
      "I9500 Galaxy S4",
      "https://www.gsmarena.com/samsung_i9500_galaxy_s4-5125.php"
    ],
    [
      "I9502 Galaxy S4",
      "https://www.gsmarena.com/samsung_i9502_galaxy_s4-5420.php"
    ],
    [
      "Galaxy S4 CDMA",
      "https://www.gsmarena.com/samsung_galaxy_s4_cdma-5388.php"
    ],
    [
      "Galaxy Note 8.0",
      "https://www.gsmarena.com/samsung_galaxy_note_8_0-5252.php"
    ],
    [
      "Galaxy Note 8.0 Wi-Fi",
      "https://www.gsmarena.com/samsung_galaxy_note_8_0_wi_fi-5253.php"
    ],
    [
      "Galaxy Y Plus S5303",
      "https://www.gsmarena.com/samsung_galaxy_y_plus_s5303-5357.php"
    ],
    [
      "Rex 90 S5292",
      "https://www.gsmarena.com/samsung_rex_90_s5292-5301.php"
    ],
    [
      "Rex 80 S5222R",
      "https://www.gsmarena.com/samsung_rex_80_s5222r-5300.php"
    ],
    [
      "Rex 70 S3802",
      "https://www.gsmarena.com/samsung_rex_70_s3802-5299.php"
    ],
    [
      "Rex 60 C3312R",
      "https://www.gsmarena.com/samsung_rex_60_c3312r-5298.php"
    ],
    [
      "Metro E2202",
      "https://www.gsmarena.com/samsung_metro_e2202-5312.php"
    ],
    [
      "E1282T",
      "https://www.gsmarena.com/samsung_e1282t-5362.php"
    ],
    [
      "E1207T",
      "https://www.gsmarena.com/samsung_e1207t-5361.php"
    ],
    [
      "Galaxy Young S6310",
      "https://www.gsmarena.com/samsung_galaxy_young_s6310-5280.php"
    ],
    [
      "Galaxy Fame S6810",
      "https://www.gsmarena.com/samsung_galaxy_fame_s6810-5279.php"
    ],
    [
      "Galaxy Express I8730",
      "https://www.gsmarena.com/samsung_galaxy_express_i8730-5271.php"
    ],
    [
      "S7710 Galaxy Xcover 2",
      "https://www.gsmarena.com/samsung_s7710_galaxy_xcover_2-5263.php"
    ],
    [
      "Ativ Odyssey I930",
      "https://www.gsmarena.com/samsung_ativ_odyssey_i930-5221.php"
    ],
    [
      "I9105 Galaxy S II Plus",
      "https://www.gsmarena.com/samsung_i9105_galaxy_s_ii_plus-5213.php"
    ],
    [
      "Galaxy Grand I9082",
      "https://www.gsmarena.com/samsung_galaxy_grand_i9082-5163.php"
    ],
    [
      "Galaxy Grand I9080",
      "https://www.gsmarena.com/samsung_galaxy_grand_i9080-5162.php"
    ],
    [
      "Star Deluxe Duos S5292",
      "https://www.gsmarena.com/samsung_star_deluxe_duos_s5292-5059.php"
    ],
    [
      "Galaxy Note LTE 10.1 N8020",
      "https://www.gsmarena.com/samsung_galaxy_note_lte_10_1_n8020-5151.php"
    ],
    [
      "A997 Rugby III",
      "https://www.gsmarena.com/samsung_a997_rugby_iii-5146.php"
    ],
    [
      "Galaxy Axiom R830",
      "https://www.gsmarena.com/samsung_galaxy_axiom_r830-5144.php"
    ],
    [
      "Galaxy Stratosphere II I415",
      "https://www.gsmarena.com/samsung_galaxy_stratosphere_ii_i415-5124.php"
    ],
    [
      "Galaxy Discover S730M",
      "https://www.gsmarena.com/samsung_galaxy_discover_s730m-5111.php"
    ],
    [
      "Galaxy Pop SHV-E220",
      "https://www.gsmarena.com/samsung_galaxy_pop_shv_e220-5272.php"
    ],
    [
      "Galaxy Premier I9260",
      "https://www.gsmarena.com/samsung_galaxy_premier_i9260-5047.php"
    ],
    [
      "Google Nexus 10 P8110",
      "https://www.gsmarena.com/samsung_google_nexus_10_p8110-5084.php"
    ],
    [
      "Ativ Tab P8510",
      "https://www.gsmarena.com/samsung_ativ_tab_p8510-5081.php"
    ],
    [
      "Comment 2 R390C",
      "https://www.gsmarena.com/samsung_comment_2_r390c-5079.php"
    ],
    [
      "I8190 Galaxy S III mini",
      "https://www.gsmarena.com/samsung_i8190_galaxy_s_iii_mini-5033.php"
    ],
    [
      "Galaxy Music Duos S6012",
      "https://www.gsmarena.com/samsung_galaxy_music_duos_s6012-5027.php"
    ],
    [
      "Galaxy Music S6010",
      "https://www.gsmarena.com/samsung_galaxy_music_s6010-5026.php"
    ],
    [
      "Galaxy Rugby Pro I547",
      "https://www.gsmarena.com/samsung_galaxy_rugby_pro_i547-5019.php"
    ],
    [
      "Galaxy Express I437",
      "https://www.gsmarena.com/samsung_galaxy_express_i437-5018.php"
    ],
    [
      "Ch@t 357",
      "https://www.gsmarena.com/samsung_ch@t_357-4931.php"
    ],
    [
      "I9305 Galaxy S III",
      "https://www.gsmarena.com/samsung_i9305_galaxy_s_iii-5001.php"
    ],
    [
      "Galaxy Victory 4G LTE L300",
      "https://www.gsmarena.com/samsung_galaxy_victory_4g_lte_l300-4992.php"
    ],
    [
      "Champ Neo Duos C3262",
      "https://www.gsmarena.com/samsung_champ_neo_duos_c3262-4999.php"
    ],
    [
      "Galaxy S Relay 4G T699",
      "https://www.gsmarena.com/samsung_galaxy_s_relay_4g_t699-4914.php"
    ],
    [
      "Galaxy Pocket Duos S5302",
      "https://www.gsmarena.com/samsung_galaxy_pocket_duos_s5302-4965.php"
    ],
    [
      "Galaxy Note II N7100",
      "https://www.gsmarena.com/samsung_galaxy_note_ii_n7100-4854.php"
    ],
    [
      "Galaxy Note II CDMA",
      "https://www.gsmarena.com/samsung_galaxy_note_ii_cdma-5152.php"
    ],
    [
      "Ativ S I8750",
      "https://www.gsmarena.com/samsung_ativ_s_i8750-4960.php"
    ],
    [
      "Galaxy Camera GC100",
      "https://www.gsmarena.com/samsung_galaxy_camera_gc100-4961.php"
    ],
    [
      "Galaxy Rush M830",
      "https://www.gsmarena.com/samsung_galaxy_rush_m830-4957.php"
    ],
    [
      "Galaxy Stellar 4G I200",
      "https://www.gsmarena.com/samsung_galaxy_stellar_4g_i200-4954.php"
    ],
    [
      "Galaxy Reverb M950",
      "https://www.gsmarena.com/samsung_galaxy_reverb_m950-4953.php"
    ],
    [
      "Galaxy Tab 2 7.0 I705",
      "https://www.gsmarena.com/samsung_galaxy_tab_2_7_0_i705-4923.php"
    ],
    [
      "Galaxy Note 10.1 N8000",
      "https://www.gsmarena.com/samsung_galaxy_note_10_1_n8000-4573.php"
    ],
    [
      "Galaxy Note 10.1 N8010",
      "https://www.gsmarena.com/samsung_galaxy_note_10_1_n8010-4670.php"
    ],
    [
      "Array M390",
      "https://www.gsmarena.com/samsung_array_m390-4909.php"
    ],
    [
      "Galaxy S Lightray 4G R940",
      "https://www.gsmarena.com/samsung_galaxy_s_lightray_4g_r940-4900.php"
    ],
    [
      "Galaxy S Duos S7562",
      "https://www.gsmarena.com/samsung_galaxy_s_duos_s7562-4883.php"
    ],
    [
      "Manhattan E3300",
      "https://www.gsmarena.com/samsung_manhattan_e3300-5342.php"
    ],
    [
      "E2262",
      "https://www.gsmarena.com/samsung_e2262-5341.php"
    ],
    [
      "E1260B",
      "https://www.gsmarena.com/samsung_e1260b-5311.php"
    ],
    [
      "E1200 Pusha",
      "https://www.gsmarena.com/samsung_e1200_pusha-4919.php"
    ],
    [
      "E2252",
      "https://www.gsmarena.com/samsung_e2252-4870.php"
    ],
    [
      "Galaxy Chat B5330",
      "https://www.gsmarena.com/samsung_galaxy_chat_b5330-4866.php"
    ],
    [
      "U485 Intensity III",
      "https://www.gsmarena.com/samsung_u485_intensity_iii-4865.php"
    ],
    [
      "Galaxy I8250",
      "https://www.gsmarena.com/samsung_galaxy_i8250-4857.php"
    ],
    [
      "Galaxy Ace Advance S6800",
      "https://www.gsmarena.com/samsung_galaxy_ace_advance_s6800-5114.php"
    ],
    [
      "Galaxy Ace Duos S6802",
      "https://www.gsmarena.com/samsung_galaxy_ace_duos_s6802-4777.php"
    ],
    [
      "Galaxy Appeal I827",
      "https://www.gsmarena.com/samsung_galaxy_appeal_i827-4771.php"
    ],
    [
      "Galaxy Tab 8.9 4G P7320T",
      "https://www.gsmarena.com/samsung_galaxy_tab_8_9_4g_p7320t-4769.php"
    ],
    [
      "C3780",
      "https://www.gsmarena.com/samsung_c3780-4851.php"
    ],
    [
      "C3782 Evan",
      "https://www.gsmarena.com/samsung_c3782_evan-4753.php"
    ],
    [
      "Galaxy Proclaim S720C",
      "https://www.gsmarena.com/samsung_galaxy_proclaim_s720c-4754.php"
    ],
    [
      "I9500 Fraser",
      "https://www.gsmarena.com/samsung_i9500_fraser-4752.php"
    ],
    [
      "Omnia M S7530",
      "https://www.gsmarena.com/samsung_omnia_m_s7530-4750.php"
    ],
    [
      "Focus 2 I667",
      "https://www.gsmarena.com/samsung_focus_2_i667-4745.php"
    ],
    [
      "Galaxy S III T999",
      "https://www.gsmarena.com/samsung_galaxy_s_iii_t999-4804.php"
    ],
    [
      "Galaxy S III I747",
      "https://www.gsmarena.com/samsung_galaxy_s_iii_i747-4803.php"
    ],
    [
      "Galaxy S III CDMA",
      "https://www.gsmarena.com/samsung_galaxy_s_iii_cdma-4799.php"
    ],
    [
      "I9300 Galaxy S III",
      "https://www.gsmarena.com/samsung_i9300_galaxy_s_iii-4238.php"
    ],
    [
      "E2350B",
      "https://www.gsmarena.com/samsung_e2350b-5343.php"
    ],
    [
      "U380 Brightside",
      "https://www.gsmarena.com/samsung_u380_brightside-4641.php"
    ],
    [
      "Galaxy Player 70 Plus",
      "https://www.gsmarena.com/samsung_galaxy_player_70_plus-4633.php"
    ],
    [
      "Galaxy Pocket plus S5301",
      "https://www.gsmarena.com/samsung_galaxy_pocket_plus_s5301-5461.php"
    ],
    [
      "Galaxy Pocket S5300",
      "https://www.gsmarena.com/samsung_galaxy_pocket_s5300-4612.php"
    ],
    [
      "I8530 Galaxy Beam",
      "https://www.gsmarena.com/samsung_i8530_galaxy_beam-4566.php"
    ],
    [
      "Galaxy Tab 2 10.1 CDMA",
      "https://www.gsmarena.com/samsung_galaxy_tab_2_10_1_cdma-5592.php"
    ],
    [
      "Galaxy Tab 2 10.1 P5110",
      "https://www.gsmarena.com/samsung_galaxy_tab_2_10_1_p5110-4669.php"
    ],
    [
      "Galaxy Tab 2 10.1 P5100",
      "https://www.gsmarena.com/samsung_galaxy_tab_2_10_1_p5100-4567.php"
    ],
    [
      "Rugby Smart I847",
      "https://www.gsmarena.com/samsung_rugby_smart_i847-4564.php"
    ],
    [
      "W999",
      "https://www.gsmarena.com/samsung_w999-4660.php"
    ],
    [
      "Galaxy Ace II X S7560M",
      "https://www.gsmarena.com/samsung_galaxy_ace_ii_x_s7560m-5340.php"
    ],
    [
      "Galaxy Ace 2 I8160",
      "https://www.gsmarena.com/samsung_galaxy_ace_2_i8160-4559.php"
    ],
    [
      "Galaxy mini 2 S6500",
      "https://www.gsmarena.com/samsung_galaxy_mini_2_s6500-3883.php"
    ],
    [
      "Galaxy Ace Duos I589",
      "https://www.gsmarena.com/samsung_galaxy_ace_duos_i589-4548.php"
    ],
    [
      "Galaxy Pop Plus S5570i",
      "https://www.gsmarena.com/samsung_galaxy_pop_plus_s5570i-4544.php"
    ],
    [
      "Galaxy Tab 2 7.0 P3110",
      "https://www.gsmarena.com/samsung_galaxy_tab_2_7_0_p3110-4671.php"
    ],
    [
      "Galaxy Tab 2 7.0 P3100",
      "https://www.gsmarena.com/samsung_galaxy_tab_2_7_0_p3100-4543.php"
    ],
    [
      "I9070 Galaxy S Advance",
      "https://www.gsmarena.com/samsung_i9070_galaxy_s_advance-4469.php"
    ],
    [
      "C3312 Duos",
      "https://www.gsmarena.com/samsung_c3312_duos-4403.php"
    ],
    [
      "Star 3 s5220",
      "https://www.gsmarena.com/samsung_star_3_s5220-4448.php"
    ],
    [
      "Star 3 Duos S5222",
      "https://www.gsmarena.com/samsung_star_3_duos_s5222-4447.php"
    ],
    [
      "Galaxy Nexus I9250M",
      "https://www.gsmarena.com/samsung_galaxy_nexus_i9250m-4432.php"
    ],
    [
      "Galaxy Attain 4G",
      "https://www.gsmarena.com/samsung_galaxy_attain_4g-4431.php"
    ],
    [
      "Galaxy S Blaze 4G T769",
      "https://www.gsmarena.com/samsung_galaxy_s_blaze_4g_t769-4420.php"
    ],
    [
      "Galaxy Tab 7.7 LTE I815",
      "https://www.gsmarena.com/samsung_galaxy_tab_7_7_lte_i815-4419.php"
    ],
    [
      "Galaxy Nexus LTE L700",
      "https://www.gsmarena.com/samsung_galaxy_nexus_lte_l700-4406.php"
    ],
    [
      "Exhilarate i577",
      "https://www.gsmarena.com/samsung_exhilarate_i577-4415.php"
    ],
    [
      "Galaxy S II Skyrocket HD I757",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_skyrocket_hd_i757-4414.php"
    ],
    [
      "Galaxy Note T879",
      "https://www.gsmarena.com/samsung_galaxy_note_t879-4952.php"
    ],
    [
      "Galaxy Note I717",
      "https://www.gsmarena.com/samsung_galaxy_note_i717-4374.php"
    ],
    [
      "Galaxy M Style M340S",
      "https://www.gsmarena.com/samsung_galaxy_m_style_m340s-4401.php"
    ],
    [
      "Galaxy Ace Plus S7500",
      "https://www.gsmarena.com/samsung_galaxy_ace_plus_s7500-4397.php"
    ],
    [
      "I929 Galaxy S II Duos",
      "https://www.gsmarena.com/samsung_i929_galaxy_s_ii_duos-4387.php"
    ],
    [
      "Galaxy Y Duos S6102",
      "https://www.gsmarena.com/samsung_galaxy_y_duos_s6102-4385.php"
    ],
    [
      "Galaxy Y Pro Duos B5512",
      "https://www.gsmarena.com/samsung_galaxy_y_pro_duos_b5512-4371.php"
    ],
    [
      "E2600",
      "https://www.gsmarena.com/samsung_e2600-4380.php"
    ],
    [
      "M370",
      "https://www.gsmarena.com/samsung_m370-4405.php"
    ],
    [
      "R680 Repp",
      "https://www.gsmarena.com/samsung_r680_repp-4350.php"
    ],
    [
      "I110 Illusion",
      "https://www.gsmarena.com/samsung_i110_illusion-4344.php"
    ],
    [
      "E1050",
      "https://www.gsmarena.com/samsung_e1050-5556.php"
    ],
    [
      "E1232B",
      "https://www.gsmarena.com/samsung_e1232b-4337.php"
    ],
    [
      "E1230",
      "https://www.gsmarena.com/samsung_e1230-4336.php"
    ],
    [
      "Focus S I937",
      "https://www.gsmarena.com/samsung_focus_s_i937-4158.php"
    ],
    [
      "Focus Flash I677",
      "https://www.gsmarena.com/samsung_focus_flash_i677-4159.php"
    ],
    [
      "Galaxy S II Skyrocket i727",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_skyrocket_i727-4303.php"
    ],
    [
      "C3330 Champ 2",
      "https://www.gsmarena.com/samsung_c3330_champ_2-4217.php"
    ],
    [
      "Exhibit II 4G T679",
      "https://www.gsmarena.com/samsung_exhibit_ii_4g_t679-4277.php"
    ],
    [
      "C3350",
      "https://www.gsmarena.com/samsung_c3350-4271.php"
    ],
    [
      "C3520",
      "https://www.gsmarena.com/samsung_c3520-4218.php"
    ],
    [
      "Galaxy Nexus i515",
      "https://www.gsmarena.com/samsung_galaxy_nexus_i515-4301.php"
    ],
    [
      "Galaxy Nexus I9250",
      "https://www.gsmarena.com/samsung_galaxy_nexus_i9250-4219.php"
    ],
    [
      "I9100G Galaxy S II",
      "https://www.gsmarena.com/samsung_i9100g_galaxy_s_ii-4327.php"
    ],
    [
      "I405 Stratosphere",
      "https://www.gsmarena.com/samsung_i405_stratosphere-4262.php"
    ],
    [
      "R730 Transfix",
      "https://www.gsmarena.com/samsung_r730_transfix-4284.php"
    ],
    [
      "i927 Captivate Glide",
      "https://www.gsmarena.com/samsung_i927_captivate_glide-4071.php"
    ],
    [
      "DoubleTime I857",
      "https://www.gsmarena.com/samsung_doubletime_i857-4239.php"
    ],
    [
      "M930 Transform Ultra",
      "https://www.gsmarena.com/samsung_m930_transform_ultra-4270.php"
    ],
    [
      "P6210 Galaxy Tab 7.0 Plus",
      "https://www.gsmarena.com/samsung_p6210_galaxy_tab_7_0_plus-4667.php"
    ],
    [
      "P6200 Galaxy Tab 7.0 Plus",
      "https://www.gsmarena.com/samsung_p6200_galaxy_tab_7_0_plus-4208.php"
    ],
    [
      "Omnia W I8350",
      "https://www.gsmarena.com/samsung_omnia_w_i8350-4200.php"
    ],
    [
      "Galaxy S II HD LTE",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_hd_lte-4198.php"
    ],
    [
      "Ch@t 527",
      "https://www.gsmarena.com/samsung_ch@t_527-4096.php"
    ],
    [
      "P6810 Galaxy Tab 7.7",
      "https://www.gsmarena.com/samsung_p6810_galaxy_tab_7_7-4668.php"
    ],
    [
      "P6800 Galaxy Tab 7.7",
      "https://www.gsmarena.com/samsung_p6800_galaxy_tab_7_7-4136.php"
    ],
    [
      "Galaxy Note N7000",
      "https://www.gsmarena.com/samsung_galaxy_note_n7000-4135.php"
    ],
    [
      "Galaxy S II I777",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_i777-4130.php"
    ],
    [
      "Galaxy S II X T989D",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_x_t989d-4468.php"
    ],
    [
      "Galaxy S II T989",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_t989-4129.php"
    ],
    [
      "Galaxy S II Epic 4G Touch",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_epic_4g_touch-3956.php"
    ],
    [
      "S8600 Wave 3",
      "https://www.gsmarena.com/samsung_s8600_wave_3-4126.php"
    ],
    [
      "Wave M S7250",
      "https://www.gsmarena.com/samsung_wave_m_s7250-4054.php"
    ],
    [
      "Wave Y S5380",
      "https://www.gsmarena.com/samsung_wave_y_s5380-4127.php"
    ],
    [
      "Galaxy S II LTE i727R",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_lte_i727r-4376.php"
    ],
    [
      "Galaxy S II LTE I9210",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_lte_i9210-4124.php"
    ],
    [
      "Galaxy Tab 8.9 LTE I957",
      "https://www.gsmarena.com/samsung_galaxy_tab_8_9_lte_i957-4125.php"
    ],
    [
      "Galaxy W I8150",
      "https://www.gsmarena.com/samsung_galaxy_w_i8150-4114.php"
    ],
    [
      "Galaxy M Pro B7800",
      "https://www.gsmarena.com/samsung_galaxy_m_pro_b7800-4115.php"
    ],
    [
      "Galaxy Y Pro B5510",
      "https://www.gsmarena.com/samsung_galaxy_y_pro_b5510-4116.php"
    ],
    [
      "Galaxy Y TV S5367",
      "https://www.gsmarena.com/samsung_galaxy_y_tv_s5367-4694.php"
    ],
    [
      "Galaxy Y S5360",
      "https://www.gsmarena.com/samsung_galaxy_y_s5360-4117.php"
    ],
    [
      "Gravity TXT T379",
      "https://www.gsmarena.com/samsung_gravity_txt_t379-4123.php"
    ],
    [
      "Galaxy Q T589R",
      "https://www.gsmarena.com/samsung_galaxy_q_t589r-3437.php"
    ],
    [
      "S5610",
      "https://www.gsmarena.com/samsung_s5610-4191.php"
    ],
    [
      "S3770",
      "https://www.gsmarena.com/samsung_s3770-4100.php"
    ],
    [
      "S5690 Galaxy Xcover",
      "https://www.gsmarena.com/samsung_s5690_galaxy_xcover-4091.php"
    ],
    [
      "Galaxy S II 4G I9100M",
      "https://www.gsmarena.com/samsung_galaxy_s_ii_4g_i9100m-4059.php"
    ],
    [
      "I9103 Galaxy R",
      "https://www.gsmarena.com/samsung_i9103_galaxy_r-3967.php"
    ],
    [
      "R720 Admire",
      "https://www.gsmarena.com/samsung_r720_admire-4269.php"
    ],
    [
      "Convoy 2",
      "https://www.gsmarena.com/samsung_convoy_2-4283.php"
    ],
    [
      "Conquer 4G",
      "https://www.gsmarena.com/samsung_conquer_4g-4026.php"
    ],
    [
      "R640 Character",
      "https://www.gsmarena.com/samsung_r640_character-4058.php"
    ],
    [
      "R380 Freeform III",
      "https://www.gsmarena.com/samsung_r380_freeform_iii-4039.php"
    ],
    [
      "Trender",
      "https://www.gsmarena.com/samsung_trender-3991.php"
    ],
    [
      "Gravity SMART",
      "https://www.gsmarena.com/samsung_gravity_smart-3969.php"
    ],
    [
      "Exhibit 4G",
      "https://www.gsmarena.com/samsung_exhibit_4g-3968.php"
    ],
    [
      "E1195",
      "https://www.gsmarena.com/samsung_e1195-4274.php"
    ],
    [
      "C414",
      "https://www.gsmarena.com/samsung_c414-4056.php"
    ],
    [
      "E1190",
      "https://www.gsmarena.com/samsung_e1190-4034.php"
    ],
    [
      "C3560",
      "https://www.gsmarena.com/samsung_c3560-4093.php"
    ],
    [
      "Dart T499",
      "https://www.gsmarena.com/samsung_dart_t499-4055.php"
    ],
    [
      "C3322",
      "https://www.gsmarena.com/samsung_c3322-3941.php"
    ],
    [
      "R260 Chrono",
      "https://www.gsmarena.com/samsung_r260_chrono-3992.php"
    ],
    [
      "E2232",
      "https://www.gsmarena.com/samsung_e2232-3940.php"
    ],
    [
      "E1182",
      "https://www.gsmarena.com/samsung_e1182-3939.php"
    ],
    [
      "I9001 Galaxy S Plus",
      "https://www.gsmarena.com/samsung_i9001_galaxy_s_plus-3908.php"
    ],
    [
      "DuosTV I6712",
      "https://www.gsmarena.com/samsung_duostv_i6712-3986.php"
    ],
    [
      "C6712 Star II DUOS",
      "https://www.gsmarena.com/samsung_c6712_star_ii_duos-3921.php"
    ],
    [
      "Ch@t 222",
      "https://www.gsmarena.com/samsung_ch@t_222-3993.php"
    ],
    [
      "Ch@t 220",
      "https://www.gsmarena.com/samsung_ch@t_220-4611.php"
    ],
    [
      "M580 Replenish",
      "https://www.gsmarena.com/samsung_m580_replenish-3927.php"
    ],
    [
      "Galaxy Prevail",
      "https://www.gsmarena.com/samsung_galaxy_prevail-3916.php"
    ],
    [
      "M220L Galaxy Neo",
      "https://www.gsmarena.com/samsung_m220l_galaxy_neo-3913.php"
    ],
    [
      "Google Nexus S I9020A",
      "https://www.gsmarena.com/samsung_google_nexus_s_i9020a-3920.php"
    ],
    [
      "Google Nexus S I9023",
      "https://www.gsmarena.com/samsung_google_nexus_s_i9023-3910.php"
    ],
    [
      "E2230",
      "https://www.gsmarena.com/samsung_e2230-3904.php"
    ],
    [
      "P1010 Galaxy Tab Wi-Fi",
      "https://www.gsmarena.com/samsung_p1010_galaxy_tab_wi_fi-3919.php"
    ],
    [
      "P7500 Galaxy Tab 10.1 3G",
      "https://www.gsmarena.com/samsung_p7500_galaxy_tab_10_1_3g-3892.php"
    ],
    [
      "Galaxy Tab 10.1 LTE I905",
      "https://www.gsmarena.com/samsung_galaxy_tab_10_1_lte_i905-4069.php"
    ],
    [
      "Galaxy Tab 10.1 P7510",
      "https://www.gsmarena.com/samsung_galaxy_tab_10_1_p7510-3894.php"
    ],
    [
      "Galaxy Tab 8.9 P7300",
      "https://www.gsmarena.com/samsung_galaxy_tab_8_9_p7300-3891.php"
    ],
    [
      "Galaxy Tab 8.9 P7310",
      "https://www.gsmarena.com/samsung_galaxy_tab_8_9_p7310-3893.php"
    ],
    [
      "Google Nexus S 4G",
      "https://www.gsmarena.com/samsung_google_nexus_s_4g-3884.php"
    ],
    [
      "M260 Factor",
      "https://www.gsmarena.com/samsung_m260_factor-3878.php"
    ],
    [
      "S3850 Corby II",
      "https://www.gsmarena.com/samsung_s3850_corby_ii-3765.php"
    ],
    [
      "Galaxy Pro B7510",
      "https://www.gsmarena.com/samsung_galaxy_pro_b7510-3850.php"
    ],
    [
      "I9100 Galaxy S II",
      "https://www.gsmarena.com/samsung_i9100_galaxy_s_ii-3621.php"
    ],
    [
      "E2652W Champ Duos",
      "https://www.gsmarena.com/samsung_e2652w_champ_duos-3844.php"
    ],
    [
      "E2652 Champ Duos",
      "https://www.gsmarena.com/samsung_e2652_champ_duos-3845.php"
    ],
    [
      "Galaxy S WiFi 5.0",
      "https://www.gsmarena.com/samsung_galaxy_s_wifi_5_0-3838.php"
    ],
    [
      "P7100 Galaxy Tab 10.1v",
      "https://www.gsmarena.com/samsung_p7100_galaxy_tab_10_1v-3831.php"
    ],
    [
      "S5780 Wave 578",
      "https://www.gsmarena.com/samsung_s5780_wave_578-3778.php"
    ],
    [
      "Ch@t 350",
      "https://www.gsmarena.com/samsung_ch@t_350-3729.php"
    ],
    [
      "E3213 Hero",
      "https://www.gsmarena.com/samsung_e3213_hero-4030.php"
    ],
    [
      "E3210",
      "https://www.gsmarena.com/samsung_e3210-3766.php"
    ],
    [
      "E2330",
      "https://www.gsmarena.com/samsung_e2330-3836.php"
    ],
    [
      "I9003 Galaxy SL",
      "https://www.gsmarena.com/samsung_i9003_galaxy_sl-3761.php"
    ],
    [
      "Galaxy Ace S5830I",
      "https://www.gsmarena.com/samsung_galaxy_ace_s5830i-4664.php"
    ],
    [
      "Galaxy Ace S5830",
      "https://www.gsmarena.com/samsung_galaxy_ace_s5830-3724.php"
    ],
    [
      "Galaxy Fit S5670",
      "https://www.gsmarena.com/samsung_galaxy_fit_s5670-3726.php"
    ],
    [
      "Galaxy Gio S5660",
      "https://www.gsmarena.com/samsung_galaxy_gio_s5660-3741.php"
    ],
    [
      "Galaxy Mini S5570",
      "https://www.gsmarena.com/samsung_galaxy_mini_s5570-3725.php"
    ],
    [
      "Galaxy Pop i559",
      "https://www.gsmarena.com/samsung_galaxy_pop_i559-3879.php"
    ],
    [
      "Galaxy S 4G T959",
      "https://www.gsmarena.com/samsung_galaxy_s_4g_t959-3719.php"
    ],
    [
      "S5260 Star II",
      "https://www.gsmarena.com/samsung_s5260_star_ii-3718.php"
    ],
    [
      "I997 Infuse 4G",
      "https://www.gsmarena.com/samsung_i997_infuse_4g-3705.php"
    ],
    [
      "R910 Galaxy Indulge",
      "https://www.gsmarena.com/samsung_r910_galaxy_indulge-3774.php"
    ],
    [
      "Droid Charge I510",
      "https://www.gsmarena.com/samsung_droid_charge_i510-3733.php"
    ],
    [
      "Google Nexus S",
      "https://www.gsmarena.com/samsung_google_nexus_s-3620.php"
    ],
    [
      "M190S Galaxy S Hoppin",
      "https://www.gsmarena.com/samsung_m190s_galaxy_s_hoppin-3736.php"
    ],
    [
      "Ch@t 335",
      "https://www.gsmarena.com/samsung_ch@t_335-3634.php"
    ],
    [
      "C3630",
      "https://www.gsmarena.com/samsung_c3630-3767.php"
    ],
    [
      "C3530",
      "https://www.gsmarena.com/samsung_c3530-3668.php"
    ],
    [
      "I9010 Galaxy S Giorgio Armani",
      "https://www.gsmarena.com/samsung_i9010_galaxy_s_giorgio_armani-3625.php"
    ],
    [
      "A817 Solstice II",
      "https://www.gsmarena.com/samsung_a817_solstice_ii-3617.php"
    ],
    [
      "A667 Evergreen",
      "https://www.gsmarena.com/samsung_a667_evergreen-3616.php"
    ],
    [
      "E2530",
      "https://www.gsmarena.com/samsung_e2530-3665.php"
    ],
    [
      "I100 Gem",
      "https://www.gsmarena.com/samsung_i100_gem-3738.php"
    ],
    [
      "R710 Suede",
      "https://www.gsmarena.com/samsung_r710_suede-3753.php"
    ],
    [
      "U750 Zeal",
      "https://www.gsmarena.com/samsung_u750_zeal-3618.php"
    ],
    [
      "Continuum I400",
      "https://www.gsmarena.com/samsung_continuum_i400-3615.php"
    ],
    [
      "Ch@t 322 Wi-Fi",
      "https://www.gsmarena.com/samsung_ch@t_322_wi_fi-4396.php"
    ],
    [
      "Ch@t 322",
      "https://www.gsmarena.com/samsung_ch@t_322-3588.php"
    ],
    [
      "E1252",
      "https://www.gsmarena.com/samsung_e1252-3586.php"
    ],
    [
      "Breeze B209",
      "https://www.gsmarena.com/samsung_breeze_b209-3742.php"
    ],
    [
      "Hero Plus B159",
      "https://www.gsmarena.com/samsung_hero_plus_b159-3743.php"
    ],
    [
      "W169 Duos",
      "https://www.gsmarena.com/samsung_w169_duos-3744.php"
    ],
    [
      "Mpower Muzik 219",
      "https://www.gsmarena.com/samsung_mpower_muzik_219-3745.php"
    ],
    [
      "Mpower Txt M369",
      "https://www.gsmarena.com/samsung_mpower_txt_m369-3746.php"
    ],
    [
      "Mpower TV S239",
      "https://www.gsmarena.com/samsung_mpower_tv_s239-3747.php"
    ],
    [
      "E1225 Dual Sim Shift",
      "https://www.gsmarena.com/samsung_e1225_dual_sim_shift-3748.php"
    ],
    [
      "Guru E1081T",
      "https://www.gsmarena.com/samsung_guru_e1081t-3749.php"
    ],
    [
      "Focus",
      "https://www.gsmarena.com/samsung_focus-3453.php"
    ],
    [
      "I8700 Omnia 7",
      "https://www.gsmarena.com/samsung_i8700_omnia_7-3537.php"
    ],
    [
      "S5750 Wave575",
      "https://www.gsmarena.com/samsung_s5750_wave575-3547.php"
    ],
    [
      "S8530 Wave II",
      "https://www.gsmarena.com/samsung_s8530_wave_ii-3536.php"
    ],
    [
      "M210S Wave2",
      "https://www.gsmarena.com/samsung_m210s_wave2-3768.php"
    ],
    [
      "Galaxy Tab T-Mobile T849",
      "https://www.gsmarena.com/samsung_galaxy_tab_t_mobile_t849-3672.php"
    ],
    [
      "P1000 Galaxy Tab",
      "https://www.gsmarena.com/samsung_p1000_galaxy_tab-3370.php"
    ],
    [
      "Galaxy Tab CDMA P100",
      "https://www.gsmarena.com/samsung_galaxy_tab_cdma_p100-3673.php"
    ],
    [
      "Galaxy Tab 4G LTE",
      "https://www.gsmarena.com/samsung_galaxy_tab_4g_lte-3769.php"
    ],
    [
      "T249",
      "https://www.gsmarena.com/samsung_t249-3582.php"
    ],
    [
      "R580 Profile",
      "https://www.gsmarena.com/samsung_r580_profile-3671.php"
    ],
    [
      "R570 Messenger III",
      "https://www.gsmarena.com/samsung_r570_messenger_iii-3614.php"
    ],
    [
      "I909 Galaxy S",
      "https://www.gsmarena.com/samsung_i909_galaxy_s-3487.php"
    ],
    [
      "A200K Nori F",
      "https://www.gsmarena.com/samsung_a200k_nori_f-3752.php"
    ],
    [
      "A220 F Nori",
      "https://www.gsmarena.com/samsung_a220_f_nori-3750.php"
    ],
    [
      "M130K Galaxy K",
      "https://www.gsmarena.com/samsung_m130k_galaxy_k-3584.php"
    ],
    [
      "Mesmerize i500",
      "https://www.gsmarena.com/samsung_mesmerize_i500-3580.php"
    ],
    [
      "M920 Transform",
      "https://www.gsmarena.com/samsung_m920_transform-3581.php"
    ],
    [
      "R360 Freeform II",
      "https://www.gsmarena.com/samsung_r360_freeform_ii-3579.php"
    ],
    [
      "Galaxy 551",
      "https://www.gsmarena.com/samsung_galaxy_551-3515.php"
    ],
    [
      "R900 Craft",
      "https://www.gsmarena.com/samsung_r900_craft-3531.php"
    ],
    [
      "Xcover 271",
      "https://www.gsmarena.com/samsung_xcover_271-3415.php"
    ],
    [
      "C3752",
      "https://www.gsmarena.com/samsung_c3752-4638.php"
    ],
    [
      "C3750",
      "https://www.gsmarena.com/samsung_c3750-4087.php"
    ],
    [
      "S7230E Wave 723",
      "https://www.gsmarena.com/samsung_s7230e_wave_723-3493.php"
    ],
    [
      "M130L Galaxy U",
      "https://www.gsmarena.com/samsung_m130l_galaxy_u-3490.php"
    ],
    [
      "E2152",
      "https://www.gsmarena.com/samsung_e2152-3448.php"
    ],
    [
      "S5530",
      "https://www.gsmarena.com/samsung_s5530-3368.php"
    ],
    [
      "U360 Gusto",
      "https://www.gsmarena.com/samsung_u360_gusto-3482.php"
    ],
    [
      "Fascinate",
      "https://www.gsmarena.com/samsung_fascinate-3492.php"
    ],
    [
      "Vibrant",
      "https://www.gsmarena.com/samsung_vibrant-3416.php"
    ],
    [
      "A927 Flight II",
      "https://www.gsmarena.com/samsung_a927_flight_ii-3467.php"
    ],
    [
      "T669 Gravity T",
      "https://www.gsmarena.com/samsung_t669_gravity_t-3336.php"
    ],
    [
      "T479 Gravity 3",
      "https://www.gsmarena.com/samsung_t479_gravity_3-3335.php"
    ],
    [
      ":) Smiley",
      "https://www.gsmarena.com/samsung__)_smiley-3334.php"
    ],
    [
      "C5010 Squash",
      "https://www.gsmarena.com/samsung_c5010_squash-3333.php"
    ],
    [
      "i897 Captivate",
      "https://www.gsmarena.com/samsung_i897_captivate-3408.php"
    ],
    [
      "B7350 Omnia PRO 4",
      "https://www.gsmarena.com/samsung_b7350_omnia_pro_4-3401.php"
    ],
    [
      "B6520 Omnia PRO 5",
      "https://www.gsmarena.com/samsung_b6520_omnia_pro_5-3402.php"
    ],
    [
      "U320 Haven",
      "https://www.gsmarena.com/samsung_u320_haven-3433.php"
    ],
    [
      "U460 Intensity II",
      "https://www.gsmarena.com/samsung_u460_intensity_ii-3450.php"
    ],
    [
      "M350 Seek",
      "https://www.gsmarena.com/samsung_m350_seek-3434.php"
    ],
    [
      "Epic 4G",
      "https://www.gsmarena.com/samsung_epic_4g-3410.php"
    ],
    [
      "Acclaim",
      "https://www.gsmarena.com/samsung_acclaim-3430.php"
    ],
    [
      "Intercept",
      "https://www.gsmarena.com/samsung_intercept-3429.php"
    ],
    [
      "R360 Messenger Touch",
      "https://www.gsmarena.com/samsung_r360_messenger_touch-3435.php"
    ],
    [
      "i225 Exec",
      "https://www.gsmarena.com/samsung_i225_exec-3436.php"
    ],
    [
      "M570 Restore",
      "https://www.gsmarena.com/samsung_m570_restore-3438.php"
    ],
    [
      "R351 Freeform",
      "https://www.gsmarena.com/samsung_r351_freeform-3439.php"
    ],
    [
      "T369",
      "https://www.gsmarena.com/samsung_t369-3928.php"
    ],
    [
      "S5330 Wave533",
      "https://www.gsmarena.com/samsung_s5330_wave533-3399.php"
    ],
    [
      "S5250 Wave525",
      "https://www.gsmarena.com/samsung_s5250_wave525-3400.php"
    ],
    [
      "I5500 Galaxy 5",
      "https://www.gsmarena.com/samsung_i5500_galaxy_5-3371.php"
    ],
    [
      "I5801 Galaxy Apollo",
      "https://www.gsmarena.com/samsung_i5801_galaxy_apollo-3432.php"
    ],
    [
      "I5800 Galaxy 3",
      "https://www.gsmarena.com/samsung_i5800_galaxy_3-3395.php"
    ],
    [
      "B7722",
      "https://www.gsmarena.com/samsung_b7722-3362.php"
    ],
    [
      "A847 Rugby II",
      "https://www.gsmarena.com/samsung_a847_rugby_ii-3355.php"
    ],
    [
      "C3300K Champ",
      "https://www.gsmarena.com/samsung_c3300k_champ-3346.php"
    ],
    [
      "W960 AMOLED 3D",
      "https://www.gsmarena.com/samsung_w960_amoled_3d-3319.php"
    ],
    [
      "M110S Galaxy S",
      "https://www.gsmarena.com/samsung_m110s_galaxy_s-3418.php"
    ],
    [
      "Galaxy A",
      "https://www.gsmarena.com/samsung_galaxy_a-3294.php"
    ],
    [
      "I9000 Galaxy S",
      "https://www.gsmarena.com/samsung_i9000_galaxy_s-3115.php"
    ],
    [
      "S3370",
      "https://www.gsmarena.com/samsung_s3370-3234.php"
    ],
    [
      "I6500U Galaxy",
      "https://www.gsmarena.com/samsung_i6500u_galaxy-3002.php"
    ],
    [
      "E1170",
      "https://www.gsmarena.com/samsung_e1170-3332.php"
    ],
    [
      "Metro TV",
      "https://www.gsmarena.com/samsung_metro_tv-3273.php"
    ],
    [
      "Corby TV F339",
      "https://www.gsmarena.com/samsung_corby_tv_f339-3272.php"
    ],
    [
      "S3650W Corby",
      "https://www.gsmarena.com/samsung_s3650w_corby-3047.php"
    ],
    [
      "W9705",
      "https://www.gsmarena.com/samsung_w9705-3243.php"
    ],
    [
      "A687 Strive",
      "https://www.gsmarena.com/samsung_a687_strive-3232.php"
    ],
    [
      "A697 Sunburst",
      "https://www.gsmarena.com/samsung_a697_sunburst-3231.php"
    ],
    [
      "S8500 Wave",
      "https://www.gsmarena.com/samsung_s8500_wave-3146.php"
    ],
    [
      "I8520 Galaxy Beam",
      "https://www.gsmarena.com/samsung_i8520_galaxy_beam-3150.php"
    ],
    [
      "B3410W Ch@t",
      "https://www.gsmarena.com/samsung_b3410w_ch@t-3151.php"
    ],
    [
      "M3710 Corby Beat",
      "https://www.gsmarena.com/samsung_m3710_corby_beat-3091.php"
    ],
    [
      "E2370 Xcover",
      "https://www.gsmarena.com/samsung_e2370_xcover-3152.php"
    ],
    [
      "E2550 Monte Slider",
      "https://www.gsmarena.com/samsung_e2550_monte_slider-3148.php"
    ],
    [
      "C3200 Monte Bar",
      "https://www.gsmarena.com/samsung_c3200_monte_bar-3149.php"
    ],
    [
      "S5620 Monte",
      "https://www.gsmarena.com/samsung_s5620_monte-3117.php"
    ],
    [
      "S5550 Shark 2",
      "https://www.gsmarena.com/samsung_s5550_shark_2-3074.php"
    ],
    [
      "S3550 Shark 3",
      "https://www.gsmarena.com/samsung_s3550_shark_3-3085.php"
    ],
    [
      "E1150",
      "https://www.gsmarena.com/samsung_e1150-3194.php"
    ],
    [
      "S5350 Shark",
      "https://www.gsmarena.com/samsung_s5350_shark-3086.php"
    ],
    [
      "C6112",
      "https://www.gsmarena.com/samsung_c6112-3018.php"
    ],
    [
      "M2520 Beat Techno",
      "https://www.gsmarena.com/samsung_m2520_beat_techno-3084.php"
    ],
    [
      "C3510 Genoa",
      "https://www.gsmarena.com/samsung_c3510_genoa-3068.php"
    ],
    [
      "M5650 Lindy",
      "https://www.gsmarena.com/samsung_m5650_lindy-3049.php"
    ],
    [
      "S7070 Diva",
      "https://www.gsmarena.com/samsung_s7070_diva-3030.php"
    ],
    [
      "C3730C",
      "https://www.gsmarena.com/samsung_c3730c-3048.php"
    ],
    [
      "S5150 Diva folder",
      "https://www.gsmarena.com/samsung_s5150_diva_folder-3031.php"
    ],
    [
      "C5130",
      "https://www.gsmarena.com/samsung_c5130-2994.php"
    ],
    [
      "B5722",
      "https://www.gsmarena.com/samsung_b5722-3017.php"
    ],
    [
      "M3310L",
      "https://www.gsmarena.com/samsung_m3310l-3347.php"
    ],
    [
      "M3310",
      "https://www.gsmarena.com/samsung_m3310-3016.php"
    ],
    [
      "A897 Mythic",
      "https://www.gsmarena.com/samsung_a897_mythic-2996.php"
    ],
    [
      "S5500 Eco",
      "https://www.gsmarena.com/samsung_s5500_eco-2995.php"
    ],
    [
      "T139",
      "https://www.gsmarena.com/samsung_t139-3123.php"
    ],
    [
      "A797 Flight",
      "https://www.gsmarena.com/samsung_a797_flight-3011.php"
    ],
    [
      "B3410",
      "https://www.gsmarena.com/samsung_b3410-2993.php"
    ],
    [
      "M715 T*OMNIA II",
      "https://www.gsmarena.com/samsung_m715_t*omnia_ii-3350.php"
    ],
    [
      "A886 Forever",
      "https://www.gsmarena.com/samsung_a886_forever-3088.php"
    ],
    [
      "S5560 Marvel",
      "https://www.gsmarena.com/samsung_s5560_marvel-2989.php"
    ],
    [
      "S5630C",
      "https://www.gsmarena.com/samsung_s5630c-3003.php"
    ],
    [
      "I5700 Galaxy Spica",
      "https://www.gsmarena.com/samsung_i5700_galaxy_spica-2965.php"
    ],
    [
      "E1390",
      "https://www.gsmarena.com/samsung_e1390-2979.php"
    ],
    [
      "E1160",
      "https://www.gsmarena.com/samsung_e1160-2978.php"
    ],
    [
      "E2130",
      "https://www.gsmarena.com/samsung_e2130-2992.php"
    ],
    [
      "E1130B",
      "https://www.gsmarena.com/samsung_e1130b-2977.php"
    ],
    [
      "T401G",
      "https://www.gsmarena.com/samsung_t401g-2975.php"
    ],
    [
      "B7620 Giorgio Armani",
      "https://www.gsmarena.com/samsung_b7620_giorgio_armani-2966.php"
    ],
    [
      "T939 Behold 2",
      "https://www.gsmarena.com/samsung_t939_behold_2-2955.php"
    ],
    [
      "W880 AMOLED 12M",
      "https://www.gsmarena.com/samsung_w880_amoled_12m-2948.php"
    ],
    [
      "Vodafone 360 H1",
      "https://www.gsmarena.com/samsung_vodafone_360_h1-2954.php"
    ],
    [
      "Vodafone 360 M1",
      "https://www.gsmarena.com/samsung_vodafone_360_m1-2953.php"
    ],
    [
      "B5310 CorbyPRO",
      "https://www.gsmarena.com/samsung_b5310_corbypro-2944.php"
    ],
    [
      "B3210 CorbyTXT",
      "https://www.gsmarena.com/samsung_b3210_corbytxt-2943.php"
    ],
    [
      "F480i",
      "https://www.gsmarena.com/samsung_f480i-2987.php"
    ],
    [
      "B7330 OmniaPRO",
      "https://www.gsmarena.com/samsung_b7330_omniapro-2940.php"
    ],
    [
      "E1085T",
      "https://www.gsmarena.com/samsung_e1085t-2988.php"
    ],
    [
      "E1080T",
      "https://www.gsmarena.com/samsung_e1080t-2941.php"
    ],
    [
      "S5510",
      "https://www.gsmarena.com/samsung_s5510-2952.php"
    ],
    [
      "S5230W Star WiFi",
      "https://www.gsmarena.com/samsung_s5230w_star_wifi-2935.php"
    ],
    [
      "S7550 Blue Earth",
      "https://www.gsmarena.com/samsung_s7550_blue_earth-2931.php"
    ],
    [
      "S3650 Corby",
      "https://www.gsmarena.com/samsung_s3650_corby-2908.php"
    ],
    [
      "T659 Scarlet",
      "https://www.gsmarena.com/samsung_t659_scarlet-2933.php"
    ],
    [
      "B3310",
      "https://www.gsmarena.com/samsung_b3310-2907.php"
    ],
    [
      "C3212",
      "https://www.gsmarena.com/samsung_c3212-2902.php"
    ],
    [
      "U450 DoubleTake",
      "https://www.gsmarena.com/samsung_u450_doubletake-3440.php"
    ],
    [
      "i220 Code",
      "https://www.gsmarena.com/samsung_i220_code-3441.php"
    ],
    [
      "R860 Caliber",
      "https://www.gsmarena.com/samsung_r860_caliber-3442.php"
    ],
    [
      "R520 Trill",
      "https://www.gsmarena.com/samsung_r520_trill-3443.php"
    ],
    [
      "M900 Moment",
      "https://www.gsmarena.com/samsung_m900_moment-3464.php"
    ],
    [
      "i350 Intrepid",
      "https://www.gsmarena.com/samsung_i350_intrepid-3465.php"
    ],
    [
      "M850 Instinct HD",
      "https://www.gsmarena.com/samsung_m850_instinct_hd-3466.php"
    ],
    [
      "U450 Intensity",
      "https://www.gsmarena.com/samsung_u450_intensity-3471.php"
    ],
    [
      "U960 Rogue",
      "https://www.gsmarena.com/samsung_u960_rogue-3472.php"
    ],
    [
      "S3100",
      "https://www.gsmarena.com/samsung_s3100-2901.php"
    ],
    [
      "A887 Solstice",
      "https://www.gsmarena.com/samsung_a887_solstice-2891.php"
    ],
    [
      "E2120",
      "https://www.gsmarena.com/samsung_e2120-2900.php"
    ],
    [
      "S5233T",
      "https://www.gsmarena.com/samsung_s5233t-3235.php"
    ],
    [
      "I6220 Star TV",
      "https://www.gsmarena.com/samsung_i6220_star_tv-2930.php"
    ],
    [
      "S9110",
      "https://www.gsmarena.com/samsung_s9110-2885.php"
    ],
    [
      "T469 Gravity 2",
      "https://www.gsmarena.com/samsung_t469_gravity_2-2884.php"
    ],
    [
      "T559 Comeback",
      "https://www.gsmarena.com/samsung_t559_comeback-2883.php"
    ],
    [
      "T746 Impact",
      "https://www.gsmarena.com/samsung_t746_impact-2870.php"
    ],
    [
      "S5600v Blade",
      "https://www.gsmarena.com/samsung_s5600v_blade-2860.php"
    ],
    [
      "S6700",
      "https://www.gsmarena.com/samsung_s6700-2855.php"
    ],
    [
      "C5510",
      "https://www.gsmarena.com/samsung_c5510-2856.php"
    ],
    [
      "M2510",
      "https://www.gsmarena.com/samsung_m2510-2854.php"
    ],
    [
      "M2310",
      "https://www.gsmarena.com/samsung_m2310-2853.php"
    ],
    [
      "W850",
      "https://www.gsmarena.com/samsung_w850-3349.php"
    ],
    [
      "S8000 Jet",
      "https://www.gsmarena.com/samsung_s8000_jet-2835.php"
    ],
    [
      "I8000 Omnia II",
      "https://www.gsmarena.com/samsung_i8000_omnia_ii-2836.php"
    ],
    [
      "B7610 OmniaPRO",
      "https://www.gsmarena.com/samsung_b7610_omniapro-2834.php"
    ],
    [
      "B7320 OmniaPRO",
      "https://www.gsmarena.com/samsung_b7320_omniapro-2838.php"
    ],
    [
      "B7300 OmniaLITE",
      "https://www.gsmarena.com/samsung_b7300_omnialite-2837.php"
    ],
    [
      "M8910 Pixon12",
      "https://www.gsmarena.com/samsung_m8910_pixon12-2813.php"
    ],
    [
      "E1107 Crest Solar",
      "https://www.gsmarena.com/samsung_e1107_crest_solar-2829.php"
    ],
    [
      "M2710 Beat Twist",
      "https://www.gsmarena.com/samsung_m2710_beat_twist-2816.php"
    ],
    [
      "S5050",
      "https://www.gsmarena.com/samsung_s5050-2808.php"
    ],
    [
      "T349",
      "https://www.gsmarena.com/samsung_t349-2807.php"
    ],
    [
      "S5200",
      "https://www.gsmarena.com/samsung_s5200-2806.php"
    ],
    [
      "i637 Jack",
      "https://www.gsmarena.com/samsung_i637_jack-2798.php"
    ],
    [
      "S3310",
      "https://www.gsmarena.com/samsung_s3310-2812.php"
    ],
    [
      "C3060R",
      "https://www.gsmarena.com/samsung_c3060r-2804.php"
    ],
    [
      "A167",
      "https://www.gsmarena.com/samsung_a167-2799.php"
    ],
    [
      "A177",
      "https://www.gsmarena.com/samsung_a177-2796.php"
    ],
    [
      "I7500 Galaxy",
      "https://www.gsmarena.com/samsung_i7500_galaxy-2791.php"
    ],
    [
      "J165",
      "https://www.gsmarena.com/samsung_j165-2797.php"
    ],
    [
      "I6210",
      "https://www.gsmarena.com/samsung_i6210-2823.php"
    ],
    [
      "E1360",
      "https://www.gsmarena.com/samsung_e1360-2784.php"
    ],
    [
      "E1210",
      "https://www.gsmarena.com/samsung_e1210-2783.php"
    ],
    [
      "A657",
      "https://www.gsmarena.com/samsung_a657-2781.php"
    ],
    [
      "A877 Impression",
      "https://www.gsmarena.com/samsung_a877_impression-2760.php"
    ],
    [
      "A257 Magnet",
      "https://www.gsmarena.com/samsung_a257_magnet-2795.php"
    ],
    [
      "Propel Pro",
      "https://www.gsmarena.com/samsung_propel_pro-2758.php"
    ],
    [
      "B2100 Xplorer",
      "https://www.gsmarena.com/samsung_b2100_xplorer-2747.php"
    ],
    [
      "S5230 Star",
      "https://www.gsmarena.com/samsung_s5230_star-2739.php"
    ],
    [
      "S5600 Preston",
      "https://www.gsmarena.com/samsung_s5600_preston-2738.php"
    ],
    [
      "i8910 Omnia HD",
      "https://www.gsmarena.com/samsung_i8910_omnia_hd-2691.php"
    ],
    [
      "S8300 UltraTOUCH",
      "https://www.gsmarena.com/samsung_s8300_ultratouch-2667.php"
    ],
    [
      "M7600 Beat DJ",
      "https://www.gsmarena.com/samsung_m7600_beat_dj-2684.php"
    ],
    [
      "M6710 Beat DISC",
      "https://www.gsmarena.com/samsung_m6710_beat_disc-2692.php"
    ],
    [
      "S7350 Ultra s",
      "https://www.gsmarena.com/samsung_s7350_ultra_s-2693.php"
    ],
    [
      "S7220 Ultra b",
      "https://www.gsmarena.com/samsung_s7220_ultra_b-2695.php"
    ],
    [
      "C5220",
      "https://www.gsmarena.com/samsung_c5220-2749.php"
    ],
    [
      "S3500",
      "https://www.gsmarena.com/samsung_s3500-2696.php"
    ],
    [
      "S3110",
      "https://www.gsmarena.com/samsung_s3110-2697.php"
    ],
    [
      "P250",
      "https://www.gsmarena.com/samsung_p250-2782.php"
    ],
    [
      "i7410",
      "https://www.gsmarena.com/samsung_i7410-2698.php"
    ],
    [
      "T929 Memoir",
      "https://www.gsmarena.com/samsung_t929_memoir-2663.php"
    ],
    [
      "C3050 Stratus",
      "https://www.gsmarena.com/samsung_c3050_stratus-2706.php"
    ],
    [
      "C3010",
      "https://www.gsmarena.com/samsung_c3010-2707.php"
    ],
    [
      "C3110",
      "https://www.gsmarena.com/samsung_c3110-2655.php"
    ],
    [
      "C6625",
      "https://www.gsmarena.com/samsung_c6625-2723.php"
    ],
    [
      "E1310",
      "https://www.gsmarena.com/samsung_e1310-2710.php"
    ],
    [
      "E1120",
      "https://www.gsmarena.com/samsung_e1120-2711.php"
    ],
    [
      "E1100",
      "https://www.gsmarena.com/samsung_e1100-2673.php"
    ],
    [
      "E1070",
      "https://www.gsmarena.com/samsung_e1070-2672.php"
    ],
    [
      "E2210B",
      "https://www.gsmarena.com/samsung_e2210b-2712.php"
    ],
    [
      "E1125",
      "https://www.gsmarena.com/samsung_e1125-2675.php"
    ],
    [
      "E2100B",
      "https://www.gsmarena.com/samsung_e2100b-2674.php"
    ],
    [
      "B520",
      "https://www.gsmarena.com/samsung_b520-2652.php"
    ],
    [
      "B5702",
      "https://www.gsmarena.com/samsung_b5702-2702.php"
    ],
    [
      "C5212",
      "https://www.gsmarena.com/samsung_c5212-2709.php"
    ],
    [
      "W259 Duos",
      "https://www.gsmarena.com/samsung_w259_duos-3463.php"
    ],
    [
      "SCH-W699",
      "https://www.gsmarena.com/samsung_sch_w699-3247.php"
    ],
    [
      "W299 Duos",
      "https://www.gsmarena.com/samsung_w299_duos-3246.php"
    ],
    [
      "S9402 Ego",
      "https://www.gsmarena.com/samsung_s9402_ego-2616.php"
    ],
    [
      "S3030 Tobi",
      "https://www.gsmarena.com/samsung_s3030_tobi-2610.php"
    ],
    [
      "U810 Renown",
      "https://www.gsmarena.com/samsung_u810_renown-2609.php"
    ],
    [
      "i770 Saga",
      "https://www.gsmarena.com/samsung_i770_saga-2608.php"
    ],
    [
      "A867 Eternity",
      "https://www.gsmarena.com/samsung_a867_eternity-2607.php"
    ],
    [
      "A777",
      "https://www.gsmarena.com/samsung_a777-2600.php"
    ],
    [
      "T919 Behold",
      "https://www.gsmarena.com/samsung_t919_behold-2587.php"
    ],
    [
      "T459 Gravity",
      "https://www.gsmarena.com/samsung_t459_gravity-2588.php"
    ],
    [
      "E2510",
      "https://www.gsmarena.com/samsung_e2510-2584.php"
    ],
    [
      "T219",
      "https://www.gsmarena.com/samsung_t219-2767.php"
    ],
    [
      "E1410",
      "https://www.gsmarena.com/samsung_e1410-2647.php"
    ],
    [
      "T119",
      "https://www.gsmarena.com/samsung_t119-2765.php"
    ],
    [
      "E1117",
      "https://www.gsmarena.com/samsung_e1117-2742.php"
    ],
    [
      "E1110",
      "https://www.gsmarena.com/samsung_e1110-2583.php"
    ],
    [
      "C6620",
      "https://www.gsmarena.com/samsung_c6620-2575.php"
    ],
    [
      "i907 Epix",
      "https://www.gsmarena.com/samsung_i907_epix-2566.php"
    ],
    [
      "A767 Propel",
      "https://www.gsmarena.com/samsung_a767_propel-2565.php"
    ],
    [
      "i7110",
      "https://www.gsmarena.com/samsung_i7110-2559.php"
    ],
    [
      "F275",
      "https://www.gsmarena.com/samsung_f275-3200.php"
    ],
    [
      "F270 Beat",
      "https://www.gsmarena.com/samsung_f270_beat-2543.php"
    ],
    [
      "M8800 Pixon",
      "https://www.gsmarena.com/samsung_m8800_pixon-2535.php"
    ],
    [
      "S3600",
      "https://www.gsmarena.com/samsung_s3600-2539.php"
    ],
    [
      "M7500 Emporio Armani",
      "https://www.gsmarena.com/samsung_m7500_emporio_armani-2529.php"
    ],
    [
      "M3200 Beat s",
      "https://www.gsmarena.com/samsung_m3200_beat_s-2528.php"
    ],
    [
      "T339",
      "https://www.gsmarena.com/samsung_t339-2593.php"
    ],
    [
      "T229",
      "https://www.gsmarena.com/samsung_t229-2592.php"
    ],
    [
      "A637",
      "https://www.gsmarena.com/samsung_a637-2551.php"
    ],
    [
      "C510",
      "https://www.gsmarena.com/samsung_c510-2545.php"
    ],
    [
      "A837 Rugby",
      "https://www.gsmarena.com/samsung_a837_rugby-2530.php"
    ],
    [
      "A237",
      "https://www.gsmarena.com/samsung_a237-2518.php"
    ],
    [
      "B320",
      "https://www.gsmarena.com/samsung_b320-2548.php"
    ],
    [
      "B210",
      "https://www.gsmarena.com/samsung_b210-2510.php"
    ],
    [
      "M3510 Beat b",
      "https://www.gsmarena.com/samsung_m3510_beat_b-2509.php"
    ],
    [
      "M200",
      "https://www.gsmarena.com/samsung_m200-2508.php"
    ],
    [
      "P270",
      "https://www.gsmarena.com/samsung_p270-2722.php"
    ],
    [
      "B2700",
      "https://www.gsmarena.com/samsung_b2700-2507.php"
    ],
    [
      "F268",
      "https://www.gsmarena.com/samsung_f268-2506.php"
    ],
    [
      "T109",
      "https://www.gsmarena.com/samsung_t109-2550.php"
    ],
    [
      "E200 ECO",
      "https://www.gsmarena.com/samsung_e200_eco-2505.php"
    ],
    [
      "D980",
      "https://www.gsmarena.com/samsung_d980-2500.php"
    ],
    [
      "B510",
      "https://www.gsmarena.com/samsung_b510-2501.php"
    ],
    [
      "E215",
      "https://www.gsmarena.com/samsung_e215-2502.php"
    ],
    [
      "B130",
      "https://www.gsmarena.com/samsung_b130-2499.php"
    ],
    [
      "i8510 INNOV8",
      "https://www.gsmarena.com/samsung_i8510_innov8-2471.php"
    ],
    [
      "S7330",
      "https://www.gsmarena.com/samsung_s7330-2487.php"
    ],
    [
      "i740",
      "https://www.gsmarena.com/samsung_i740-2472.php"
    ],
    [
      "M150",
      "https://www.gsmarena.com/samsung_m150-2466.php"
    ],
    [
      "J800 Luxe",
      "https://www.gsmarena.com/samsung_j800_luxe-2436.php"
    ],
    [
      "M140",
      "https://www.gsmarena.com/samsung_m140-2490.php"
    ],
    [
      "M130",
      "https://www.gsmarena.com/samsung_m130-2489.php"
    ],
    [
      "L700",
      "https://www.gsmarena.com/samsung_l700-2435.php"
    ],
    [
      "i900 Omnia",
      "https://www.gsmarena.com/samsung_i900_omnia-2422.php"
    ],
    [
      "U800 Soul b",
      "https://www.gsmarena.com/samsung_u800_soul_b-2389.php"
    ],
    [
      "L870",
      "https://www.gsmarena.com/samsung_l870-2387.php"
    ],
    [
      "Impact sf",
      "https://www.gsmarena.com/samsung_impact_sf-2443.php"
    ],
    [
      "M620",
      "https://www.gsmarena.com/samsung_m620-2360.php"
    ],
    [
      "B300",
      "https://www.gsmarena.com/samsung_b300-2463.php"
    ],
    [
      "Impact b",
      "https://www.gsmarena.com/samsung_impact_b-2442.php"
    ],
    [
      "Impact",
      "https://www.gsmarena.com/samsung_impact-2441.php"
    ],
    [
      "D780",
      "https://www.gsmarena.com/samsung_d780-2340.php"
    ],
    [
      "V820L",
      "https://www.gsmarena.com/samsung_v820l-2757.php"
    ],
    [
      "F110",
      "https://www.gsmarena.com/samsung_f110-2315.php"
    ],
    [
      "G400 Soul",
      "https://www.gsmarena.com/samsung_g400_soul-2314.php"
    ],
    [
      "U900 Soul",
      "https://www.gsmarena.com/samsung_u900_soul-2243.php"
    ],
    [
      "L810v Steel",
      "https://www.gsmarena.com/samsung_l810v_steel-2410.php"
    ],
    [
      "F480",
      "https://www.gsmarena.com/samsung_f480-2268.php"
    ],
    [
      "G810",
      "https://www.gsmarena.com/samsung_g810-2257.php"
    ],
    [
      "M310",
      "https://www.gsmarena.com/samsung_m310-2423.php"
    ],
    [
      "B200",
      "https://www.gsmarena.com/samsung_b200-2359.php"
    ],
    [
      "B110",
      "https://www.gsmarena.com/samsung_b110-2358.php"
    ],
    [
      "A827 Access",
      "https://www.gsmarena.com/samsung_a827_access-2408.php"
    ],
    [
      "P960",
      "https://www.gsmarena.com/samsung_p960-2259.php"
    ],
    [
      "P220",
      "https://www.gsmarena.com/samsung_p220-2404.php"
    ],
    [
      "P180",
      "https://www.gsmarena.com/samsung_p180-2403.php"
    ],
    [
      "i200",
      "https://www.gsmarena.com/samsung_i200-2260.php"
    ],
    [
      "F400",
      "https://www.gsmarena.com/samsung_f400-2261.php"
    ],
    [
      "J700",
      "https://www.gsmarena.com/samsung_j700-2262.php"
    ],
    [
      "J630",
      "https://www.gsmarena.com/samsung_j630-2323.php"
    ],
    [
      "J210",
      "https://www.gsmarena.com/samsung_j210-2324.php"
    ],
    [
      "J150",
      "https://www.gsmarena.com/samsung_j150-2263.php"
    ],
    [
      "E251",
      "https://www.gsmarena.com/samsung_e251-2265.php"
    ],
    [
      "L770",
      "https://www.gsmarena.com/samsung_l770-2264.php"
    ],
    [
      "A746",
      "https://www.gsmarena.com/samsung_a746-2373.php"
    ],
    [
      "A727",
      "https://www.gsmarena.com/samsung_a727-2378.php"
    ],
    [
      "A711",
      "https://www.gsmarena.com/samsung_a711-2450.php"
    ],
    [
      "A717",
      "https://www.gsmarena.com/samsung_a717-2377.php"
    ],
    [
      "A437",
      "https://www.gsmarena.com/samsung_a437-2376.php"
    ],
    [
      "A127",
      "https://www.gsmarena.com/samsung_a127-2375.php"
    ],
    [
      "A117",
      "https://www.gsmarena.com/samsung_a117-2374.php"
    ],
    [
      "SPH-i325 Ace",
      "https://www.gsmarena.com/samsung_sph_i325_ace-2320.php"
    ],
    [
      "i640",
      "https://www.gsmarena.com/samsung_i640-2289.php"
    ],
    [
      "L320",
      "https://www.gsmarena.com/samsung_l320-2234.php"
    ],
    [
      "L310",
      "https://www.gsmarena.com/samsung_l310-2233.php"
    ],
    [
      "L170",
      "https://www.gsmarena.com/samsung_l170-2219.php"
    ],
    [
      "T819",
      "https://www.gsmarena.com/samsung_t819-2220.php"
    ],
    [
      "F490",
      "https://www.gsmarena.com/samsung_f490-2203.php"
    ],
    [
      "G800",
      "https://www.gsmarena.com/samsung_g800-2145.php"
    ],
    [
      "C500",
      "https://www.gsmarena.com/samsung_c500-2546.php"
    ],
    [
      "Serenata",
      "https://www.gsmarena.com/samsung_serenata-2130.php"
    ],
    [
      "D880 Duos",
      "https://www.gsmarena.com/samsung_d880_duos-2156.php"
    ],
    [
      "i780",
      "https://www.gsmarena.com/samsung_i780-2124.php"
    ],
    [
      "i560",
      "https://www.gsmarena.com/samsung_i560-2154.php"
    ],
    [
      "i550",
      "https://www.gsmarena.com/samsung_i550-2115.php"
    ],
    [
      "F330",
      "https://www.gsmarena.com/samsung_f330-2090.php"
    ],
    [
      "F250",
      "https://www.gsmarena.com/samsung_f250-2186.php"
    ],
    [
      "i450",
      "https://www.gsmarena.com/samsung_i450-2076.php"
    ],
    [
      "Armani",
      "https://www.gsmarena.com/samsung_armani-2046.php"
    ],
    [
      "J610",
      "https://www.gsmarena.com/samsung_j610-2164.php"
    ],
    [
      "J400",
      "https://www.gsmarena.com/samsung_j400-2224.php"
    ],
    [
      "P260",
      "https://www.gsmarena.com/samsung_p260-2045.php"
    ],
    [
      "J750",
      "https://www.gsmarena.com/samsung_j750-2161.php"
    ],
    [
      "A737",
      "https://www.gsmarena.com/samsung_a737-2214.php"
    ],
    [
      "A517",
      "https://www.gsmarena.com/samsung_a517-2215.php"
    ],
    [
      "T639",
      "https://www.gsmarena.com/samsung_t639-2384.php"
    ],
    [
      "T509",
      "https://www.gsmarena.com/samsung_t509-2383.php"
    ],
    [
      "T439",
      "https://www.gsmarena.com/samsung_t439-2217.php"
    ],
    [
      "T429",
      "https://www.gsmarena.com/samsung_t429-2382.php"
    ],
    [
      "T409",
      "https://www.gsmarena.com/samsung_t409-2381.php"
    ],
    [
      "T739 Katalyst",
      "https://www.gsmarena.com/samsung_t739_katalyst-2216.php"
    ],
    [
      "T729 Blast",
      "https://www.gsmarena.com/samsung_t729_blast-2380.php"
    ],
    [
      "T539 Beat",
      "https://www.gsmarena.com/samsung_t539_beat-2379.php"
    ],
    [
      "L600",
      "https://www.gsmarena.com/samsung_l600-2047.php"
    ],
    [
      "Z630",
      "https://www.gsmarena.com/samsung_z630-2044.php"
    ],
    [
      "G600",
      "https://www.gsmarena.com/samsung_g600-2039.php"
    ],
    [
      "M610",
      "https://www.gsmarena.com/samsung_m610-2111.php"
    ],
    [
      "M600",
      "https://www.gsmarena.com/samsung_m600-2092.php"
    ],
    [
      "M110",
      "https://www.gsmarena.com/samsung_m110-2196.php"
    ],
    [
      "B500",
      "https://www.gsmarena.com/samsung_b500-2318.php"
    ],
    [
      "B100",
      "https://www.gsmarena.com/samsung_b100-2317.php"
    ],
    [
      "S730i",
      "https://www.gsmarena.com/samsung_s730i-2034.php"
    ],
    [
      "S720i",
      "https://www.gsmarena.com/samsung_s720i-2043.php"
    ],
    [
      "J200",
      "https://www.gsmarena.com/samsung_j200-2093.php"
    ],
    [
      "L760",
      "https://www.gsmarena.com/samsung_l760-2025.php"
    ],
    [
      "F700",
      "https://www.gsmarena.com/samsung_f700-1849.php"
    ],
    [
      "E950",
      "https://www.gsmarena.com/samsung_e950-2015.php"
    ],
    [
      "F210",
      "https://www.gsmarena.com/samsung_f210-2014.php"
    ],
    [
      "F200",
      "https://www.gsmarena.com/samsung_f200-2016.php"
    ],
    [
      "i620",
      "https://www.gsmarena.com/samsung_i620-1998.php"
    ],
    [
      "A411",
      "https://www.gsmarena.com/samsung_a411-2138.php"
    ],
    [
      "ZV60",
      "https://www.gsmarena.com/samsung_zv60-2035.php"
    ],
    [
      "Z240",
      "https://www.gsmarena.com/samsung_z240-1969.php"
    ],
    [
      "Z170",
      "https://www.gsmarena.com/samsung_z170-2030.php"
    ],
    [
      "J600",
      "https://www.gsmarena.com/samsung_j600-1970.php"
    ],
    [
      "i400",
      "https://www.gsmarena.com/samsung_i400-1967.php"
    ],
    [
      "U700",
      "https://www.gsmarena.com/samsung_u700-1852.php"
    ],
    [
      "U600",
      "https://www.gsmarena.com/samsung_u600-1851.php"
    ],
    [
      "U300",
      "https://www.gsmarena.com/samsung_u300-1853.php"
    ],
    [
      "U100",
      "https://www.gsmarena.com/samsung_u100-1854.php"
    ],
    [
      "i710",
      "https://www.gsmarena.com/samsung_i710-1916.php"
    ],
    [
      "i520",
      "https://www.gsmarena.com/samsung_i520-1906.php"
    ],
    [
      "F520",
      "https://www.gsmarena.com/samsung_f520-1856.php"
    ],
    [
      "F510",
      "https://www.gsmarena.com/samsung_f510-1855.php"
    ],
    [
      "P110",
      "https://www.gsmarena.com/samsung_p110-1868.php"
    ],
    [
      "E840",
      "https://www.gsmarena.com/samsung_e840-1866.php"
    ],
    [
      "E830",
      "https://www.gsmarena.com/samsung_e830-1867.php"
    ],
    [
      "E740",
      "https://www.gsmarena.com/samsung_e740-1870.php"
    ],
    [
      "E590",
      "https://www.gsmarena.com/samsung_e590-1869.php"
    ],
    [
      "E230",
      "https://www.gsmarena.com/samsung_e230-2083.php"
    ],
    [
      "E210",
      "https://www.gsmarena.com/samsung_e210-2082.php"
    ],
    [
      "E200",
      "https://www.gsmarena.com/samsung_e200-1919.php"
    ],
    [
      "C520",
      "https://www.gsmarena.com/samsung_c520-1971.php"
    ],
    [
      "C450",
      "https://www.gsmarena.com/samsung_c450-2042.php"
    ],
    [
      "C275",
      "https://www.gsmarena.com/samsung_c275-2737.php"
    ],
    [
      "C270",
      "https://www.gsmarena.com/samsung_c270-2751.php"
    ],
    [
      "C260",
      "https://www.gsmarena.com/samsung_c260-1920.php"
    ],
    [
      "C250",
      "https://www.gsmarena.com/samsung_c250-1921.php"
    ],
    [
      "i600",
      "https://www.gsmarena.com/samsung_i600-1807.php"
    ],
    [
      "i617 BlackJack II",
      "https://www.gsmarena.com/samsung_i617_blackjack_ii-2213.php"
    ],
    [
      "i607 BlackJack",
      "https://www.gsmarena.com/samsung_i607_blackjack-2212.php"
    ],
    [
      "F500",
      "https://www.gsmarena.com/samsung_f500-1806.php"
    ],
    [
      "F300",
      "https://www.gsmarena.com/samsung_f300-1805.php"
    ],
    [
      "P940",
      "https://www.gsmarena.com/samsung_p940-1809.php"
    ],
    [
      "P930",
      "https://www.gsmarena.com/samsung_p930-1808.php"
    ],
    [
      "E790",
      "https://www.gsmarena.com/samsung_e790-1850.php"
    ],
    [
      "E490",
      "https://www.gsmarena.com/samsung_e490-1784.php"
    ],
    [
      "E480",
      "https://www.gsmarena.com/samsung_e480-1783.php"
    ],
    [
      "E390",
      "https://www.gsmarena.com/samsung_e390-1770.php"
    ],
    [
      "X550",
      "https://www.gsmarena.com/samsung_x550-2131.php"
    ],
    [
      "X540",
      "https://www.gsmarena.com/samsung_x540-1816.php"
    ],
    [
      "E250",
      "https://www.gsmarena.com/samsung_e250-1772.php"
    ],
    [
      "X520",
      "https://www.gsmarena.com/samsung_x520-1771.php"
    ],
    [
      "X530",
      "https://www.gsmarena.com/samsung_x530-1769.php"
    ],
    [
      "C300",
      "https://www.gsmarena.com/samsung_c300-1768.php"
    ],
    [
      "M300",
      "https://www.gsmarena.com/samsung_m300-1992.php"
    ],
    [
      "E690",
      "https://www.gsmarena.com/samsung_e690-1796.php"
    ],
    [
      "E570",
      "https://www.gsmarena.com/samsung_e570-1756.php"
    ],
    [
      "E420",
      "https://www.gsmarena.com/samsung_e420-1757.php"
    ],
    [
      "X830",
      "https://www.gsmarena.com/samsung_x830-1681.php"
    ],
    [
      "E890",
      "https://www.gsmarena.com/samsung_e890-1797.php"
    ],
    [
      "E898",
      "https://www.gsmarena.com/samsung_e898-1779.php"
    ],
    [
      "P310",
      "https://www.gsmarena.com/samsung_p310-1682.php"
    ],
    [
      "Z720",
      "https://www.gsmarena.com/samsung_z720-1677.php"
    ],
    [
      "Z650i",
      "https://www.gsmarena.com/samsung_z650i-1782.php"
    ],
    [
      "Z620",
      "https://www.gsmarena.com/samsung_z620-1676.php"
    ],
    [
      "Z370",
      "https://www.gsmarena.com/samsung_z370-1675.php"
    ],
    [
      "Z360",
      "https://www.gsmarena.com/samsung_z360-2069.php"
    ],
    [
      "P200",
      "https://www.gsmarena.com/samsung_p200-1643.php"
    ],
    [
      "S501i",
      "https://www.gsmarena.com/samsung_s501i-1630.php"
    ],
    [
      "S401i",
      "https://www.gsmarena.com/samsung_s401i-1629.php"
    ],
    [
      "X510",
      "https://www.gsmarena.com/samsung_x510-1762.php"
    ],
    [
      "X500",
      "https://www.gsmarena.com/samsung_x500-1615.php"
    ],
    [
      "D840",
      "https://www.gsmarena.com/samsung_d840-1614.php"
    ],
    [
      "C400",
      "https://www.gsmarena.com/samsung_c400-1707.php"
    ],
    [
      "C240",
      "https://www.gsmarena.com/samsung_c240-1616.php"
    ],
    [
      "C180",
      "https://www.gsmarena.com/samsung_c180-2041.php"
    ],
    [
      "C170",
      "https://www.gsmarena.com/samsung_c170-2010.php"
    ],
    [
      "C160",
      "https://www.gsmarena.com/samsung_c160-1922.php"
    ],
    [
      "C140",
      "https://www.gsmarena.com/samsung_c140-1830.php"
    ],
    [
      "D830",
      "https://www.gsmarena.com/samsung_d830-1600.php"
    ],
    [
      "D900i",
      "https://www.gsmarena.com/samsung_d900i-1989.php"
    ],
    [
      "D900",
      "https://www.gsmarena.com/samsung_d900-1587.php"
    ],
    [
      "X820",
      "https://www.gsmarena.com/samsung_x820-1556.php"
    ],
    [
      "C130",
      "https://www.gsmarena.com/samsung_c130-1608.php"
    ],
    [
      "E500",
      "https://www.gsmarena.com/samsung_e500-825.php"
    ],
    [
      "E380",
      "https://www.gsmarena.com/samsung_e380-1624.php"
    ],
    [
      "D870",
      "https://www.gsmarena.com/samsung_d870-1506.php"
    ],
    [
      "D780 flip",
      "https://www.gsmarena.com/samsung_d780_flip-1507.php"
    ],
    [
      "E900",
      "https://www.gsmarena.com/samsung_e900-1508.php"
    ],
    [
      "Z400",
      "https://www.gsmarena.com/samsung_z400-1510.php"
    ],
    [
      "Z550",
      "https://www.gsmarena.com/samsung_z550-1509.php"
    ],
    [
      "i320",
      "https://www.gsmarena.com/samsung_i320-1442.php"
    ],
    [
      "i310",
      "https://www.gsmarena.com/samsung_i310-1496.php"
    ],
    [
      "ZV50",
      "https://www.gsmarena.com/samsung_zv50-1592.php"
    ],
    [
      "ZV40",
      "https://www.gsmarena.com/samsung_zv40-1642.php"
    ],
    [
      "Z560",
      "https://www.gsmarena.com/samsung_z560-1439.php"
    ],
    [
      "P920",
      "https://www.gsmarena.com/samsung_p920-1599.php"
    ],
    [
      "P900",
      "https://www.gsmarena.com/samsung_p900-1427.php"
    ],
    [
      "P910",
      "https://www.gsmarena.com/samsung_p910-1440.php"
    ],
    [
      "T629",
      "https://www.gsmarena.com/samsung_t629-2754.php"
    ],
    [
      "D520",
      "https://www.gsmarena.com/samsung_d520-1441.php"
    ],
    [
      "D300",
      "https://www.gsmarena.com/samsung_d300-1478.php"
    ],
    [
      "Z710",
      "https://www.gsmarena.com/samsung_z710-1450.php"
    ],
    [
      "Z600",
      "https://www.gsmarena.com/samsung_z600-1449.php"
    ],
    [
      "Z520",
      "https://www.gsmarena.com/samsung_z520-1447.php"
    ],
    [
      "Z350",
      "https://www.gsmarena.com/samsung_z350-1446.php"
    ],
    [
      "Z230",
      "https://www.gsmarena.com/samsung_z230-1725.php"
    ],
    [
      "Z330",
      "https://www.gsmarena.com/samsung_z330-1448.php"
    ],
    [
      "Z310",
      "https://www.gsmarena.com/samsung_z310-1445.php"
    ],
    [
      "Z150",
      "https://www.gsmarena.com/samsung_z150-1444.php"
    ],
    [
      "X680",
      "https://www.gsmarena.com/samsung_x680-1620.php"
    ],
    [
      "X630",
      "https://www.gsmarena.com/samsung_x630-1619.php"
    ],
    [
      "X210",
      "https://www.gsmarena.com/samsung_x210-1579.php"
    ],
    [
      "X300",
      "https://www.gsmarena.com/samsung_x300-1479.php"
    ],
    [
      "X160",
      "https://www.gsmarena.com/samsung_x160-1555.php"
    ],
    [
      "E870",
      "https://www.gsmarena.com/samsung_e870-1424.php"
    ],
    [
      "E780",
      "https://www.gsmarena.com/samsung_e780-1617.php"
    ],
    [
      "i300x",
      "https://www.gsmarena.com/samsung_i300x-1408.php"
    ],
    [
      "D820",
      "https://www.gsmarena.com/samsung_d820-1356.php"
    ],
    [
      "D810",
      "https://www.gsmarena.com/samsung_d810-1416.php"
    ],
    [
      "D800",
      "https://www.gsmarena.com/samsung_d800-1355.php"
    ],
    [
      "Z540",
      "https://www.gsmarena.com/samsung_z540-1354.php"
    ],
    [
      "Z510",
      "https://www.gsmarena.com/samsung_z510-1353.php"
    ],
    [
      "P300",
      "https://www.gsmarena.com/samsung_p300-1351.php"
    ],
    [
      "E770",
      "https://www.gsmarena.com/samsung_e770-1344.php"
    ],
    [
      "E370",
      "https://www.gsmarena.com/samsung_e370-1554.php"
    ],
    [
      "E360",
      "https://www.gsmarena.com/samsung_e360-1343.php"
    ],
    [
      "ZV30",
      "https://www.gsmarena.com/samsung_zv30-1342.php"
    ],
    [
      "ZV10",
      "https://www.gsmarena.com/samsung_zv10-1357.php"
    ],
    [
      "S400i",
      "https://www.gsmarena.com/samsung_s400i-1477.php"
    ],
    [
      "Z320i",
      "https://www.gsmarena.com/samsung_z320i-1398.php"
    ],
    [
      "S500i",
      "https://www.gsmarena.com/samsung_s500i-1341.php"
    ],
    [
      "E860",
      "https://www.gsmarena.com/samsung_e860-1340.php"
    ],
    [
      "Serene",
      "https://www.gsmarena.com/samsung_serene-1314.php"
    ],
    [
      "D600",
      "https://www.gsmarena.com/samsung_d600-1103.php"
    ],
    [
      "P850",
      "https://www.gsmarena.com/samsung_p850-1113.php"
    ],
    [
      "E730",
      "https://www.gsmarena.com/samsung_e730-1115.php"
    ],
    [
      "D510",
      "https://www.gsmarena.com/samsung_d510-1130.php"
    ],
    [
      "X700",
      "https://www.gsmarena.com/samsung_x700-1298.php"
    ],
    [
      "X670",
      "https://www.gsmarena.com/samsung_x670-1545.php"
    ],
    [
      "X660",
      "https://www.gsmarena.com/samsung_x660-1316.php"
    ],
    [
      "X650",
      "https://www.gsmarena.com/samsung_x650-1567.php"
    ],
    [
      "X490",
      "https://www.gsmarena.com/samsung_x490-1419.php"
    ],
    [
      "X200",
      "https://www.gsmarena.com/samsung_x200-1317.php"
    ],
    [
      "X150",
      "https://www.gsmarena.com/samsung_x150-1381.php"
    ],
    [
      "Z500",
      "https://www.gsmarena.com/samsung_z500-1050.php"
    ],
    [
      "Z300",
      "https://www.gsmarena.com/samsung_z300-1049.php"
    ],
    [
      "Z130",
      "https://www.gsmarena.com/samsung_z130-1051.php"
    ],
    [
      "Z140",
      "https://www.gsmarena.com/samsung_z140-1142.php"
    ],
    [
      "D550",
      "https://www.gsmarena.com/samsung_d550-1206.php"
    ],
    [
      "P860",
      "https://www.gsmarena.com/samsung_p860-1114.php"
    ],
    [
      "Z700",
      "https://www.gsmarena.com/samsung_z700-1105.php"
    ],
    [
      "D730",
      "https://www.gsmarena.com/samsung_d730-1124.php"
    ],
    [
      "i750",
      "https://www.gsmarena.com/samsung_i750-1125.php"
    ],
    [
      "D720",
      "https://www.gsmarena.com/samsung_d720-1052.php"
    ],
    [
      "E760",
      "https://www.gsmarena.com/samsung_e760-1251.php"
    ],
    [
      "E750",
      "https://www.gsmarena.com/samsung_e750-1127.php"
    ],
    [
      "E720",
      "https://www.gsmarena.com/samsung_e720-1021.php"
    ],
    [
      "E880",
      "https://www.gsmarena.com/samsung_e880-1143.php"
    ],
    [
      "i300",
      "https://www.gsmarena.com/samsung_i300-1104.php"
    ],
    [
      "E350",
      "https://www.gsmarena.com/samsung_e350-1048.php"
    ],
    [
      "E640",
      "https://www.gsmarena.com/samsung_e640-1128.php"
    ],
    [
      "E620",
      "https://www.gsmarena.com/samsung_e620-1126.php"
    ],
    [
      "E530",
      "https://www.gsmarena.com/samsung_e530-1129.php"
    ],
    [
      "E340",
      "https://www.gsmarena.com/samsung_e340-1078.php"
    ],
    [
      "X810",
      "https://www.gsmarena.com/samsung_x810-1209.php"
    ],
    [
      "X800",
      "https://www.gsmarena.com/samsung_x800-1131.php"
    ],
    [
      "C120",
      "https://www.gsmarena.com/samsung_c120-1380.php"
    ],
    [
      "C230",
      "https://www.gsmarena.com/samsung_c230-1208.php"
    ],
    [
      "C210",
      "https://www.gsmarena.com/samsung_c210-1265.php"
    ],
    [
      "X640",
      "https://www.gsmarena.com/samsung_x640-1023.php"
    ],
    [
      "X480",
      "https://www.gsmarena.com/samsung_x480-1022.php"
    ],
    [
      "X620",
      "https://www.gsmarena.com/samsung_x620-1141.php"
    ],
    [
      "X140",
      "https://www.gsmarena.com/samsung_x140-1102.php"
    ],
    [
      "S410i",
      "https://www.gsmarena.com/samsung_s410i-1315.php"
    ],
    [
      "S342i",
      "https://www.gsmarena.com/samsung_s342i-1148.php"
    ],
    [
      "SCH-B100",
      "https://www.gsmarena.com/samsung_sch_b100-1024.php"
    ],
    [
      "D500",
      "https://www.gsmarena.com/samsung_d500-900.php"
    ],
    [
      "P730",
      "https://www.gsmarena.com/samsung_p730-707.php"
    ],
    [
      "P710",
      "https://www.gsmarena.com/samsung_p710-715.php"
    ],
    [
      "D428",
      "https://www.gsmarena.com/samsung_d428-1005.php"
    ],
    [
      "D488",
      "https://www.gsmarena.com/samsung_d488-1003.php"
    ],
    [
      "E850",
      "https://www.gsmarena.com/samsung_e850-964.php"
    ],
    [
      "E630",
      "https://www.gsmarena.com/samsung_e630-932.php"
    ],
    [
      "E810",
      "https://www.gsmarena.com/samsung_e810-716.php"
    ],
    [
      "E800",
      "https://www.gsmarena.com/samsung_e800-709.php"
    ],
    [
      "Z110",
      "https://www.gsmarena.com/samsung_z110-1080.php"
    ],
    [
      "D710",
      "https://www.gsmarena.com/samsung_d710-718.php"
    ],
    [
      "C200",
      "https://www.gsmarena.com/samsung_c200-917.php"
    ],
    [
      "i530",
      "https://www.gsmarena.com/samsung_i530-834.php"
    ],
    [
      "i505",
      "https://www.gsmarena.com/samsung_i505-643.php"
    ],
    [
      "i500",
      "https://www.gsmarena.com/samsung_i500-440.php"
    ],
    [
      "i250",
      "https://www.gsmarena.com/samsung_i250-807.php"
    ],
    [
      "i700",
      "https://www.gsmarena.com/samsung_i700-441.php"
    ],
    [
      "X910",
      "https://www.gsmarena.com/samsung_x910-720.php"
    ],
    [
      "X900",
      "https://www.gsmarena.com/samsung_x900-719.php"
    ],
    [
      "X610",
      "https://www.gsmarena.com/samsung_x610-800.php"
    ],
    [
      "X460",
      "https://www.gsmarena.com/samsung_x460-870.php"
    ],
    [
      "X120",
      "https://www.gsmarena.com/samsung_x120-799.php"
    ],
    [
      "E330",
      "https://www.gsmarena.com/samsung_e330-833.php"
    ],
    [
      "E310",
      "https://www.gsmarena.com/samsung_e310-832.php"
    ],
    [
      "E300",
      "https://www.gsmarena.com/samsung_e300-717.php"
    ],
    [
      "C110",
      "https://www.gsmarena.com/samsung_c110-721.php"
    ],
    [
      "Z107",
      "https://www.gsmarena.com/samsung_z107-1014.php"
    ],
    [
      "Z105",
      "https://www.gsmarena.com/samsung_z105-675.php"
    ],
    [
      "E610",
      "https://www.gsmarena.com/samsung_e610-765.php"
    ],
    [
      "E600",
      "https://www.gsmarena.com/samsung_e600-606.php"
    ],
    [
      "P510",
      "https://www.gsmarena.com/samsung_p510-687.php"
    ],
    [
      "E410",
      "https://www.gsmarena.com/samsung_e410-608.php"
    ],
    [
      "X450",
      "https://www.gsmarena.com/samsung_x450-614.php"
    ],
    [
      "X430",
      "https://www.gsmarena.com/samsung_x430-588.php"
    ],
    [
      "P705",
      "https://www.gsmarena.com/samsung_p705-531.php"
    ],
    [
      "D410",
      "https://www.gsmarena.com/samsung_d410-545.php"
    ],
    [
      "X600",
      "https://www.gsmarena.com/samsung_x600-511.php"
    ],
    [
      "X100",
      "https://www.gsmarena.com/samsung_x100-494.php"
    ],
    [
      "E715",
      "https://www.gsmarena.com/samsung_e715-486.php"
    ],
    [
      "E700",
      "https://www.gsmarena.com/samsung_e700-558.php"
    ],
    [
      "C100",
      "https://www.gsmarena.com/samsung_c100-444.php"
    ],
    [
      "D700",
      "https://www.gsmarena.com/samsung_d700-439.php"
    ],
    [
      "P500",
      "https://www.gsmarena.com/samsung_p500-438.php"
    ],
    [
      "E105",
      "https://www.gsmarena.com/samsung_e105-462.php"
    ],
    [
      "E100",
      "https://www.gsmarena.com/samsung_e100-526.php"
    ],
    [
      "X410",
      "https://www.gsmarena.com/samsung_x410-437.php"
    ],
    [
      "X400",
      "https://www.gsmarena.com/samsung_x400-436.php"
    ],
    [
      "S200",
      "https://www.gsmarena.com/samsung_s200-455.php"
    ],
    [
      "E400",
      "https://www.gsmarena.com/samsung_e400-435.php"
    ],
    [
      "D100",
      "https://www.gsmarena.com/samsung_d100-490.php"
    ],
    [
      "Watch Phone",
      "https://www.gsmarena.com/samsung_watch_phone-418.php"
    ],
    [
      "V200",
      "https://www.gsmarena.com/samsung_v200-417.php"
    ],
    [
      "Z100",
      "https://www.gsmarena.com/samsung_z100-537.php"
    ],
    [
      "P100",
      "https://www.gsmarena.com/samsung_p100-457.php"
    ],
    [
      "P400",
      "https://www.gsmarena.com/samsung_p400-377.php"
    ],
    [
      "T700",
      "https://www.gsmarena.com/samsung_t700-376.php"
    ],
    [
      "S500",
      "https://www.gsmarena.com/samsung_s500-520.php"
    ],
    [
      "T500",
      "https://www.gsmarena.com/samsung_t500-378.php"
    ],
    [
      "S300",
      "https://www.gsmarena.com/samsung_s300-375.php"
    ],
    [
      "T400",
      "https://www.gsmarena.com/samsung_t400-374.php"
    ],
    [
      "T200",
      "https://www.gsmarena.com/samsung_t200-379.php"
    ],
    [
      "V100",
      "https://www.gsmarena.com/samsung_v100-314.php"
    ],
    [
      "S100",
      "https://www.gsmarena.com/samsung_s100-313.php"
    ],
    [
      "Q300",
      "https://www.gsmarena.com/samsung_q300-312.php"
    ],
    [
      "A800",
      "https://www.gsmarena.com/samsung_a800-347.php"
    ],
    [
      "A500",
      "https://www.gsmarena.com/samsung_a500-311.php"
    ],
    [
      "T100",
      "https://www.gsmarena.com/samsung_t100-308.php"
    ],
    [
      "Q200",
      "https://www.gsmarena.com/samsung_q200-307.php"
    ],
    [
      "N620",
      "https://www.gsmarena.com/samsung_n620-309.php"
    ],
    [
      "N500",
      "https://www.gsmarena.com/samsung_n500-328.php"
    ],
    [
      "N400",
      "https://www.gsmarena.com/samsung_n400-277.php"
    ],
    [
      "Q105",
      "https://www.gsmarena.com/samsung_q105-293.php"
    ],
    [
      "N300",
      "https://www.gsmarena.com/samsung_n300-261.php"
    ],
    [
      "A400",
      "https://www.gsmarena.com/samsung_a400-276.php"
    ],
    [
      "A300",
      "https://www.gsmarena.com/samsung_a300-260.php"
    ],
    [
      "R220",
      "https://www.gsmarena.com/samsung_r220-291.php"
    ],
    [
      "R210",
      "https://www.gsmarena.com/samsung_r210-290.php"
    ],
    [
      "R200",
      "https://www.gsmarena.com/samsung_r200-289.php"
    ],
    [
      "A200",
      "https://www.gsmarena.com/samsung_a200-259.php"
    ],
    [
      "Q100",
      "https://www.gsmarena.com/samsung_q100-229.php"
    ],
    [
      "N105",
      "https://www.gsmarena.com/samsung_n105-292.php"
    ],
    [
      "N100",
      "https://www.gsmarena.com/samsung_n100-212.php"
    ],
    [
      "M100",
      "https://www.gsmarena.com/samsung_m100-208.php"
    ],
    [
      "A110",
      "https://www.gsmarena.com/samsung_a110-207.php"
    ],
    [
      "A100",
      "https://www.gsmarena.com/samsung_a100-55.php"
    ],
    [
      "SGH-2400",
      "https://www.gsmarena.com/samsung_sgh_2400-54.php"
    ],
    [
      "SGH-2200",
      "https://www.gsmarena.com/samsung_sgh_2200-53.php"
    ],
    [
      "SGH-2100",
      "https://www.gsmarena.com/samsung_sgh_2100-52.php"
    ],
    [
      "SGH-810",
      "https://www.gsmarena.com/samsung_sgh_810-51.php"
    ],
    [
      "SGH-800",
      "https://www.gsmarena.com/samsung_sgh_800-50.php"
    ],
    [
      "SGH-600",
      "https://www.gsmarena.com/samsung_sgh_600-49.php"
    ],
    [
      "SGH-500",
      "https://www.gsmarena.com/samsung_sgh_500-48.php"
    ],
    [
      "SGH-250",
      "https://www.gsmarena.com/samsung_sgh_250-47.php"
    ],
    [
      "Galaxy A04 Core",
      "https://www.gsmarena.com/samsung_galaxy_a04_core-11616.php"
    ],
    [
      "Galaxy S21",
      "https://www.gsmarena.com/samsung_galaxy_s21-10693.php"
    ],
    [
      "Galaxy A72 5G",
      "https://www.gsmarena.com/samsung_galaxy_a72_5g-10647.php"
    ],
    [
      "Galaxy A91",
      "https://www.gsmarena.com/samsung_galaxy_a91-9912.php"
    ],
    [
      "Galaxy On5 (2016)",
      "https://www.gsmarena.com/samsung_galaxy_on5_(2016)-8266.php"
    ],
    [
      "Galaxy J5 Prime (2017)",
      "https://www.gsmarena.com/samsung_galaxy_j5_prime_(2017)-8958.php"
    ],
    [
      "Galaxy S7 mini",
      "https://www.gsmarena.com/samsung_galaxy_s7_mini-7979.php"
    ],
    [
      "Galaxy S6 Plus",
      "https://www.gsmarena.com/samsung_galaxy_s6_plus-7270.php"
    ],
    [
      "Galaxy Grand 3",
      "https://www.gsmarena.com/samsung_galaxy_grand_3-6778.php"
    ],
    [
      "Galaxy F",
      "https://www.gsmarena.com/samsung_galaxy_f-6485.php"
    ],
    [
      "Galaxy Tab 3 Plus 10.1 P8220",
      "https://www.gsmarena.com/samsung_galaxy_tab_3_plus_10_1_p8220-5351.php"
    ],
    [
      "Galaxy C10",
      "https://www.gsmarena.com/samsung_galaxy_c10-8718.php"
    ],
    [
      "Guru Dual 26",
      "https://www.gsmarena.com/samsung_guru_dual_26-3528.php"
    ],
    [
      "C5030",
      "https://www.gsmarena.com/samsung_c5030-3369.php"
    ],
    [
      "M8920",
      "https://www.gsmarena.com/samsung_m8920-2967.php"
    ]
  ]
}
brand_devices_paths = {folder:[[str(file).split(' - ')[0].split('/')[2], str(file)] for file in (Path(html_pages_path)/folder).glob("*.html")] for folder in brands_to_use }

Ну вот мы и получили для каждого бренда ссылки на конкретные девайся на сайте

In [4]:
def get_device_specs(device_link, get_bs=get_bs):
    bs = get_bs(device_link)
    if not bs:
        return {}

    specs_collector = {}

    for table in bs.select("table"):
        header_tag = table.find("th")
        if not header_tag:
            continue

        table_header = header_tag.get_text(strip=True)
        ttl_collector = {}

        for row in table.select("tr"):
            ttl_tag = row.find("td", class_="ttl")
            nfo_tag = row.find("td", class_="nfo")

            if not ttl_tag or not nfo_tag:
                continue

            ttl_text = ttl_tag.get_text(strip=True)
            nfo_text = nfo_tag.get_text(strip=True)

            ttl_text = ttl_text if ttl_text else None
            nfo_text = nfo_text if nfo_text else None

            ttl_collector[ttl_text] = nfo_text

        if ttl_collector:
            specs_collector[table_header] = ttl_collector
    return specs_collector


In [5]:
def get_brand_data(brand, from_id = 0, num=20, brand_devices=brand_devices_refs, get_bs=get_bs):
    brand_devices_data = {}
    val = brand_devices[brand][from_id:from_id+num]
    for device in val:
        if (brand not in brand_devices_data.keys()):
            brand_devices_data[brand] = dict()
        if (device[0] not in brand_devices_data[brand].keys()):
            brand_devices_data[brand][device[0]] = dict()
        brand_devices_data[brand][device[0]] = get_device_specs(device[1], get_bs)
    return brand_devices_data


In [8]:
acer_devices_data = get_brand_data("acer")
print(json.dumps(acer_devices_data, indent = 2, ensure_ascii=False))
time.sleep(60)
acer_devices_data_2 = get_brand_data("acer", 20)
acer_devices_data["acer"].update(acer_devices_data_2["acer"])
print(json.dumps(acer_devices_data, indent = 2, ensure_ascii=False))
time.sleep(60)
acer_devices_data_3 = get_brand_data("acer", 40, 10)
acer_devices_data["acer"].update(acer_devices_data_3["acer"])
print(json.dumps(acer_devices_data, indent = 2, ensure_ascii=False))
time.sleep(60)


NameError: name 'get_brand_data' is not defined

In [6]:
apple_devices_data = get_brand_data("apple", 0, 50, brand_devices_paths, get_bs_file)
print(json.dumps(apple_devices_data, indent = 2, ensure_ascii=False))


{
  "apple": {
    "Apple Watch Series 8": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 1700(AWS) / 1900 / 2100 - International, China, HK",
        "null": "1, 2, 3, 4, 5, 6, 7, 12, 13, 14, 17, 18, 19, 25, 26, 39, 40, 41, 66 - USA, LATAM, Canada",
        "4G bands": "1, 2, 3, 4, 5, 6, 7, 8, 18, 19, 20, 25, 26, 28, 39, 40, 41, 66 - International, China, HK",
        "Speed": "HSPA, LTE"
      },
      "Launch": {
        "Announced": "2022, September 07",
        "Status": "Available. Released 2022, September 16"
      },
      "Body": {
        "Dimensions": "45 x 38 x 10.7 mm (1.77 x 1.50 x 0.42 in)",
        "Weight": "42.3 g (41mm), 51.5 g (45mm) (1.48 oz)",
        "Build": "Sapphire crystal front, ceramic/sapphire crystal back, stainless steel frame",
        "SIM": "eSIM",
        "null": "IP6X certified50m water resistantECG certified (region dependent SW application; HW

In [7]:
honor_devices_data = get_brand_data("honor", 0, 50, brand_devices_paths, get_bs_file)
print(json.dumps(honor_devices_data, indent = 2, ensure_ascii=False))


{
  "honor": {
    "Honor X8d": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 1900 / 2100",
        "4G bands": "LTE",
        "Speed": "HSPA, LTE"
      },
      "Launch": {
        "Announced": "2025, December",
        "Status": "Available. Released 2025, December"
      },
      "Body": {
        "Dimensions": "162.9 x 76.3 x 7.5 mm (6.41 x 3.00 x 0.30 in)",
        "Weight": "188 g (6.63 oz)",
        "Build": "Glass front, plastic frame, plastic back",
        "SIM": "Nano-SIM + Nano-SIM",
        "null": "IP65 dust tight and water resistant (low pressure water jets)"
      },
      "Display": {
        "Type": "AMOLED, 1B colors, 120Hz",
        "Size": "6.77 inches, 110.9 cm2(~89.2% screen-to-body ratio)",
        "Resolution": "1080 x 2392 pixels (~388 ppi density)"
      },
      "Platform": {
        "OS": "Android 15, MagicOS 10",
        "Chipset": "Qualcomm Snapdrago

In [8]:
samsung_devices_data = get_brand_data("samsung", 0, 50, brand_devices_paths, get_bs_file)
print(json.dumps(samsung_devices_data, indent = 2, ensure_ascii=False))


{
  "samsung": {
    "Samsung Galaxy Tab S10 Lite": {
      "Network": {
        "Technology": "GSM / HSPA / LTE / 5G",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 1700(AWS) / 1900 / 2100",
        "4G bands": "1, 2, 3, 4, 5, 7, 8, 12, 13, 17, 20, 25, 26, 28, 32, 38, 40, 41, 66",
        "5G bands": "1, 3, 5, 7, 8, 20, 26, 28, 38, 40, 41, 66, 71, 77, 78 SA/NSA/Sub6",
        "Speed": "HSPA, LTE, 5G - cellular model only"
      },
      "Launch": {
        "Announced": "2025, August 25",
        "Status": "Available. Released 2025, August 25"
      },
      "Body": {
        "Dimensions": "254.3 x 165.8 x 6.6 mm (10.01 x 6.53 x 0.26 in)",
        "Weight": "524 g (1.16 lb)",
        "Build": "Glass front, aluminum back, aluminum frame",
        "SIM": "Nano-SIM +eSIM(cellular model only)",
        "null": "Stylus support"
      },
      "Display": {
        "Type": "TFT LCD, 90Hz",
        "Size": "10.9 inches, 344.5 cm2(~81.7% screen-to-bod

In [9]:
acer_devices_data = {
  "acer": {
    "Iconia V12": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, May 16",
        "Status": "Available. Released 2025, July"
      },
      "Body": {
        "Dimensions": "7.9 mm thickness",
        "Weight": "595 g (1.31 lb)",
        "Build": "Glass front, aluminum alloy frame",
        "SIM": "No",
        "Extras": "Stylus supportBuilt-in three-level kickstand"
      },
      "Display": {
        "Type": "IPS LCD, 90Hz",
        "Size": "11.97 inches, 407.8 cm2",
        "Resolution": "1200 x 2000 pixels, 5:3 ratio (~195 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Mediatek Helio G99 (6 nm)",
        "CPU": "Octa-core (2x2.2 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "256GB 8GB RAM",
        "Extras": "eMMC"
      },
      "Main Camera": {
        "Single": "8 MP, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac/6, dual-band",
        "Bluetooth": "5.2, A2DP, LE",
        "Positioning": "Unspecified",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Unspecified"
      },
      "Battery": {
        "Type": "8000 mAh"
      },
      "Misc": {
        "Colors": "Green",
        "Models": "V12-11M",
        "Price": "About 290 EUR"
      }
    },
    "Iconia V11": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900 - cellular model only",
        "3G bands": "HSDPA 900 / 2100 - cellular model only",
        "4G bands": "LTE - cellular model only",
        "Speed": "HSPA, LTE - cellular model only"
      },
      "Launch": {
        "Announced": "2025, May 16",
        "Status": "Available. Released 2025, August"
      },
      "Body": {
        "Dimensions": "7.9 mm thickness",
        "Weight": "500 g (1.10 lb)",
        "Build": "Glass front, aluminum alloy frame",
        "SIM": "Nano-SIM",
        "Extras": "Stylus supportBuilt-in three-level kickstand"
      },
      "Display": {
        "Type": "IPS LCD, 90Hz",
        "Size": "10.92 inches, 345.8 cm2",
        "Resolution": "1200 x 1920 pixels, 16:10 ratio (~207 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Mediatek Helio G99 (6 nm)",
        "CPU": "Octa-core (2x2.2 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "256GB 8GB RAM",
        "Extras": "eMMC"
      },
      "Main Camera": {
        "Single": "8 MP, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac/6, dual-band",
        "Bluetooth": "5.2, A2DP, LE",
        "Positioning": "Unspecified",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Unspecified"
      },
      "Battery": {
        "Type": "8000 mAh"
      },
      "Misc": {
        "Colors": "Gold",
        "Models": "V11-21M, V11-22M",
        "Price": "About 230 EUR"
      }
    },
    "Acerone Liquid S262F5": {
      "Network": {
        "Technology": "GSM / HSPA / LTE / 5G",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 2100",
        "4G bands": "1, 3, 5, 8, 40, 41",
        "5G bands": "1, 3, 5, 7, 8, 26, 28, 40, 41, 66, 77, 78 SA/NSA",
        "Speed": "HSPA, LTE, 5G"
      },
      "Launch": {
        "Announced": "2025, October 22",
        "Status": "Available. Released Exp. release 2025, October"
      },
      "Body": {
        "Dimensions": "171 x 77.8 x 8.9 mm (6.73 x 3.06 x 0.35 in)",
        "Weight": "202.8 g (7.16 oz)",
        "Build": "Glass front (Gorilla Glass 5), plastic back, plastic frame",
        "SIM": "Nano-SIM + Nano-SIM",
        "Extras": "IP54 dust protected and water resistant (water splashes)"
      },
      "Display": {
        "Type": "IPS LCD, 90Hz, 380 nits",
        "Size": "6.76 inches, 108.5 cm2(~81.6% screen-to-body ratio)",
        "Resolution": "720 x 1640 pixels (~265 ppi density)",
        "Protection": "Corning Gorilla Glass 5"
      },
      "Platform": {
        "OS": "Android 16",
        "Chipset": "Mediatek Dimensity 6400 (6 nm)",
        "CPU": "Octa-core (2x2.5 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
        "Card slot": "microSDXC (uses shared SIM slot)",
        "Internal": "128GB 8GB RAM",
        "Extras": "UFS 2.2"
      },
      "Main Camera": {
        "Triple": "64 MP, (wide), PDAF8 MP, (ultrawide)2 MP (macro)",
        "Features": "LED flash",
        "Video": "1080p"
      },
      "Selfie camera": {
        "Single": "16 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "No"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac/6e, dual-band",
        "Bluetooth": "5.2, A2DP, LE",
        "Positioning": "GPS, GLONASS, GALILEO, BDS",
        "NFC": "No",
        "Radio": "FM radio",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Fingerprint (side-mounted), accelerometer, gyro, proximity, compass"
      },
      "Battery": {
        "Type": "Li-Ion 5000 mAh"
      },
      "Misc": {
        "Colors": "Dark Blue",
        "Models": "ZL.A01SU.04P"
      }
    },
    "Iconia A14": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, September 03",
        "Status": "Available. Released 2025, November"
      },
      "Body": {
        "Dimensions": "324.6 x 211 x 8 mm (12.78 x 8.31 x 0.31 in)",
        "Weight": "910 g (2.01 lb)",
        "Build": "Glass front, aluminum frame",
        "SIM": "No",
        "Extras": "Built-in kickstand"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "14.0 inches, 568.3 cm2(~83.0% screen-to-body ratio)",
        "Resolution": "1200 x 1920 pixels, 16:10 ratio (~162 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Allwinner A733 (12 nm)",
        "CPU": "Octa-core (2x2.0 GHz Cortex-A76 & 6x1.8 GHz Cortex-A55)",
        "GPU": "IMG BXM-4-64 MC01"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "256GB 8GB RAM"
      },
      "Main Camera": {
        "Single": "8 MP, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers (4 speakers)",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac/6, dual-band",
        "Bluetooth": "5.4, A2DP, LE",
        "Positioning": "No",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer"
      },
      "Battery": {
        "Type": "8000 mAh"
      },
      "Misc": {
        "Colors": "Vapor Silver",
        "Models": "A14-M1N",
        "Price": "About 260 EUR"
      }
    },
    "Iconia A16": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, September 03",
        "Status": "Available. Released 2026, January"
      },
      "Body": {
        "Dimensions": "369.5 x 240.2 x 8.9 mm (14.55 x 9.46 x 0.35 in)",
        "Weight": "980 g (2.16 lb)",
        "Build": "Glass front, aluminum frame",
        "SIM": "No",
        "Extras": "Built-in kickstand"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "16.0 inches, 742.3 cm2(~83.6% screen-to-body ratio)",
        "Resolution": "1200 x 1920 pixels, 16:10 ratio (~142 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Allwinner A733 (12 nm)",
        "CPU": "Octa-core (2x2.0 GHz Cortex-A76 & 6x1.8 GHz Cortex-A55)",
        "GPU": "IMG BXM-4-64 MC01"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "256GB 8GB RAM"
      },
      "Main Camera": {
        "Single": "8 MP, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers (4 speakers)",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac/6, dual-band",
        "Bluetooth": "5.4, A2DP, LE",
        "Positioning": "No",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer"
      },
      "Battery": {
        "Type": "8000 mAh"
      },
      "Misc": {
        "Colors": "Vapor Silver",
        "Models": "A16-M1N",
        "Price": "About 300 EUR"
      }
    },
    "Iconia X14": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, September 03",
        "Status": "Available. Released 2025, November"
      },
      "Body": {
        "Dimensions": "324.6 x 211 x 8 mm (12.78 x 8.31 x 0.31 in)",
        "Weight": "910 g (2.01 lb)",
        "SIM": "No"
      },
      "Display": {
        "Type": "OLED",
        "Size": "14.0 inches, 568.3 cm2(~83.0% screen-to-body ratio)",
        "Resolution": "1200 x 1920 pixels, 16:10 ratio (~162 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Allwinner A733 (12 nm)",
        "CPU": "Octa-core (2x2.0 GHz Cortex-A76 & 6x1.8 GHz Cortex-A55)",
        "GPU": "IMG BXM-4-64 MC01"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "256GB 8GB RAM",
        "Extras": "UFS 3.1"
      },
      "Main Camera": {
        "Single": "8 MP, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers (4 speakers)",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac/6, dual-band",
        "Bluetooth": "5.4, A2DP, LE",
        "Positioning": "No",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer"
      },
      "Battery": {
        "Type": "Li-Ion 8000 mAh"
      },
      "Misc": {
        "Colors": "Iron Gray",
        "Models": "X14-M1N",
        "Price": "About 310 EUR"
      }
    },
    "Iconia X12": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, September 03",
        "Status": "Available. Released 2025, November"
      },
      "Body": {
        "Dimensions": "286.2 x 184.4 x 6.5 mm (11.27 x 7.26 x 0.26 in)",
        "Weight": "500 g (1.10 lb)",
        "Build": "Glass front, aluminum alloy frame",
        "SIM": "No",
        "Extras": "Stylus support"
      },
      "Display": {
        "Type": "AMOLED, 400 nits",
        "Size": "12.6 inches, 460.3 cm2(~87.2% screen-to-body ratio)",
        "Resolution": "1600 x 2560 pixels, 16:10 ratio (~240 ppi density)",
        "Protection": "Mohs level 4"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Mediatek Helio G99 (6 nm)",
        "CPU": "Octa-core (2x2.2 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "256GB 8GB RAM",
        "Extras": "UFS"
      },
      "Main Camera": {
        "Single": "13 MP, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers (4 speakers)",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac, dual-band",
        "Bluetooth": "5.2, A2DP, LE",
        "Positioning": "GPS",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0, magnetic accessory connector"
      },
      "Features": {
        "Sensors": "Accelerometer, proximity (accessories only)"
      },
      "Battery": {
        "Type": "Li-Ion 8000 mAh",
        "Charging": "20W wired"
      },
      "Misc": {
        "Colors": "Matte Black",
        "Models": "A25014, X12-21M, X12-11-845L",
        "Price": "About 280 EUR"
      },
      "EU LABEL": {
        "Energy": "Class F",
        "Battery": "80:31h endurance, 800 cycles",
        "Free fall": "Class E (2 falls)",
        "Repairability": "Class C"
      }
    },
    "Iconia Tab P11 (2025)": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, Q4",
        "Status": "Available. Released 2025, Q4"
      },
      "Body": {
        "Dimensions": "265.2 x 169.9 x 8.1 mm (10.44 x 6.69 x 0.32 in)",
        "Weight": "550 g (1.21 lb)",
        "Build": "Glass front, aluminum alloy frame",
        "SIM": "No",
        "Extras": "Stylus support"
      },
      "Display": {
        "Type": "IPS LCD, 90Hz",
        "Size": "11.0 inches, 344.4 cm2(~76.4% screen-to-body ratio)",
        "Resolution": "1200 x 2000 pixels, 5:3 ratio (~212 ppi density)"
      },
      "Platform": {
        "OS": "Android 14",
        "Chipset": "Mediatek Helio G99 (6 nm)",
        "CPU": "Octa-core (2x2.2 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "256GB 6GB RAM, 256GB 8GB RAM",
        "Extras": "UFS"
      },
      "Main Camera": {
        "Single": "13 MP, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers (4 speakers)",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac, dual-band",
        "Bluetooth": "Yes",
        "Positioning": "Unspecified",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Unspecified"
      },
      "Battery": {
        "Type": "8000 mAh",
        "Charging": "20W wired"
      },
      "Misc": {
        "Colors": "Iron Gray",
        "Models": "P11-11-8113",
        "Price": "About 260 EUR"
      }
    },
    "Iconia Tab P10 (2025)": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, Q4",
        "Status": "Available. Released 2025, Q4"
      },
      "Body": {
        "Dimensions": "8.1 mm thickness",
        "Weight": "495 g (1.09 lb)",
        "Build": "Glass front, aluminum alloy frame",
        "SIM": "No"
      },
      "Display": {
        "Type": "IPS LCD, 90Hz",
        "Size": "10.4 inches, 307.9 cm2",
        "Resolution": "1200 x 2000 pixels, 5:3 ratio (~224 ppi density)"
      },
      "Platform": {
        "OS": "Android 14",
        "Chipset": "Mediatek Helio G99 (6 nm)",
        "CPU": "Octa-core (2x2.2 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "128GB 6GB RAM, 256GB 6GB RAM",
        "Extras": "UFS"
      },
      "Main Camera": {
        "Single": "13 MP",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac, dual-band",
        "Bluetooth": "5.0, A2DP, LE",
        "Positioning": "GPS",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Gyro"
      },
      "Battery": {
        "Type": "Li-Ion 8000 mAh",
        "Charging": "10W wired"
      },
      "Misc": {
        "Colors": "Iron Gray",
        "Models": "P10-21Q-870Q, NT.LHBAA.001",
        "Price": "About 220 EUR"
      }
    },
    "Super ZX Pro": {
      "Network": {
        "Technology": "GSM / HSPA / LTE / 5G",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 2100",
        "4G bands": "LTE",
        "5G bands": "SA/NSA",
        "Speed": "HSPA, LTE, 5G"
      },
      "Launch": {
        "Announced": "2025, May 26",
        "Status": "Cancelled"
      },
      "Body": {
        "Dimensions": "-",
        "Weight": "182 g (6.42 oz)",
        "Build": "Glass front, glass back, plastic frame",
        "SIM": "Nano-SIM + Nano-SIM",
        "Extras": "IP64 dust tight and water resistant (water splashes)"
      },
      "Display": {
        "Type": "AMOLED, 120Hz, 1000 nits (HBM)",
        "Size": "6.67 inches, 107.4 cm2",
        "Resolution": "1080 x 2400 pixels, 20:9 ratio (~395 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Mediatek Dimensity 7400 (4 nm)",
        "CPU": "Octa-core (4x2.6 GHz Cortex-A78 & 4x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G615 MC2"
      },
      "Memory": {
        "Card slot": "Unspecified",
        "Internal": "128GB 6GB RAM, 128GB 8GB RAM, 256GB 8GB RAM, 512GB 8GB RAM, 512GB 12GB RAM"
      },
      "Main Camera": {
        "Triple": "64 MP, f/1.8, (wide), 1/1.73\", 0.8µm, PDAF, OIS5 MP, (ultrawide)Auxiliary lens",
        "Features": "LED flash, HDR, panorama",
        "Video": "1440p@30fps, 1080p@30fps"
      },
      "Selfie camera": {
        "Single": "13 MP",
        "Features": "HDR",
        "Video": "1440p@30fps, 1080p@30fps"
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers (with Dolby Atmos)",
        "3.5mm jack": "Unspecified"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac/6, dual-band",
        "Bluetooth": "5.4, A2DP, LE",
        "Positioning": "GPS, GLONASS, GALILEO, BDS",
        "NFC": "Unspecified",
        "Radio": "Unspecified",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Fingerprint (under display, optical), accelerometer, proximity, compass"
      },
      "Battery": {
        "Type": "5000 mAh",
        "Charging": "33W wired"
      },
      "Misc": {
        "Colors": "Black; other colors",
        "Price": "About 190 EUR"
      }
    },
    "Super ZX": {
      "Network": {
        "Technology": "GSM / HSPA / LTE / 5G",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 2100",
        "4G bands": "LTE",
        "5G bands": "SA/NSA",
        "Speed": "HSPA, LTE, 5G"
      },
      "Launch": {
        "Announced": "2025, May 26",
        "Status": "Available. Released 2025, May 26"
      },
      "Body": {
        "Dimensions": "8.6 mm thickness",
        "Weight": "200 g (7.05 oz)",
        "SIM": "Nano-SIM + Nano-SIM",
        "Extras": "IP50 dust protected"
      },
      "Display": {
        "Type": "IPS LCD, 120Hz",
        "Size": "6.78 inches, 111.0 cm2(~85.3% screen-to-body ratio)",
        "Resolution": "1080 x 2400 pixels, 20:9 ratio (~388 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Mediatek Dimensity 6300 (6 nm)",
        "CPU": "Octa-core (2x2.4 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
        "Card slot": "Unspecified",
        "Internal": "128GB 4GB RAM, 128GB 6GB RAM, 128GB 8GB RAM"
      },
      "Main Camera": {
        "Triple": "64 MP, f/1.8, (wide), 1/1.73\", 0.8µm, PDAF2 MP (macro)Auxiliary lens",
        "Features": "LED flash, HDR, panorama",
        "Video": "Yes"
      },
      "Selfie camera": {
        "Single": "Yes",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Yes",
        "Bluetooth": "Yes",
        "Positioning": "GPS, GLONASS, GALILEO, BDS",
        "NFC": "Unspecified",
        "Radio": "Unspecified",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Fingerprint (side-mounted), accelerometer, proximity, compass"
      },
      "Battery": {
        "Type": "5000 mAh",
        "Charging": "33W wired, 50% in 35 min"
      },
      "Misc": {
        "Colors": "Carbon Black, Cosmic Green, Lunar Blue",
        "Price": "₹ 10,399"
      }
    },
    "Acerone Liquid S272E4": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 2100",
        "4G bands": "1, 3, 5, 8, 40, 41",
        "Speed": "HSPA, LTE"
      },
      "Launch": {
        "Announced": "2025, March 25",
        "Status": "Available. Released 2025, May"
      },
      "Body": {
        "Dimensions": "171 x 78.6 x 8.9 mm (6.73 x 3.09 x 0.35 in)",
        "Weight": "200 g (7.05 oz)",
        "Build": "Glass front (Gorilla Glass 4), plastic back, plastic frame",
        "SIM": "Nano-SIM + Nano-SIM"
      },
      "Display": {
        "Type": "IPS LCD, 350 nits (typ)",
        "Size": "6.75 inches, 110.0 cm2(~81.8% screen-to-body ratio)",
        "Resolution": "720 x 1600 pixels, 20:9 ratio (~260 ppi density)",
        "Protection": "Corning Gorilla Glass 4"
      },
      "Platform": {
        "OS": "Android 14",
        "Chipset": "Mediatek MT6765X (12 nm)",
        "CPU": "Octa-core (4x2.2 GHz Cortex-A53 & 4x1.8 GHz Cortex-A53)",
        "GPU": "PowerVR GE8320"
      },
      "Memory": {
        "Card slot": "microSDXC (uses shared SIM slot)",
        "Internal": "64GB 4GB RAM"
      },
      "Main Camera": {
        "Dual": "20 MP, (wide), AF0.3 MP",
        "Features": "LED flash",
        "Video": "1080p"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac, dual-band",
        "Bluetooth": "5.0, A2DP",
        "Positioning": "GPS, GLONASS, GALILEO, BDS",
        "NFC": "No",
        "Infrared port": "Yes",
        "Radio": "FM radio",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Fingerprint (side-mounted), accelerometer, proximity"
      },
      "Battery": {
        "Type": "Li-Ion 5000 mAh"
      },
      "Misc": {
        "Colors": "Dark Blue",
        "Models": "ZL.H01SI.002"
      }
    },
    "Acerone Liquid S162E4": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 850 / 900 / 2100",
        "4G bands": "1, 3, 5, 8, 40, 41",
        "Speed": "HSPA, LTE"
      },
      "Launch": {
        "Announced": "2025, March 25",
        "Status": "Available. Released 2025, May"
      },
      "Body": {
        "Dimensions": "165.4 x 76.9 x 9 mm (6.51 x 3.03 x 0.35 in)",
        "Weight": "210 g (7.41 oz)",
        "Build": "Glass front (Gorilla Glass 5), plastic back, plastic frame",
        "SIM": "Nano-SIM + Nano-SIM"
      },
      "Display": {
        "Type": "IPS LCD, 350 nits (typ)",
        "Size": "6.52 inches, 102.6 cm2(~80.7% screen-to-body ratio)",
        "Resolution": "720 x 1600 pixels, 20:9 ratio (~269 ppi density)",
        "Protection": "Corning Gorilla Glass 5"
      },
      "Platform": {
        "OS": "Android 14",
        "Chipset": "Mediatek MT6765 (12 nm)",
        "CPU": "Octa-core (4x2.3 GHz Cortex-A53 & 4x1.8 GHz Cortex-A53)",
        "GPU": "PowerVR GE8320"
      },
      "Memory": {
        "Card slot": "microSDXC (uses shared SIM slot)",
        "Internal": "64GB 4GB RAM"
      },
      "Main Camera": {
        "Single": "16 MP, (wide), AF",
        "Features": "LED flash",
        "Video": "1080p"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": "Yes"
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac, dual-band",
        "Bluetooth": "5.0, A2DP",
        "Positioning": "GPS, GLONASS, GALILEO, BDS",
        "NFC": "No",
        "Infrared port": "Yes",
        "Radio": "FM radio",
        "USB": "USB Type-C 2.0"
      },
      "Features": {
        "Sensors": "Fingerprint (side-mounted), accelerometer, proximity"
      },
      "Battery": {
        "Type": "Li-Ion 5000 mAh"
      },
      "Misc": {
        "Colors": "Blue",
        "Models": "ZL.H01SI.001"
      }
    },
    "Chromebook Tab 10": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2018, March. Released 2018, July",
        "Status": "Discontinued"
      },
      "Body": {
        "Dimensions": "238.3 x 172.2 x 9.9 mm (9.38 x 6.78 x 0.39 in)",
        "Weight": "544.3 g (1.20 lb)",
        "SIM": "No",
        "Extras": "Stylus"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "9.7 inches, 291.4 cm2(~71.0% screen-to-body ratio)",
        "Resolution": "1536 x 2048 pixels, 4:3 ratio (~264 ppi density)"
      },
      "Platform": {
        "OS": "Chrome OS",
        "Chipset": "Rockchip RK3399",
        "CPU": "Hexa-core (4x Cortex-A53 & 2x Cortex-A72)"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "32GB 4GB RAM"
      },
      "Main Camera": {
        "Single": "5 MP",
        "Video": "720p"
      },
      "Selfie camera": {
        "Single": "2 MP",
        "Video": ""
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 b/g/n/ac, dual-band, Wi-Fi Direct",
        "Bluetooth": "4.1, A2DP, LE",
        "Positioning": "",
        "NFC": "No",
        "Radio": "No",
        "USB": "USB Type-C 3.1"
      },
      "Features": {
        "Sensors": "Accelerometer"
      },
      "Battery": {
        "Type": "Li-Po 4500 mAh, non-removable (34 Wh)",
        "Talk time": "Up to 9 h (multimedia)"
      },
      "Misc": {
        "Colors": "Black, Blue",
        "Price": "About 330 EUR"
      }
    },
    "Iconia Talk S": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900 - SIM 1 & SIM 2",
        "3G bands": "HSDPA 850 / 1900 / 2100",
        "4G bands": "1, 3, 7, 8, 20",
        "Extras": "3, 7, 8, 28",
        "Speed": "HSPA 42.2/11.5 Mbps, LTE Cat4 150/50 Mbps"
      },
      "Launch": {
        "Announced": "2016, August. Released 2016, October",
        "Status": "Discontinued"
      },
      "Body": {
        "Dimensions": "191.7 x 101 x 9.4 mm (7.55 x 3.98 x 0.37 in)",
        "Weight": "260 g (9.17 oz)",
        "SIM": "Nano-SIM + Micro-SIM"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "7.0 inches, 135.1 cm2(~69.8% screen-to-body ratio)",
        "Resolution": "720 x 1280 pixels, 16:9 ratio (~210 ppi density)"
      },
      "Platform": {
        "OS": "Android 6.0 (Marshmallow)",
        "Chipset": "Mediatek MT8735",
        "CPU": "Quad-core 1.3 GHz Cortex-A53",
        "GPU": "Mali-T720MP2"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "16GB 2GB RAM, 32GB 2GB RAM"
      },
      "Main Camera": {
        "Single": "13 MP, AF",
        "Features": "HDR, panorama",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "2 MP",
        "Video": "720p"
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n, Wi-Fi Direct",
        "Bluetooth": "4.0, A2DP",
        "Positioning": "GPS, GLONASS",
        "NFC": "No",
        "Radio": "FM radio",
        "USB": "microUSB 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer, proximity"
      },
      "Battery": {
        "Type": "Li-Ion 3400 mAh, non-removable (12.92 Wh)",
        "Talk time": "Up to 9 h (multimedia)"
      },
      "Misc": {
        "Colors": "Black",
        "Price": "About 170 EUR"
      }
    },
    "Liquid Z6 Plus": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900 - SIM 1 & SIM 2 (dual-SIM model only)",
        "3G bands": "HSDPA",
        "4G bands": "LTE",
        "Speed": "HSPA 42.2/5.76 Mbps, LTE Cat4 150/50 Mbps",
        "GPRS": "Yes",
        "EDGE": "Yes"
      },
      "Launch": {
        "Announced": "2016, August. Released 2016, December",
        "Status": "Discontinued"
      },
      "Body": {
        "Dimensions": "153.8 x 75.6 x 8.5 mm (6.06 x 2.98 x 0.33 in)",
        "Weight": "169 g (5.96 oz)",
        "SIM": "Single SIM (Micro-SIM) or Hybrid Dual SIM (Micro-SIM, dual stand-by)"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "5.5 inches, 83.4 cm2(~71.7% screen-to-body ratio)",
        "Resolution": "1080 x 1920 pixels, 16:9 ratio (~401 ppi density)"
      },
      "Platform": {
        "OS": "Android 6.0 (Marshmallow)",
        "Chipset": "Mediatek MT6753 (28 nm)",
        "CPU": "Octa-core 1.3 GHz Cortex-A53",
        "GPU": "Mali-T720MP3"
      },
      "Memory": {
        "Card slot": "microSDXC (uses shared SIM slot)",
        "Internal": "32GB 3GB RAM"
      },
      "Main Camera": {
        "Single": "13 MP, AF",
        "Features": "LED flash, HDR, panorama",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "5 MP",
        "Video": ""
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 b/g/n, Wi-Fi Direct",
        "Bluetooth": "4.0, A2DP",
        "Positioning": "GPS",
        "NFC": "No",
        "Radio": "FM radio",
        "USB": "microUSB 2.0"
      },
      "Features": {
        "Sensors": "Fingerprint (front-mounted), accelerometer, proximity"
      },
      "Battery": {
        "Type": "Li-Po 4080 mAh, non-removable"
      },
      "Misc": {
        "Colors": "Black, White",
        "Price": "About 250 EUR"
      }
    },
    "Liquid Z6": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900 - SIM 1 & SIM 2 (dual-SIM model only)",
        "3G bands": "HSDPA",
        "4G bands": "LTE",
        "Speed": "HSPA, LTE",
        "GPRS": "Yes",
        "EDGE": "Yes"
      },
      "Launch": {
        "Announced": "2016, August. Released 2016, December",
        "Status": "Discontinued"
      },
      "Body": {
        "Dimensions": "145.5 x 72.5 x 8.5 mm (5.73 x 2.85 x 0.33 in)",
        "Weight": "126 g (4.44 oz)",
        "SIM": "Single SIM (Micro-SIM) or Dual SIM (Micro-SIM, dual stand-by)"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "5.0 inches, 68.9 cm2(~65.3% screen-to-body ratio)",
        "Resolution": "720 x 1280 pixels, 16:9 ratio (~294 ppi density)"
      },
      "Platform": {
        "OS": "Android 6.0 (Marshmallow)",
        "Chipset": "Mediatek MT6737 (28 nm)",
        "CPU": "Quad-core 1.25 GHz Cortex-A53",
        "GPU": "Mali-T720MP2"
      },
      "Memory": {
        "Card slot": "microSDXC",
        "Internal": "8GB 1GB RAM",
        "Extras": "eMMC 5.0"
      },
      "Main Camera": {
        "Single": "8 MP, AF",
        "Features": "LED flash, HDR, panorama",
        "Video": "Yes"
      },
      "Selfie camera": {
        "Single": "2 MP",
        "Video": ""
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 b/g/n",
        "Bluetooth": "4.0, A2DP",
        "Positioning": "GPS",
        "NFC": "No",
        "Radio": "FM radio",
        "USB": "microUSB 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer, proximity"
      },
      "Battery": {
        "Type": "Li-Ion 2000 mAh, removable"
      },
      "Misc": {
        "Colors": "Black, White",
        "Price": "About 120 EUR"
      }
    },
    "Iconia Tab 10 A3-A40": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2016, April. Released 2016, June",
        "Status": "Discontinued"
      },
      "Body": {
        "Dimensions": "259 x 167 x 8.9 mm (10.20 x 6.57 x 0.35 in)",
        "Weight": "-",
        "SIM": "No"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "10.1 inches, 295.8 cm2(~68.4% screen-to-body ratio)",
        "Resolution": "1920 x 1200 pixels, 16:10 ratio (~224 ppi density)"
      },
      "Platform": {
        "OS": "Android 6.0 (Marshmallow)",
        "Chipset": "Mediatek MT8163A",
        "CPU": "Quad-core 1.3 GHz Cortex-A53",
        "GPU": "Mali-T720 MP2"
      },
      "Memory": {
        "Card slot": "microSDXC (dedicated slot)",
        "Internal": "16GB 2GB RAM, 32GB 2GB RAM, 64GB 2GB RAM"
      },
      "Main Camera": {
        "Single": "5 MP",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "2 MP",
        "Video": ""
      },
      "Sound": {
        "Loudspeaker": "Yes, with stereo speakers (4 speakers)",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac, dual-band",
        "Bluetooth": "4.0, A2DP",
        "Positioning": "",
        "NFC": "No",
        "Radio": "No",
        "USB": "microUSB 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer",
        "Extras": "HDMI"
      },
      "Battery": {
        "Type": "Li-Ion 6100 mAh, non-removable"
      },
      "Misc": {
        "Colors": "Black",
        "Price": "About 230 EUR"
      }
    },
    "Liquid X2": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900",
        "3G bands": "HSDPA 900 / 1900 / 2100 - Europe, Taiwan",
        "Extras": "LTE 700 / 900 / 1800 - Taiwan",
        "4G bands": "LTE 800 / 1800 / 2100 / 2600 - Europe",
        "Speed": "HSPA 42.2/5.76 Mbps, LTE Cat4 150/50 Mbps"
      },
      "Launch": {
        "Announced": "2015, April. Released 2016, February",
        "Status": "Discontinued"
      },
      "Body": {
        "Dimensions": "153.3 x 78.8 x 8.5 mm (6.04 x 3.10 x 0.33 in)",
        "Weight": "166 g (5.86 oz)",
        "SIM": "Triple SIM (Micro-SIM)"
      },
      "Display": {
        "Type": "IPS LCD",
        "Size": "5.5 inches, 83.4 cm2(~69.0% screen-to-body ratio)",
        "Resolution": "720 x 1280 pixels, 16:9 ratio (~267 ppi density)"
      },
      "Platform": {
        "OS": "Android 5.1 (Lollipop)",
        "Chipset": "Mediatek MT6753 (28 nm)",
        "CPU": "Octa-core 1.3 GHz Cortex-A53",
        "GPU": "Mali-T720MP4"
      },
      "Memory": {
        "Card slot": "microSDHC (dedicated slot)",
        "Internal": "32GB 3GB RAM"
      },
      "Main Camera": {
        "Single": "13 MP, f/1.8, AF",
        "Features": "LED flash",
        "Video": "1080p@30fps"
      },
      "Selfie camera": {
        "Single": "13 MP, f/1.8, AF",
        "Features": "LED flash",
        "Video": ""
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes",
        "Extras": "24-bit/192kHz audio"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 b/g/n, hotspot",
        "Bluetooth": "4.0, A2DP, LE",
        "Positioning": "GPS",
        "NFC": "No",
        "Radio": "FM radio",
        "USB": "microUSB 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer, proximity, compass"
      },
      "Battery": {
        "Type": "Li-Po 4020 mAh, removable",
        "Stand-by": "Up to 820 h (3G)",
        "Talk time": "Up to 23 h (3G)"
      },
      "Misc": {
        "Colors": "Black, Gold",
        "Price": "About 230 EUR"
      },
      "Our Tests": {
        "Performance": "Basemark OS II 2.0: 825Basemark X: 4300",
        "Display": "Contrast ratio: 950 (nominal), 2.084 (sunlight)",
        "Camera": "Photo/Video",
        "Loudspeaker": "Voice 61dB / Noise 62dB / Ring 67dB",
        "Audio quality": "Noise -86.1dB / Crosstalk -88.1dB",
        "Battery (old)": "Endurance rating 85h"
      }
    },
    "Liquid Jade 2": {
      "Network": {
        "Technology": "GSM / HSPA / LTE",
        "2G bands": "GSM 850 / 900 / 1800 / 1900 - SIM 1 & SIM 2",
        "3G bands": "HSDPA 900 / 2100",
        "4G bands": "LTE",
        "Speed": "HSPA 42.2/5.76 Mbps, LTE Cat4 150/50 Mbps"
      },
      "Launch": {
        "Announced": "2016, February",
        "Status": "Discontinued"
      },
      "Body": {
        "Dimensions": "156.6 x 75.9 x 8.4 mm (6.17 x 2.99 x 0.33 in)",
        "Weight": "150 g (5.29 oz)",
        "SIM": "Nano-SIM + Nano-SIM"
      },
      "Display": {
        "Type": "Super AMOLED",
        "Size": "5.5 inches, 83.4 cm2(~70.2% screen-to-body ratio)",
        "Resolution": "1080 x 1920 pixels, 16:9 ratio (~401 ppi density)",
        "Protection": "Corning Gorilla Glass 4"
      },
      "Platform": {
        "OS": "Android 6.0 (Marshmallow)",
        "Chipset": "Qualcomm MSM8992 Snapdragon 808 (20 nm)",
        "CPU": "Hexa-core (4x1.4 GHz Cortex-A53 & 2x1.8 GHz Cortex-A57)",
        "GPU": "Adreno 418"
      },
      "Memory": {
        "Card slot": "microSDXC",
        "Internal": "32GB 3GB RAM",
        "Extras": "eMMC 5.0"
      },
      "Main Camera": {
        "Single": "21 MP, AF",
        "Features": "Dual-LED flash, HDR, panorama",
        "Video": "4K@30fps, 1080p@30fps"
      },
      "Selfie camera": {
        "Single": "8 MP",
        "Video": "1080p@30fps"
      },
      "Sound": {
        "Loudspeaker": "Yes",
        "3.5mm jack": "Yes"
      },
      "Comms": {
        "WLAN": "Wi-Fi 802.11 a/b/g/n/ac, dual-band, Wi-Fi Direct",
        "Bluetooth": "4.0, A2DP, LE",
        "Positioning": "GPS",
        "NFC": "No",
        "Radio": "FM radio",
        "USB": "microUSB 2.0"
      },
      "Features": {
        "Sensors": "Accelerometer, gyro, proximity, compass"
      },
      "Battery": {
        "Type": "Li-Po 3000 mAh, non-removable"
      },
      "Misc": {
        "Colors": "Black"
      }
    }
  }
}
acer_devices_data["acer"] = (
  acer_devices_data["acer"]
  | get_brand_data("acer", 0, 30, brand_devices_paths, get_bs_file)["acer"]
)
# парсить GMS кравлером -- похороны; 7 лет назад норм, сейчас GLHF.
# NB: Ботиночки детские, неношенные
devices_data = (
    acer_devices_data
    | samsung_devices_data
    | apple_devices_data
    | honor_devices_data
)
print(json.dumps(devices_data, indent = 2, ensure_ascii=False))

{
  "acer": {
    "Iconia V12": {
      "Network": {
        "Technology": "No cellular connectivity",
        "2G bands": "N/A",
        "3G bands": "N/A",
        "4G bands": "N/A",
        "GPRS": "No",
        "EDGE": "No"
      },
      "Launch": {
        "Announced": "2025, May 16",
        "Status": "Available. Released 2025, July"
      },
      "Body": {
        "Dimensions": "7.9 mm thickness",
        "Weight": "595 g (1.31 lb)",
        "Build": "Glass front, aluminum alloy frame",
        "SIM": "No",
        "Extras": "Stylus supportBuilt-in three-level kickstand"
      },
      "Display": {
        "Type": "IPS LCD, 90Hz",
        "Size": "11.97 inches, 407.8 cm2",
        "Resolution": "1200 x 2000 pixels, 5:3 ratio (~195 ppi density)"
      },
      "Platform": {
        "OS": "Android 15",
        "Chipset": "Mediatek Helio G99 (6 nm)",
        "CPU": "Octa-core (2x2.2 GHz Cortex-A76 & 6x2.0 GHz Cortex-A55)",
        "GPU": "Mali-G57 MC2"
      },
      "Memory": {
 

In [10]:
import csv
import re
from datetime import datetime, date

RATES_TO_EUR = {
    "$": 0.92,
    "£": 1.17,
    "₹": 0.011,
}

def extract_number(pattern, text):
    if not text:
        return ""
    m = re.search(pattern, text)
    return m.group(1) if m else ""

def parse_weight_g(text):
    return extract_number(r"([\d.]+)\s*g", text)

def parse_chipset(chipset: str):
    if not chipset:
        return None
    
    name = re.sub(r"\(\d+\s*nm\)", "", chipset).strip()
    
    manufacturer = name.split()[0]
    
    return manufacturer

def parse_battery_mah(text):
    return extract_number(r"(\d+)\s*mAh", text)

def parse_price(text):
    eur = extract_number(r"About (\d+) EUR", text)
    if eur != "":
        return eur
    
    eur = extract_number(r"€\s?([\d.,]+)", text).replace(",", "")
    if eur != "":
        return eur

    for symbol, rate in RATES_TO_EUR.items():
        match = extract_number(rf"\{symbol}\s?([\d.,]+)", text).replace(",", "")
        if match != "":
            return f"{float(match) * rate:.2f}"
    return ""

def parse_os(os_text):
    if not os_text:
        return ""
    cleaned = re.split(r"\d|\(", os_text)[0]

    return cleaned.strip()

def parse_sim(sim):
    sim = re.search(r"[^A-Za-z1-9]*([A-Za-z1-9]+)", sim).group(1)
    if not sim:
        return "Undetected"
    return sim

def parse_ram(storage_text):
    return extract_number(r"(\d+)\s*GB\s*RAM", storage_text)

def parse_storage(storage_text):
    return extract_number(r"(\d+)\s*GB", storage_text)

def parse_fps(video_text):
    return extract_number(r"@(\d+)fps", video_text)

def parse_announced(text):
    if not text:
        return None

    text = text.strip()

    text = text.split(".")[0].strip()

    q_match = re.match(r"(\d{4}),\s*Q([1-4])", text)
    if q_match:
        year = int(q_match.group(1))
        quarter = int(q_match.group(2))
        month = (quarter - 1) * 3 + 1
        return str(date(year, month, 1))

    try:
        return str(datetime.strptime(text, "%Y, %B %d").date())
    except:
        pass

    try:
        return str(datetime.strptime(text, "%Y, %B").date())
    except:
        pass

    try:
        return str(datetime.strptime(text, "%Y").date())
    except:
        return None


def parse_dimensions(text):
    if not text:
        return None, None, None

    m = re.search(r"([\d.]+)\s*x\s*([\d.]+)\s*x\s*([\d.]+)", text)
    if m:
        return float(m.group(1)), float(m.group(2)), float(m.group(3))

    m = re.search(r"([\d.]+)\s*mm", text)
    if m:
        return None, None, float(m.group(1))

    return None, None, None

def parse_status(text):
    if not text:
        return "Unknown"

    first_word = text.split()[0]
    
    if first_word == []:
        return "Unknown"

    return first_word.rstrip(".")

def parse_cpu(text):
    if not text:
        return "Unknown"
    res = re.search(r"\b[A-Za-z0-9]+(?:-[A-Za-z0-9]+)*", text)
    if (res):
        return res.group(0)
    return "Unknown"

def parse_display(text):
    if not text:
        return "Unknown"
    return text.split(",")[0]

In [11]:
import json

FEATURES = [
    ("status", "category"),
    ("OS", "category"),
    ("Chipset", "category"),
    ("SIM", "category"),
    ("Cardslot", "category"),
    ("CPU", "category"),
    ("GPU", "category"),
    ("Display", "category"),
    ("price[EUR]", "numeric"),
    ("weight[g]", "numeric"),
    ("length[mm]", "numeric"),
    ("width[mm]", "numeric"),
    ("thikness[mm]", "numeric"),
    ("Internal memory[GB]", "numeric"),
    ("Internal memory RAM[GB]", "numeric"),
    ("Video[fps]", "numeric"),
    ("Battery Type[mAh]", "numeric"),
    ("Announced", "datetime"),
]

def data_to_rows(data: dict):
    rows = []
    for _, devices in data.items():
        for _, specs in devices.items():
            launch = specs.get("Launch", {})
            platform = specs.get("Platform", {})
            body = specs.get("Body", {})
            memory = specs.get("Memory", {})
            main_cam = specs.get("Main Camera", {})
            battery = specs.get("Battery", {})
            length, width, thikness = parse_dimensions(body.get("Dimensions", ""))
            misc = specs.get("Misc", {})
            display = specs.get("Display", {})

            internal = memory.get("Internal", "")

            row = {
                "status": parse_status(launch.get("Status", "")),
                "OS": parse_os(platform.get("OS", "")),
                "Chipset": parse_chipset(platform.get("Chipset", "")),
                "SIM": parse_sim(body.get("SIM", "")),
                "Cardslot": parse_sim(memory.get("Card slot", "")),
                "CPU": parse_cpu(platform.get("CPU", "")),
                "GPU": parse_cpu(platform.get("GPU", "")),
                "Display": parse_display(display.get("Type", "")),
                "price[EUR]": parse_price(misc.get("Price", "")),
                "weight[g]": parse_weight_g(body.get("Weight", "")),
                "length[mm]": length,
                "width[mm]": width,
                "thikness[mm]": thikness,
                "Internal memory[GB]": parse_storage(internal),
                "Internal memory RAM[GB]": parse_ram(internal),
                "Video[fps]": parse_fps(main_cam.get("Video", "")),
                "Battery Type[mAh]": parse_battery_mah(battery.get("Type", "")),
                "Announced": parse_announced(launch.get("Announced", "")),
            }
            rows.append(row)
    return rows

def to_number_or_none(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == "":
        return None
    try:
        return float(s) if "." in s else int(s)
    except ValueError:
        return None

def save_tsv(data: dict, out_path="data.tsv"):
    headers = [
        [x for (x, _) in FEATURES]
    ]
    rows = [list(i.values()) for i in data_to_rows(data)]
    headers.extend(rows)

    with open(out_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerows(headers)

    print(f"Saved: {out_path}")

def save_json(data: dict, out_path="data.json"):
    rows = data_to_rows(data)

    values = {x: sorted({r[x] for r in rows if r.get(x)}) for (x, y) in FEATURES if y == "category"}

    header = []
    for name, ftype in FEATURES:
        item = {"feature_name": name, "type": ftype}
        if ftype == "category":
            item["values"] = values.get(name, [])
        header.append(item)

    typed_rows = []
    numeric_fields = {n for n, t in FEATURES if t == "numeric"}
    for r in rows:
        rr = dict(r)
        for nf in numeric_fields:
            rr[nf] = to_number_or_none(rr.get(nf))
        typed_rows.append(rr)

    payload = {"header": header, "data": typed_rows}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"Saved: {out_path}")

def arff_quote(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\\", "\\\\").replace('"', '\\"')
    return f'"{s}"'

def save_arff(data: dict, out_path="data.arff"):
    rows = data_to_rows(data)
    
    values = {x: sorted({r[x] for r in rows if r.get(x)}) for (x, y) in FEATURES if y == "category"}

    with open(out_path, "w", encoding="utf-8", newline="\n") as f:
        f.write(f"@RELATION devices\n\n")

        for name, ftype in FEATURES:
            if ftype == "numeric":
                f.write(f"@ATTRIBUTE {arff_quote(name)} NUMERIC\n")
            elif ftype == "category":
                nom = ",".join(arff_quote(v) for v in values.get(name, []))
                f.write(f"@ATTRIBUTE {arff_quote(name)} {{{nom}}}\n")
            else:
                f.write(f"@ATTRIBUTE {arff_quote(name)} STRING\n")

        f.write("\n@DATA\n")
        for r in rows:
            values = []
            for name, ftype in FEATURES:
                v = r.get(name, "")
                if ftype == "numeric":
                    num = to_number_or_none(v)
                    values.append("?" if num is None else str(num))
                elif ftype == "category":
                    values.append("?" if not v else arff_quote(v))
                else:
                    values.append(arff_quote(v))
            f.write(",".join(values) + "\n")

    print(f"Saved: {out_path}")

save_tsv(devices_data)
save_json(devices_data)
save_arff(devices_data)


Saved: data.tsv
Saved: data.json
Saved: data.arff


In [12]:
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

with open("data.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

df = pd.DataFrame(dataset["data"])

y = df["OS"]
X = df.drop(columns=["OS"])

target_mapping = {label: idx for idx, label in enumerate(sorted(y.unique()))}
y = y.map(target_mapping)

current_year = 2026

approx_month = 2
data_announce = pd.to_datetime(X["Announced"], errors="coerce").dt
X["Announced"] = (current_year - data_announce.year)**2 + (approx_month - data_announce.month)

numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

categorical_cols = X.select_dtypes(include=["object"]).columns
X[categorical_cols] = X[categorical_cols].fillna("Unknown")

for c in categorical_cols:
    freqs = X[c].value_counts()
    X[c] = X[c].map(freqs)

scaler = MinMaxScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

df_final = X.copy()
df_final["OS"] = y

df_final.to_csv("data.csv", index=False)
print("Saved: data.csv")

Saved: data.csv
